# LLM4Rec Midterm — sweettv TV Program Recommender

Kaggle competition: sweettv-tv-program-recommender — predict top-5 TV programs per user, metric MAP@5.
HSE LLM4Rec exam project. Built step by step in midterm.ipynb, one section per prompt.

This notebook contains markdown headings only (plus this first code cell). Code cells will be added in later steps.


In [49]:
# --- notebook wall-clock start (consumed by the environment/runtime cell at the end) ---
import time as _time
NOTEBOOK_T0 = _time.time()
print(f"notebook started at {_time.strftime('%Y-%m-%d %H:%M:%S')}")


notebook started at 2026-11-01 12:48:11


## 0. Setup and config


In [50]:
import sys
import os
from pathlib import Path

# Paths (relative to this notebook's folder)
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# Global random seed (fixed for reproducibility)
RANDOM_SEED = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

print("Python version:", sys.version)
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Outputs dir:", OUTPUTS_DIR)
print("Random seed:", RANDOM_SEED)


Python version: 3.13.6 (v3.13.6:4e665351082, Aug  6 2025, 11:22:35) [Clang 16.0.0 (clang-1600.0.26.6)]
Project root: /Users/yuliavodopanova/Desktop/llm4rec_midterm
Data dir: /Users/yuliavodopanova/Desktop/llm4rec_midterm/data
Outputs dir: /Users/yuliavodopanova/Desktop/llm4rec_midterm/outputs
Random seed: 42


## 1. Data download and loading


In [51]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")

FILES = {
    "train_log": "dataset11-30.csv",
    "schedule_11_20": "export_arh_11-20-final.csv",
    "schedule_21_30": "export_arh_21-30-final.csv",
    "schedule_31_42": "export_arh_31-42-final.csv",
    "sample_submission": "submission.csv",
}

print("=" * 72)
print("SEPARATOR / ENCODING CHECK (first 2 KB of each file)")
print("=" * 72)
for key, fname in FILES.items():
    p = DATA_DIR / fname
    with open(p, "rb") as f:
        head_bytes = f.read(2048)
    head_text = head_bytes.decode("utf-8", errors="replace")
    first_line = head_text.split("\n", 1)[0]
    n_comma = first_line.count(",")
    n_semicolon = first_line.count(";")
    n_tab = first_line.count("\t")
    if n_comma >= n_semicolon and n_comma >= n_tab:
        sep = ","
    elif n_semicolon >= n_tab:
        sep = ";"
    else:
        sep = "\t"
    print(f"{key:18s} | {fname:38s} | enc=utf-8 | sep={sep!r:4s} | est_ncols={first_line.count(sep)+1} | {first_line[:65]!r}")

print()
print("=" * 72)
print("LOADING TABLES")
print("=" * 72)

# Train log: user_id as str (some ids exceed int64); keep header typo "duraton"
train = pd.read_csv(DATA_DIR / FILES["train_log"], dtype={"user_id": str, "vsetv_id": str})

# Schedule: concatenate the three EPG windows
schedule_parts = []
for key in ["schedule_11_20", "schedule_21_30", "schedule_31_42"]:
    df = pd.read_csv(DATA_DIR / FILES[key], on_bad_lines="warn")
    schedule_parts.append(df)
schedule = pd.concat(schedule_parts, ignore_index=True)
del schedule_parts

# Sample submission
sub = pd.read_csv(DATA_DIR / FILES["sample_submission"], dtype={"user_id": str})

table_info = [
    ("train_log", FILES["train_log"], train),
    ("schedule (3 windows concatenated)", "export_arh_*_final.csv", schedule),
    ("sample_submission", FILES["sample_submission"], sub),
]

for name, src, df in table_info:
    print(f"\n{'='*72}\nTABLE: {name}  source: {src}\n{'='*72}")
    print("shape:", df.shape)
    print("dtypes:")
    print(df.dtypes.to_string())
    print("missing values per column:")
    print(df.isna().sum().to_string())
    time_cols = [c for c in df.columns if "time" in c.lower()]
    for c in time_cols:
        print(f"  raw time col '{c}': min={df[c].min()!r}  max={df[c].max()!r}")

# Parse time columns to datetime
train["start_dt"] = pd.to_datetime(train["start_time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
train["stop_dt"] = pd.to_datetime(train["stop_time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
schedule["start_dt"] = pd.to_datetime(schedule["start_time"], format="%d.%m.%Y %H:%M:%S", errors="coerce")

print("\nParsed datetime columns:")
for name, src, df in table_info:
    dt_cols = [c for c in df.columns if c.endswith("_dt")]
    for c in dt_cols:
        print(f"  {name}.{c}: min={df[c].min()}  max={df[c].max()}  na={df[c].isna().sum()}")

print("\nTrain head(3):")
print(train.head(3).to_string())
print("\nSchedule head(3):")
print(schedule.head(3).to_string())
print("\nSubmission head(3):")
print(sub.head(3).to_string())


SEPARATOR / ENCODING CHECK (first 2 KB of each file)
train_log          | dataset11-30.csv                       | enc=utf-8 | sep=','  | est_ncols=5 | '"user_id","vsetv_id","start_time","stop_time","duraton"'
schedule_11_20     | export_arh_11-20-final.csv             | enc=utf-8 | sep=','  | est_ncols=13 | 'channel_id,channel_title,start_time,duration,tv_show_title,tv_sho'
schedule_21_30     | export_arh_21-30-final.csv             | enc=utf-8 | sep=','  | est_ncols=13 | 'channel_id,channel_title,start_time,duration,tv_show_title,tv_sho'
schedule_31_42     | export_arh_31-42-final.csv             | enc=utf-8 | sep=','  | est_ncols=13 | 'channel_id,channel_title,start_time,duration,tv_show_title,tv_sho'
sample_submission  | submission.csv                         | enc=utf-8 | sep=','  | est_ncols=2 | 'user_id,tv_show_id'

LOADING TABLES


/var/folders/b_/ntqb5st965z0z5cvz5qqv7pw0000gn/T/ipykernel_16845/1902801675.py:46: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_DIR / FILES[key], on_bad_lines="warn")



TABLE: train_log  source: dataset11-30.csv
shape: (5821987, 5)
dtypes:
user_id       object
vsetv_id      object
start_time    object
stop_time     object
duraton        int64
missing values per column:
user_id       0
vsetv_id      0
start_time    0
stop_time     0
duraton       0
  raw time col 'start_time': min='2020-03-09 00:00:02'  max='2020-07-26 23:58:42'
  raw time col 'stop_time': min='2020-03-09 00:01:26'  max='2020-07-26 23:59:58'

TABLE: schedule (3 windows concatenated)  source: export_arh_*_final.csv
shape: (3444007, 13)
dtypes:
channel_id             int64
channel_title         object
start_time            object
duration               int64
tv_show_title         object
tv_show_id             int64
tv_show_category      object
tv_show_genre_1       object
tv_show_genre_2       object
tv_show_genre_3       object
year_of_production    object
director              object
actors                object
missing values per column:
channel_id                  0
channel_title   

## 2. EDA


In [52]:
import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 72)
print("EDA — SECTION 2")
print("=" * 72)

# --- Users ---
train_users = train["user_id"].nunique()
sub_users = sub["user_id"].nunique()
train_user_set = set(train["user_id"].unique())
sub_user_set = set(sub["user_id"].unique())
sub_not_in_train = len(sub_user_set - train_user_set)
print(f"unique users in train log: {train_users:,}")
print(f"unique users in sample submission: {sub_users:,}")
print(f"submission users NOT in train: {sub_not_in_train:,}")

# --- Channels ---
train_channels = train["vsetv_id"].nunique()
sched_channels = schedule["channel_id"].nunique()
train_chan_set = set(train["vsetv_id"].unique())
sched_chan_set = set(schedule["channel_id"].unique())
chan_overlap = len(train_chan_set & sched_chan_set)
print(f"\nunique channels in train log (vsetv_id): {train_channels:,}")
print(f"unique channels in schedule (channel_id): {sched_channels:,}")
print(f"channel overlap between train log and schedule: {chan_overlap:,}")

# --- tv_show_id == 0 ---
sched_tv = schedule["tv_show_id"]
zero_share = (sched_tv == 0).mean()
nonzero_unique = schedule.loc[sched_tv != 0, "tv_show_id"].nunique()
print(f"\nshare of schedule rows with tv_show_id == 0: {zero_share:.6f} ({zero_share*100:.4f}%)")
print(f"unique non-zero tv_show_id in schedule: {nonzero_unique:,}")

# --- View duration (end - start) ---
train["view_dur_sec"] = (train["stop_dt"] - train["start_dt"]).dt.total_seconds()
# cross-check with duraton column if present
if "duraton" in train.columns:
    dur_match = (train["view_dur_sec"] == train["duraton"]).mean()
    print(f"\nview_dur_sec == duraton column share: {dur_match:.6f}")
dur_desc = train["view_dur_sec"].describe()
print("\nview duration (stop - start) seconds — describe():")
print(dur_desc.to_string())
zero_neg_dur = (train["view_dur_sec"] <= 0).sum()
long_12h = (train["view_dur_sec"] > 12 * 3600).sum()
print(f"\ncount of zero/negative durations: {zero_neg_dur:,}")
print(f"count of views longer than 12 hours: {long_12h:,}")

# --- Views per user ---
views_per_user = train.groupby("user_id").size()
vpu_desc = views_per_user.describe()
print("\nviews per user — describe():")
print(vpu_desc.to_string())

# --- ISO weeks covered + views per week ---
train["iso_year"] = train["start_dt"].dt.isocalendar().year
train["iso_week"] = train["start_dt"].dt.isocalendar().week
weekly = train.groupby(["iso_year", "iso_week"]).size()
n_weeks = len(weekly)
print(f"\nnumber of ISO weeks covered by log: {n_weeks}")
print("\nviews per week (first 12 rows):")
print(weekly.head(12).to_string())

# --- Local time vs UTC / DST check ---
hour_dist = train["start_dt"].dt.hour.value_counts().sort_index()
print("\nview start hour distribution (train log):")
print(hour_dist.to_string())
# DST gap check: does any calendar date have fewer than 24 distinct start hours in schedule?
sched_dates = schedule["start_dt"].dt.date
sched_hours = schedule["start_dt"].dt.hour
sched_by_date_hour = schedule.groupby([sched_dates, sched_hours]).size().unstack(fill_value=0)
hours_present_per_date = (sched_by_date_hour > 0).sum(axis=1)
odd_dates = hours_present_per_date[hours_present_per_date != 24]
print(f"\nschedule dates with != 24 distinct start hours: {len(odd_dates)}")
if len(odd_dates) > 0:
    print(odd_dates.head(10).to_string())

# --- Schedule extra columns (title/category/genre) — 5 example rows ---
print("\nschedule example rows (extra columns):")
print(schedule[["channel_title","start_time","tv_show_title","tv_show_id","tv_show_category","tv_show_genre_1","tv_show_genre_2","tv_show_genre_3","year_of_production"]].head(5).to_string())


EDA — SECTION 2
unique users in train log: 4,914
unique users in sample submission: 1,260
submission users NOT in train: 0

unique channels in train log (vsetv_id): 202
unique channels in schedule (channel_id): 405
channel overlap between train log and schedule: 0

share of schedule rows with tv_show_id == 0: 0.889114 (88.9114%)
unique non-zero tv_show_id in schedule: 9,671

view_dur_sec == duraton column share: 0.999980

view duration (stop - start) seconds — describe():
count    5.821987e+06
mean     1.480532e+03
std      3.123872e+03
min      6.100000e+01
25%      1.330000e+02
50%      3.960000e+02
75%      1.452000e+03
max      3.421450e+05

count of zero/negative durations: 0
count of views longer than 12 hours: 2,635

views per user — describe():
count     4914.000000
mean      1184.775539
std       1243.477444
min          1.000000
25%        476.000000
50%        874.000000
75%       1547.000000
max      38114.000000

number of ISO weeks covered by log: 20

views per week (firs

### Findings so far (numbers printed in sections 1–2)

- Train log: 5,821,987 viewing events; 4,914 unique users; time range 2020-03-09 00:00:02 → 2020-07-26 23:58:42.
- Sample submission: 1,260 users; all 1,260 are present in the train log (0 submission users are not in train).
- Schedule: 3,444,007 rows across the 3 EPG windows; 405 unique channels; 9,671 unique non-zero tv_show_id; 88.91% of schedule rows have tv_show_id == 0.
- Channels: train log has 202 unique vsetv_id; schedule has 405 unique channel_id; raw overlap = 0 because train vsetv_id is str while schedule channel_id is int64 (must cast to a common type before matching).
- View duration (stop − start): median 396 s (~6.6 min), mean 1,481 s; 0 zero/negative durations; 2,635 views exceed 12 hours; the duraton column agrees with stop−start for 99.998% of rows.
- Views per user: mean 1,185; min 1; max 38,114.
- Train log spans 20 ISO weeks; in the shown weeks 11–22, weekly views range from 182,044 (week 21) to 465,318 (week 13).
- Viewing peaks in the evening (hour 21: 542,318 views; hour 20: 511,782), consistent with local time; night hours are much lower.
- Schedule parsed range 2020-03-08 → 2020-10-19 extends beyond the train-log end (2020-07-26): future schedule exists, relevant for a leakage-free temporal split.
- Three schedule dates have ≠24 distinct start hours (2020-03-08: 1 h; 2020-03-29: 23 h; 2020-10-19: 12 h); the first and last are range edges.

### Open questions
- Are train vsetv_id and schedule channel_id the same identifier namespace once cast to a common type? (Raw overlap was 0 only because of str-vs-int.)
- What do the schedule file windows "11-20", "21-30", "31-42" refer to?
- Is the 23-hour date 2020-03-29 a genuine missing EPG hour or a parsing artifact? (Russia dropped DST in 2014, so a DST gap is unlikely.)
- Do schedule programs on the same channel overlap, or are there gaps within a broadcast day?
- Are there test-period users with no train history (cold start) who still appear in the submission?
- What does tv_show_id == 0 represent (filler/news), and should top-5 predictions include it?


## 3. Matching views to TV schedule (rule + handling of missing/zero tv_show_id)


### Part A — quick checks


In [53]:
# --- Part A: quick checks ---
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")

def _load():
    train = pd.read_csv(DATA_DIR / "dataset11-30.csv", dtype={"user_id": str, "vsetv_id": str})
    schedule = pd.concat([
        pd.read_csv(DATA_DIR / f, on_bad_lines="warn")
        for f in ["export_arh_11-20-final.csv", "export_arh_21-30-final.csv", "export_arh_31-42-final.csv"]
    ], ignore_index=True)
    sub = pd.read_csv(DATA_DIR / "submission.csv", dtype={"user_id": str})
    return train, schedule, sub

try:
    train
except NameError:
    train, schedule, sub = _load()

for df, fmt in [(train, "%Y-%m-%d %H:%M:%S"), (schedule, "%d.%m.%Y %H:%M:%S")]:
    for c in ("start_time", "stop_time"):
        if c in df.columns and (c[:-5] + "dt") not in df.columns:
            df[c[:-5] + "dt"] = pd.to_datetime(df[c], format=fmt, errors="coerce")

train["vsetv_id_int"] = train["vsetv_id"].astype(int)
train["view_dur_sec"] = (train["stop_dt"] - train["start_dt"]).dt.total_seconds()
train["iso_week"] = train["start_dt"].dt.isocalendar().week
train["iso_year"] = train["start_dt"].dt.isocalendar().year

schedule["duration_s"] = schedule["duration"].astype("int64")
schedule["end_dt_s"] = schedule["start_dt"] + pd.to_timedelta(schedule["duration_s"], unit="s")
schedule_sorted = schedule.sort_values(["channel_id", "start_dt"])
schedule_sorted["next_start"] = schedule_sorted.groupby("channel_id")["start_dt"].shift(-1)
schedule_sorted["gap_to_next"] = (schedule_sorted["next_start"] - schedule_sorted["end_dt_s"]).dt.total_seconds()

print("=" * 70)
print("PART A — quick checks")
print("=" * 70)

# A1
sched_channels = set(schedule["channel_id"].unique())
train_ch = set(train["vsetv_id_int"].unique())
overlap_ch = train_ch & sched_channels
unk = train[~train["vsetv_id_int"].isin(sched_channels)]
print(f"[A1] train unique channels (vsetv_id int): {len(train_ch):,}")
print(f"[A1] schedule unique channels (channel_id): {len(sched_channels):,}")
print(f"[A1] channel overlap after casting: {len(overlap_ch):,}")
print(f"[A1] views on channels NOT in schedule: {len(unk):,} ({len(unk)/len(train)*100:.4f}%)")
print(f"[A1] view-seconds on channels NOT in schedule: {unk['duraton'].sum():,} ({unk['duraton'].sum()/train['duraton'].sum()*100:.4f}%)")

# A2
print(f"[A2] schedule duration (s): min={schedule['duration_s'].min()}, max={schedule['duration_s'].max()}, median={schedule['duration_s'].median()}")
print(f"[A2] gap_to_next (s) describe:")
print(schedule_sorted["gap_to_next"].describe().to_string())
print(f"[A2] negative gaps: {(schedule_sorted['gap_to_next']<0).sum():,}, median gap: {schedule_sorted['gap_to_next'].median()}")

# A3
before = len(schedule)
schedule_nd = schedule.drop_duplicates()
print(f"[A3] rows before dedup: {before:,}, after drop_duplicates: {len(schedule_nd):,}, removed exact duplicates: {before-len(schedule_nd):,}")
for f in ["export_arh_11-20-final.csv","export_arh_21-30-final.csv","export_arh_31-42-final.csv"]:
    d = pd.read_csv(DATA_DIR / f)
    d["sd"] = pd.to_datetime(d["start_time"], format="%d.%m.%Y %H:%M:%S", errors="coerce")
    print(f"[A3] {f}: rows={len(d):,}, range {d['sd'].min()} -> {d['sd'].max()}")

# A4
gaps = schedule_nd.sort_values(["channel_id","start_dt"]).copy()
gaps["next_start"] = gaps.groupby("channel_id")["start_dt"].shift(-1)
gaps["gap"] = (gaps["next_start"] - (gaps["start_dt"] + pd.to_timedelta(gaps["duration_s"], unit="s"))).dt.total_seconds()
print(f"[A4] transitions: overlap={(gaps['gap']<0).sum():,}, exact={(gaps['gap']==0).sum():,}, gap>0={(gaps['gap']>0).sum():,}")
print("[A4] overlap examples (gap<0):")
print(gaps[gaps["gap"]<0][["channel_id","start_time","duration","tv_show_title","tv_show_id"]].head(5).to_string())
print("[A4] gap examples (gap>0):")
print(gaps[gaps["gap"]>0][["channel_id","start_time","duration","tv_show_title","tv_show_id"]].head(5).to_string())

# A5
for d in ["2020-03-28","2020-03-29","2020-03-30"]:
    td = pd.Timestamp(d).date()
    tm = train["start_dt"].dt.date == td
    sm = schedule_nd["start_dt"].dt.date == td
    print(f"[A5] {d}: train views={tm.sum():,} | schedule programs={sm.sum():,}")

# A6
for wk in [13, 21]:
    m = (train["iso_year"]==2020) & (train["iso_week"]==wk)
    print(f"[A6] ISO week {wk}: views={m.sum():,}")
    print(train.loc[m].groupby(train.loc[m,"start_dt"].dt.date).size().to_string())

# A7
print(f"[A7] submission shape: {sub.shape} | unique user_id: {sub['user_id'].nunique():,} | data rows: {len(sub):,}")


PART A — quick checks
[A1] train unique channels (vsetv_id int): 202
[A1] schedule unique channels (channel_id): 405
[A1] channel overlap after casting: 197
[A1] views on channels NOT in schedule: 113,579 (1.9509%)
[A1] view-seconds on channels NOT in schedule: 116,312,068 (1.3495%)
[A2] schedule duration (s): min=0, max=86400, median=1500.0
[A2] gap_to_next (s) describe:
count    3.443602e+06
mean     1.628028e+00
std      1.494102e+03
min     -1.800000e+04
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.419200e+06
[A2] negative gaps: 13, median gap: 0.0
[A3] rows before dedup: 3,444,007, after drop_duplicates: 3,444,007, removed exact duplicates: 0
[A3] export_arh_11-20-final.csv: rows=1,074,235, range 2020-03-08 23:00:00 -> 2020-05-18 11:45:00
[A3] export_arh_21-30-final.csv: rows=1,074,394, range 2020-05-18 00:00:00 -> 2020-07-27 11:55:00


/var/folders/b_/ntqb5st965z0z5cvz5qqv7pw0000gn/T/ipykernel_16845/184459049.py:64: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  d = pd.read_csv(DATA_DIR / f)


[A3] export_arh_31-42-final.csv: rows=1,295,378, range 2020-07-27 00:00:00 -> 2020-10-19 11:55:00
[A4] transitions: overlap=13, exact=3,443,185, gap>0=404
[A4] overlap examples (gap<0):
         channel_id           start_time  duration                          tv_show_title  tv_show_id
2187546          25  30.08.2020 23:30:00     30600                               Повторы.           0
186399          282  30.03.2020 04:30:00      7200                       Т/с 'Вечеринка'.    10003055
382527          487  17.05.2020 22:35:00      8700     Х/ф 'Летучий отряд Скотланд Ярда'.           0
2755814         652  28.09.2020 05:05:00      3900  Лучшие друзья собаки, 2 сезон, 10 эп.           0
1611820         667  20.07.2020 02:00:00     32400                          Профилактика.           0
[A4] gap examples (gap>0):
      channel_id           start_time  duration                 tv_show_title  tv_show_id
444            3  29.03.2020 01:25:00      6000  'Лига смеха', 6 сезон, 3 эп.      70

### Part B — matching rule

**Rule.** A view (user, channel, [view_start, view_stop)) matches a schedule airing (channel, [air_start, air_end)) when:
- same channel (`vsetv_id` == `channel_id` after casting to int), AND
- the intervals overlap by strictly more than 0 seconds: `max(view_start, air_start) < min(view_stop, air_end)`.

A view can span several airings; it is expanded so each (view, airing) pair gets its own `overlap_seconds` and `overlap_frac = overlap_seconds / airing_duration`.

**Implementation.** Per channel, schedule airings are sorted by `start_dt`. For each view we take the bounded window of airings that can possibly overlap (`air_start` in `(view_start - max_duration, view_stop)`), found with `np.searchsorted`, then keep those with `air_end > view_start`. No full cross join in memory.

**Exclusions.** Pairs to an airing with `tv_show_id == 0`, and views matching no airing, are kept only in diagnostics. They are EXCLUDED from training interactions and from the recommendation candidate pool.


In [54]:
# --- Part B: matching rule implementation ---
schedule_nd = schedule.drop_duplicates().reset_index(drop=True)
schedule_nd["duration_s"] = schedule_nd["duration"].astype("int64")
schedule_nd["end_dt"] = schedule_nd["start_dt"] + pd.to_timedelta(schedule_nd["duration_s"], unit="s")

sched_channels = set(schedule_nd["channel_id"].unique())
train_match = train[train["vsetv_id_int"].isin(sched_channels)].copy()
max_dur = int(schedule_nd["duration_s"].max())

print("=" * 70)
print("PART B — matching")
print("=" * 70)
print(f"[B] schedule rows after dedup: {len(schedule_nd):,}  | max airing duration: {max_dur} s ({max_dur/3600:.2f} h)")
print(f"[B] train views on known channels: {len(train_match):,} ({len(train_match)/len(train)*100:.2f}% of all views)")

import time, resource
t0 = time.time()
mem0 = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss

pairs_vk = []
pairs_ai = []
pairs_os = []

for ch in sorted(train_match["vsetv_id_int"].unique()):
    grp = train_match[train_match["vsetv_id_int"] == ch]
    sch = schedule_nd[schedule_nd["channel_id"] == ch].sort_values("start_dt")
    if len(sch) == 0 or len(grp) == 0:
        continue
    a_start = sch["start_dt"].values.astype("datetime64[ns]").view("int64") // 10**9
    a_end = sch["end_dt"].values.astype("datetime64[ns]").view("int64") // 10**9
    ai_global = sch.index.values
    v_start = grp["start_dt"].values.astype("datetime64[ns]").view("int64") // 10**9
    v_stop = grp["stop_dt"].values.astype("datetime64[ns]").view("int64") // 10**9
    v_key = grp.index.values
    V = len(v_start)
    lo_all = np.searchsorted(a_start, v_start - max_dur, side="right")
    hi_all = np.searchsorted(a_start, v_stop, side="left")
    n_cand = np.maximum(hi_all - lo_all, 0)
    total_cand = int(n_cand.sum())
    if total_cand == 0:
        continue
    block_start = np.repeat(np.cumsum(n_cand) - n_cand, n_cand)
    pos_in_block = np.arange(total_cand) - block_start
    airing_idx = np.repeat(lo_all, n_cand) + pos_in_block
    view_idx = np.repeat(np.arange(V), n_cand)
    ov = np.minimum(a_end[airing_idx], v_stop[view_idx]) - np.maximum(a_start[airing_idx], v_start[view_idx])
    m = ov > 0
    pairs_vk.append(v_key[view_idx][m])
    pairs_ai.append(ai_global[airing_idx][m])
    pairs_os.append(ov[m].astype("int64"))

import sys
_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss - mem0
_rss_mb = _rss / (1024 * 1024) if sys.platform == "darwin" else _rss / 1024
print(f"[B] matching runtime: {time.time()-t0:.2f} s  | peak RSS delta: {_rss_mb:.1f} MB ({sys.platform} ru_maxrss units normalised)")

if pairs_vk:
    pairs = pd.DataFrame({"view_key": np.concatenate(pairs_vk), "airing_idx": np.concatenate(pairs_ai), "overlap_sec": np.concatenate(pairs_os)})
else:
    pairs = pd.DataFrame(columns=["view_key", "airing_idx", "overlap_sec"])

print(f"[B] matched (view, airing) pairs: {len(pairs):,}")
print(f"[B] unique views matched: {pairs['view_key'].nunique():,} / {len(train_match):,}")

# consistency check on one channel, first 6 views (vectorized pairs only)
ch0 = sorted(train_match["vsetv_id_int"].unique())[0]
grp0 = train_match[train_match["vsetv_id_int"] == ch0].head(6)
sub0 = pairs[pairs["view_key"].isin(set(grp0.index))]
views_checked = int(sub0["view_key"].nunique())
print(f"[B-check] vectorized pairs present for {views_checked}/{len(grp0)} sampled views on channel {ch0} | all overlap_sec > 0: {bool((sub0['overlap_sec'] > 0).all())}")

# enrich with view + airing info (renamed to avoid suffix clashes)
tv = train_match[["user_id", "vsetv_id_int", "start_dt", "stop_dt", "duraton"]].rename(columns={"start_dt": "view_start_dt", "stop_dt": "view_stop_dt"})
sv = schedule_nd[["channel_id", "start_dt", "end_dt", "tv_show_id", "tv_show_title", "tv_show_category", "duration_s"]].rename(columns={"start_dt": "air_start_dt", "end_dt": "air_end_dt", "duration_s": "airing_dur_sec"})
pairs = pairs.merge(tv, left_on="view_key", right_index=True, how="left").merge(sv, left_on="airing_idx", right_index=True, how="left")
pairs["overlap_frac"] = pairs["overlap_sec"] / pairs["airing_dur_sec"]
pairs["iso_week"] = pairs["view_start_dt"].dt.isocalendar().week
print("[B] sample matched pairs:")
print(pairs[["view_key", "user_id", "vsetv_id_int", "view_start_dt", "view_stop_dt", "air_start_dt", "air_end_dt", "tv_show_id", "tv_show_title", "overlap_sec", "overlap_frac"]].head(10).to_string())

# shares of views / view-seconds
all_views = set(train.index)
matched = set(pairs["view_key"].unique())
labelled = set(pairs[pairs["tv_show_id"] != 0]["view_key"].unique())
id0_only = matched - labelled
nothing = all_views - matched
def _sec(s):
    return train.loc[list(s), "duraton"].sum() if len(s) else 0
tot_sec = train["duraton"].sum()
print(f"[B-share] views matched to labelled airing (id!=0): {len(labelled):,} ({len(labelled)/len(train)*100:.2f}%), view-sec {_sec(labelled):,} ({_sec(labelled)/tot_sec*100:.2f}%)")
print(f"[B-share] views matched to id-0 airing only: {len(id0_only):,} ({len(id0_only)/len(train)*100:.2f}%), view-sec {_sec(id0_only):,} ({_sec(id0_only)/tot_sec*100:.2f}%)")
print(f"[B-share] views matched to nothing: {len(nothing):,} ({len(nothing)/len(train)*100:.2f}%), view-sec {_sec(nothing):,} ({_sec(nothing)/tot_sec*100:.2f}%)")

# sensitivity table
print("[B-sens] threshold             | interactions | unique users | unique shows")
for name, mask in [("overlap>0s", pairs["overlap_sec"] > 0),
                   (">=60s", pairs["overlap_sec"] >= 60),
                   (">=300s", pairs["overlap_sec"] >= 300),
                   ("overlap_frac>=0.5", pairs["overlap_frac"] >= 0.5)]:
    s = pairs[mask]
    print(f"[B-sens] {name:18s} | {len(s):,} | {s['user_id'].nunique():,} | {s['tv_show_id'].nunique():,}")
print("[B-sens] overlap_frac quantiles:")
print(pairs["overlap_frac"].quantile([0, 0.5, 0.9, 0.99, 1.0]).to_string())

# interaction table + save to data/interim/
INTERIM_DIR = DATA_DIR / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
lab = pairs[pairs["tv_show_id"] != 0].copy()
inter = lab.groupby(["user_id", "tv_show_id"]).agg(
    n_airings_watched=("overlap_sec", "size"),
    total_seconds=("overlap_sec", "sum"),
    first_time=("view_start_dt", "min"),
    last_time=("view_stop_dt", "max"),
    n_distinct_days=("view_start_dt", lambda s: s.dt.date.nunique()),
).reset_index()
inter["iso_week_first"] = inter["first_time"].dt.isocalendar().week
inter["iso_week_last"] = inter["last_time"].dt.isocalendar().week
print(f"[B] interaction table: {len(inter):,} rows, {inter['user_id'].nunique():,} users, {inter['tv_show_id'].nunique():,} shows")
try:
    import pyarrow
    pairs.to_parquet(INTERIM_DIR / "matched_pairs.parquet", index=False)
    inter.to_parquet(INTERIM_DIR / "interactions.parquet", index=False)
    fmt = "parquet"
except Exception as e:
    pairs.to_csv(INTERIM_DIR / "matched_pairs.csv", index=False)
    inter.to_csv(INTERIM_DIR / "interactions.csv", index=False)
    fmt = "csv"
print(f"[B] saved to data/interim/ as {fmt}")


PART B — matching
[B] schedule rows after dedup: 3,444,007  | max airing duration: 86400 s (24.00 h)
[B] train views on known channels: 5,708,408 (98.05% of all views)
[B] matching runtime: 4.04 s  | peak RSS delta: 0.0 MB (darwin ru_maxrss units normalised)
[B] matched (view, airing) pairs: 7,752,433
[B] unique views matched: 5,408,598 / 5,708,408
[B-check] vectorized pairs present for 0/6 sampled views on channel 3 | all overlap_sec > 0: True
[B] sample matched pairs:
   view_key               user_id  vsetv_id_int       view_start_dt        view_stop_dt        air_start_dt          air_end_dt  tv_show_id                      tv_show_title  overlap_sec  overlap_frac
0      3556   7675639230638827735             3 2020-03-09 04:56:29 2020-03-09 06:47:04 2020-03-09 06:00:00 2020-03-09 07:40:00           0  'Жизнь известных людей', 2 сезон.         2824      0.470667
1      3623  16524941906607473660             3 2020-03-09 05:09:24 2020-03-09 07:34:25 2020-03-09 06:00:00 2020-03-09 07

In [55]:
# --- Part A revisit: naive-loop cross-check on a NON-empty sample ---
import numpy as np
rng = np.random.default_rng(7)

print("=" * 70)
print("PART A revisit — naive vs vectorized matching (non-empty sample)")
print("=" * 70)

busiest_ch = int(train_match["vsetv_id_int"].value_counts().idxmax())
print(f"busiest channel: {busiest_ch} | views on it: {int(train_match['vsetv_id_int'].value_counts().iloc[0]):,}")

ch = busiest_ch
grp = train_match[train_match["vsetv_id_int"] == ch]
sch = schedule_nd[schedule_nd["channel_id"] == ch].sort_values("start_dt").reset_index(drop=True)
max_dur = int(schedule_nd["duration_s"].max())

a_start = sch["start_dt"].values.astype("datetime64[ns]").view("int64") // 10**9
a_end = sch["end_dt"].values.astype("datetime64[ns]").view("int64") // 10**9
ai_global = sch.index.values
v_start = grp["start_dt"].values.astype("datetime64[ns]").view("int64") // 10**9
v_stop = grp["stop_dt"].values.astype("datetime64[ns]").view("int64") // 10**9
v_key = grp.index.values
V = len(v_start)
lo_all = np.searchsorted(a_start, v_start - max_dur, side="right")
hi_all = np.searchsorted(a_start, v_stop, side="left")
n_cand = np.maximum(hi_all - lo_all, 0)
total_cand = int(n_cand.sum())
if total_cand:
    block_start = np.repeat(np.cumsum(n_cand) - n_cand, n_cand)
    pos_in_block = np.arange(total_cand) - block_start
    airing_idx = np.repeat(lo_all, n_cand) + pos_in_block
    view_idx = np.repeat(np.arange(V), n_cand)
    ov = np.minimum(a_end[airing_idx], v_stop[view_idx]) - np.maximum(a_start[airing_idx], v_start[view_idx])
    m = ov > 0
    vec = pd.DataFrame({"view_key": v_key[view_idx][m], "airing_idx": ai_global[airing_idx][m], "overlap_sec": ov[m].astype("int64")})
    vec = vec.sort_values(["view_key", "airing_idx"]).reset_index(drop=True)
else:
    vec = pd.DataFrame(columns=["view_key", "airing_idx", "overlap_sec"])

matched_views = vec["view_key"].unique()
n_sample = min(2000, len(matched_views))
sample_views = rng.choice(matched_views, size=n_sample, replace=False)
sample_set = set(sample_views.tolist())

naive_rows = []
for vk in sample_views:
    v = train.loc[vk]
    vs_ts = int(v["start_dt"].value // 10**9)
    vstop_ts = int(v["stop_dt"].value // 10**9)
    lo = int(np.searchsorted(a_start, vs_ts - max_dur, side="right"))
    hi = int(np.searchsorted(a_start, vstop_ts, side="left"))
    for pos in range(lo, hi):
        ai = int(ai_global[pos])
        a = sch.loc[ai]
        os_ = (min(v["stop_dt"], a["end_dt"]) - max(v["start_dt"], a["start_dt"])).total_seconds()
        if os_ > 0:
            naive_rows.append((vk, ai, int(os_)))
naive = pd.DataFrame(naive_rows, columns=["view_key", "airing_idx", "overlap_sec"]).sort_values(["view_key", "airing_idx"]).reset_index(drop=True)

comp = vec[vec["view_key"].isin(sample_set)].sort_values(["view_key", "airing_idx"]).reset_index(drop=True)
print(f"sampled matched views: {n_sample}")
print(f"naive pairs: {len(naive):,} | vectorized pairs for the same views: {len(comp):,}")
if len(naive) != len(comp):
    print(f"COUNT MISMATCH: naive={len(naive)} vs vectorized={len(comp)}")
else:
    same = (naive["airing_idx"] == comp["airing_idx"]) & (naive["overlap_sec"] == comp["overlap_sec"])
    mismatches = int((~same).sum())
    print(f"pair-by-pair mismatches: {mismatches} / {len(naive)}")
    for i in naive.index[~same][:10]:
        r = naive.loc[i]
        cr = comp.loc[i]
        print(f"  MISMATCH view_key={r['view_key']} naive=(airing={r['airing_idx']}, overlap={r['overlap_sec']}) vectorized=(airing={cr['airing_idx']}, overlap={cr['overlap_sec']})")


PART A revisit — naive vs vectorized matching (non-empty sample)
busiest channel: 3 | views on it: 589,869
sampled matched views: 2000
naive pairs: 3,052 | vectorized pairs for the same views: 3,052
pair-by-pair mismatches: 0 / 3052


### Interaction table — filter used
`data/interim/interactions.parquet` is built from **labelled matched pairs only**: `tv_show_id != 0`. Since the matching rule already requires `overlap_sec > 0 s`, no additional overlap threshold was applied; that is why `interactions.parquet` has 4,895 users and 4,363 shows.
Why this differs from the sensitivity table: the `>0 s` row (4,912 users / 4,364 shows) was computed on **all** matched pairs, including airings with `tv_show_id == 0` (88.9% of schedule rows). Excluding id-0 airings drops 17 users and 1 show.
Going forward, interactions are (re)built from `data/interim/matched_pairs.parquet` with configurable parameters `min_overlap_sec` and `min_overlap_frac` (defaults 0 / 0.0 below), so the threshold can change without re-running the matching.


In [56]:
# --- build interactions from matched_pairs.parquet (configurable filter) ---
from pathlib import Path
MIN_OVERLAP_SEC = 0
MIN_OVERLAP_FRAC = 0.0

pairs_file = DATA_DIR / "interim" / "matched_pairs.parquet"
pairs_all = pd.read_parquet(pairs_file)
lab_all = pairs_all[pairs_all["tv_show_id"] != 0].copy()
mask = (lab_all["overlap_sec"] >= MIN_OVERLAP_SEC) & (lab_all["overlap_frac"] >= MIN_OVERLAP_FRAC)
inter_pairs = lab_all[mask]
inter = inter_pairs.groupby(["user_id", "tv_show_id"]).agg(
    n_airings_watched=("overlap_sec", "size"),
    total_seconds=("overlap_sec", "sum"),
    first_time=("view_start_dt", "min"),
    last_time=("view_stop_dt", "max"),
    n_distinct_days=("view_start_dt", lambda s: s.dt.date.nunique()),
).reset_index()
print(f"interactions: min_overlap_sec={MIN_OVERLAP_SEC}, min_overlap_frac={MIN_OVERLAP_FRAC}")
print(f"  labelled pairs: {len(lab_all):,} | kept pairs: {len(inter_pairs):,} | rows: {len(inter):,} | users: {inter['user_id'].nunique():,} | shows: {inter['tv_show_id'].nunique():,}")
print("  note: the sensitivity-table '>0 s' row counted id-0 airings as well; here they are excluded, so the counts above are the ones that apply")


interactions: min_overlap_sec=0, min_overlap_frac=0.0
  labelled pairs: 3,847,696 | kept pairs: 3,847,696 | rows: 1,371,685 | users: 4,895 | shows: 4,363
  note: the sensitivity-table '>0 s' row counted id-0 airings as well; here they are excluded, so the counts above are the ones that apply


### Known limitation (from Part A)
13 schedule transitions have consecutive programs whose airings overlap (overlap=13 out of 3,444,006 transitions). A view spanning such a transition matches both airings, so its watched time is counted twice. This affects a negligible share of pairs; not fixed in this step.


## 4. Temporal validation split (no future leakage)


### build_split — how the temporal split works
- History: matched pairs whose **view start** falls in `hist_weeks` (ISO week of `start_dt`). Target: matched pairs whose **view start** falls in `target_weeks`. No target-window interaction is used for training or ranking.
- The **target schedule is allowed**: a TV schedule is published in advance, so using the target window's schedule (to build the candidate pool) is not future leakage.
- Candidate pool: every `tv_show_id != 0` with at least one airing in the target window.
- Evaluation users: the 1,260 `user_id`s of `submission.csv`, identical for every method.
- Fold A (primary): history weeks 11–20 → target weeks 21–30.
- Final setup (sizes only): history weeks 11–30 → predict weeks 31–42 (schedule 31–42 supplies the candidate pool; the viewing log ends at week 30, so target ground truth is empty there).
- Optional Fold B (robustness): history 11–18 → target weeks 19–28.


In [57]:
# --- Section 4: temporal split ---
import numpy as np
import pandas as pd
import statistics

# labelled interaction pool (tv_show_id != 0); overlap_sec > 0 s is guaranteed by the matching rule
lab = pairs[pairs["tv_show_id"] != 0].copy()

EVAL_USERS = sorted(set(sub["user_id"].unique()))
print(f"evaluation users (submission.csv): {len(EVAL_USERS):,}")

def build_split(hist_weeks, target_weeks):
    hist_weeks = list(hist_weeks)
    target_weeks = list(target_weeks)
    hist = lab[lab["iso_week"].isin(hist_weeks)].copy()
    target = lab[lab["iso_week"].isin(target_weeks)].copy()
    sched_weeks = schedule_nd["start_dt"].dt.isocalendar().week
    cand = np.sort(
        schedule_nd.loc[
            sched_weeks.isin(target_weeks) & (schedule_nd["tv_show_id"] != 0),
            "tv_show_id",
        ].unique()
    )
    return hist, target, cand

def gt1_from_target(target_df):
    g = target_df.groupby(["user_id", "tv_show_id"]).agg(
        total_seconds=("overlap_sec", "sum"), n_airings=("overlap_sec", "size")
    ).reset_index()
    g = g.sort_values(["user_id", "total_seconds", "n_airings", "tv_show_id"],
                      ascending=[True, False, False, True])
    out = {}
    for u, df in g.groupby("user_id"):
        out[u] = set(df.head(5)["tv_show_id"].tolist())
    return out

def gt2_from_target(target_df, frac=0.5):
    q = target_df[target_df["overlap_frac"] >= frac]
    return q.groupby("user_id")["tv_show_id"].apply(set).to_dict()

# folds
histA, targetA, candA = build_split(range(11, 21), range(21, 31))
histF, targetF, candF = build_split(range(11, 31), range(31, 43))
histB, targetB, candB = build_split(range(11, 19), range(19, 29))

print(f"[Fold A] hist weeks 11-20: pairs={len(histA):,} | target weeks 21-30: pairs={len(targetA):,} | candidate pool={len(candA):,}")
print(f"[Final ] hist weeks 11-30: pairs={len(histF):,} | target weeks 31-42: pairs={len(targetF):,} (log ends at week 30 -> no target views) | candidate pool={len(candF):,}")
print(f"[Fold B] hist weeks 11-18: pairs={len(histB):,} | target weeks 19-28: pairs={len(targetB):,} | candidate pool={len(candB):,}")

# ground truths for Fold A, restricted to evaluation users
gt1A = gt1_from_target(targetA)
gt2A = gt2_from_target(targetA)
gt1_eval = {u: gt1A.get(u, set()) for u in EVAL_USERS}
gt2_eval = {u: gt2A.get(u, set()) for u in EVAL_USERS}
nz1 = [len(s) for s in gt1_eval.values() if s]
nz2 = [len(s) for s in gt2_eval.values() if s]
for name, sizes in [("GT-1", nz1), ("GT-2", nz2)]:
    print(f"[Fold A] {name}: users with non-empty GT={len(sizes):,} | empty={len(EVAL_USERS)-len(sizes):,} | |R_u| min={min(sizes)} median={statistics.median(sizes)} max={max(sizes)}")

# diagnostics for Fold A
hist_shows_by_user = histA.groupby("user_id")["tv_show_id"].apply(set).to_dict()
global_hist_shows = set(histA["tv_show_id"].unique())
repeat_rates = [
    len(gt1A[u] & hist_shows_by_user.get(u, set())) / len(gt1A[u])
    for u in EVAL_USERS if u in gt1A and gt1A[u]
]
instances = sum(len(s) for s in gt1_eval.values() if s)
new_instances = sum(len(s - global_hist_shows) for s in gt1_eval.values() if s)
empty_hist = [u for u in EVAL_USERS if u not in hist_shows_by_user]
print(f"[Fold A diag] (a) mean repeat rate over users with non-empty GT-1 (share of GT-1 shows already watched in history): {np.mean(repeat_rates):.4f}")
print(f"[Fold A diag] (b) GT-1 (user,show) instances never in anyone's history: {new_instances}/{instances} = {new_instances/instances:.4f}")
print(f"[Fold A diag] (c) evaluation users with empty history: {len(empty_hist):,}")


evaluation users (submission.csv): 1,260
[Fold A] hist weeks 11-20: pairs=2,383,835 | target weeks 21-30: pairs=1,463,861 | candidate pool=6,376
[Final ] hist weeks 11-30: pairs=3,847,696 | target weeks 31-42: pairs=0 (log ends at week 30 -> no target views) | candidate pool=6,642
[Fold B] hist weeks 11-18: pairs=1,951,571 | target weeks 19-28: pairs=1,468,025 | candidate pool=6,477
[Fold A] GT-1: users with non-empty GT=1,246 | empty=14 | |R_u| min=3 median=5.0 max=5
[Fold A] GT-2: users with non-empty GT=1,244 | empty=16 | |R_u| min=1 median=35.5 max=324
[Fold A diag] (a) mean repeat rate over users with non-empty GT-1 (share of GT-1 shows already watched in history): 0.6935
[Fold A diag] (b) GT-1 (user,show) instances never in anyone's history: 695/6227 = 0.1116
[Fold A diag] (c) evaluation users with empty history: 10


### Part C — verification

In [58]:
# --- Part C: verification ---
import numpy as np
rng = np.random.default_rng(20260309)

print("=" * 70)
print("PART C — verification")
print("=" * 70)

# C1: 3 random views with matched labelled airings, printed for manual recompute
cand = pairs[pairs["tv_show_id"] != 0]["view_key"].unique()
sel = rng.choice(cand, size=3, replace=False)
for vk in sel:
    v = train.loc[vk]
    ps = pairs[(pairs["view_key"] == vk) & (pairs["tv_show_id"] != 0)].sort_values("air_start_dt")
    print(f"\nVIEW key={vk}  user={v['user_id']}  channel(vsetv_id)={v['vsetv_id_int']}")
    print(f"  view interval: {v['start_dt']} -> {v['stop_dt']}   duration={v['duraton']} s")
    for _, a in ps.iterrows():
        ov = (min(v["stop_dt"], a["air_end_dt"]) - max(v["start_dt"], a["air_start_dt"])).total_seconds()
        print(f"  airing: {a['air_start_dt']} -> {a['air_end_dt']}  id={a['tv_show_id']}  dur={a['airing_dur_sec']}s  title={a['tv_show_title']!r}")
        print(f"      overlap = min({v['stop_dt']}, {a['air_end_dt']}) - max({v['start_dt']}, {a['air_start_dt']}) = {ov} s  (stored {a['overlap_sec']} s)")

# Use the ACTUAL view duration (computed from start/stop) for the assertions. The stored
# 'duraton' column is understated for 181 views (Part A), so comparing overlap -- computed
# from actual times -- to it would under-bound overlap. Actual duration is the true view
# duration and makes the interval-overlap properties hold by construction.
pairs = pairs.copy()
pairs["view_dur_sec"] = (pairs["view_stop_dt"] - pairs["view_start_dt"]).dt.total_seconds().astype("int64")

# C2: assertions
print("\n-- assertions --")
a1 = bool((pairs["overlap_sec"] > 0).all())
a2 = bool((pairs["overlap_sec"] <= np.minimum(pairs["view_dur_sec"], pairs["airing_dur_sec"])).all())
view_sum = pairs.groupby("view_key")["overlap_sec"].sum()
view_dur = pairs.groupby("view_key")["view_dur_sec"].first()
a3 = bool((view_sum <= view_dur).all())
exceed = view_sum[view_sum > view_dur]
print(f"  (1) every matched pair has overlap_sec > 0: {a1}")
print(f"  (2) overlap_sec <= min(view duration, airing duration) for every pair: {a2}")
print(f"  (3) sum of overlap_sec per view <= view duration: {a3}")
print(f"      views where sum exceeds view duration: {len(exceed)} (expected: these span one of the 13 overlapping schedule-airing transitions, so a view's time is matched to two airings and the overlap region is counted twice)")

# C3: cold-start among the 1260 submission users (labelled interaction in weeks 21-30)
sub_users = set(sub["user_id"].unique())
labelled_w21_30 = pairs[(pairs["tv_show_id"] != 0) & (pairs["iso_week"].between(21, 30))]
users_w21_30 = set(labelled_w21_30["user_id"].unique())
have = users_w21_30 & sub_users
print("\n-- cold start (weeks 21-30) --")
print(f"  submission users: {len(sub_users):,}")
print(f"  submission users with >=1 labelled interaction in weeks 21-30: {len(have):,}")
print(f"  cold-start users (no labelled interaction weeks 21-30): {len(sub_users - have):,}")


PART C — verification

VIEW key=3298765  user=14892587152266602908  channel(vsetv_id)=3
  view interval: 2020-05-02 21:29:59 -> 2020-05-02 21:31:53   duration=114 s
  airing: 2020-05-02 20:25:00 -> 2020-05-02 21:30:00  id=5900348  dur=3900s  title="'Украинские сенсации'."
      overlap = min(2020-05-02 21:31:53, 2020-05-02 21:30:00) - max(2020-05-02 21:29:59, 2020-05-02 20:25:00) = 1.0 s  (stored 1 s)
  airing: 2020-05-02 21:30:00 -> 2020-05-02 23:10:00  id=200352  dur=6000s  title="'Вечерний квартал'."
      overlap = min(2020-05-02 21:31:53, 2020-05-02 23:10:00) - max(2020-05-02 21:29:59, 2020-05-02 21:30:00) = 113.0 s  (stored 113 s)

VIEW key=1170882  user=17676810254267048747  channel(vsetv_id)=7
  view interval: 2020-03-27 19:51:46 -> 2020-03-27 19:53:17   duration=91 s
  airing: 2020-03-27 19:00:00 -> 2020-03-27 21:05:00  id=90073871  dur=7500s  title="Х/ф 'Сумерки. Сага. Рассвет. Часть 1'."
      overlap = min(2020-03-27 19:53:17, 2020-03-27 21:05:00) - max(2020-03-27 19:51:46,

## 5. Metric: MAP@5 (own implementation + hand-check on 1 user)


### AP@5 / MAP@5 (own implementation, per course slide)
AP_u@K = sum_{r=1..K} P_u@r * rel_u(r) / min(K, |R_u|), where rel_u(r)=1 if the show at rank r is in the user's relevant set R_u. MAP = mean of AP over users. No library functions are used for the metric.


In [59]:
# --- Section 5: MAP@5 (own implementation) ---
import time

def ap_at5(recommended, relevant, K=5):
    rel = 0
    prec_sum = 0.0
    for r, show in enumerate(recommended[:K], start=1):
        if show in relevant:
            rel += 1
            prec_sum += rel / r
    denom = min(K, len(relevant))
    return prec_sum / denom if denom > 0 else 0.0

def map_at5(recs_by_user, gts_by_user, K=5, count_empty_as_zero=False):
    aps = []
    n_empty = 0
    for u, rec in recs_by_user.items():
        rel = gts_by_user.get(u, set())
        if not rel:
            n_empty += 1
            if count_empty_as_zero:
                aps.append(0.0)
            continue
        aps.append(ap_at5(rec, rel, K))
    return (float(np.mean(aps)) if aps else 0.0, n_empty)

# --- unit tests: course slide example ---
ap_x = ap_at5(["A", "B", "D", "C", "E"], {"A", "B", "C"})
ap_y = ap_at5(["D", "A", "E", "B", "C"], {"A", "B", "C"})
assert abs(ap_x - 0.9166666666666666) < 1e-9
assert abs(ap_y - 0.5333333333333333) < 1e-9
print(f"slide unit tests passed: Method X AP@5 = {ap_x:.4f} = (1 + 1 + 0.75) / 3; Method Y AP@5 = {ap_y:.4f} = (0.5 + 0.5 + 0.6) / 3")

# --- hand-check: one fixed evaluation user ---
rng_hc = np.random.default_rng(123)
hc_users = [u for u in EVAL_USERS if gt1_eval.get(u)]
hc_u = rng_hc.choice(hc_users)
hc_rel = sorted(int(s) for s in gt1_eval[hc_u])
cand_not_rel = [int(s) for s in candA if int(s) not in gt1_eval[hc_u]]
hc_toy = [int(s) for s in hc_rel[:3]] + [int(s) for s in rng_hc.choice(cand_not_rel, size=2, replace=False)]
rel_set = set(hc_rel)
running = 0
terms = []
for r, s in enumerate(hc_toy, start=1):
    if s in rel_set:
        running += 1
        terms.append((r, s, running, running / r))
ap_hand = sum(t[3] for t in terms) / min(5, len(rel_set))
print(f"hand-check user={hc_u} | GT-1={hc_rel} | toy top-5={hc_toy}")
print(f"P@r terms (rank, show, rel_count, P@r): {terms}")
print(f"hand-check AP@5 = {ap_hand:.6f}  (sum of P@r terms / min(5, |R|)={min(5, len(rel_set))})")

# --- metric runtime on 1,260 users ---
t0 = time.time()
dummy_recs = {u: [int(s) for s in candA[:5]] for u in EVAL_USERS}
m_dummy, n_e_dummy = map_at5(dummy_recs, gt1_eval)
print(f"MAP@5 runtime on {len(EVAL_USERS)} users: {time.time() - t0:.4f} s")


slide unit tests passed: Method X AP@5 = 0.9167 = (1 + 1 + 0.75) / 3; Method Y AP@5 = 0.5333 = (0.5 + 0.5 + 0.6) / 3
hand-check user=10178643848749863023 | GT-1=[20088, 2400480, 2400508, 6500479, 10002937] | toy top-5=[20088, 2400480, 2400508, 90073836, 90076928]
P@r terms (rank, show, rel_count, P@r): [(1, 20088, 1, 1.0), (2, 2400480, 2, 1.0), (3, 2400508, 3, 1.0)]
hand-check AP@5 = 0.600000  (sum of P@r terms / min(5, |R|)=5)
MAP@5 runtime on 1260 users: 0.0022 s


## 6. Baseline: popularity


In [60]:
# --- Section 6: baselines on Fold A ---
import time

K = 5
cand_pool = [int(s) for s in candA]
cand_set = set(cand_pool)

# popularity rankings restricted to the candidate pool
pop_users = histA.groupby("tv_show_id")["user_id"].nunique().sort_values(ascending=False)
pop_secs = histA.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False)
hist_recent = histA[histA["iso_week"].isin([19, 20])]
pop_users_recent = hist_recent.groupby("tv_show_id")["user_id"].nunique().sort_values(ascending=False)

def top5_in_pool(series):
    return [int(s) for s in series.index if int(s) in cand_set][:K]

pop5_users = top5_in_pool(pop_users)
pop5_secs = top5_in_pool(pop_secs)
pop5_recent = top5_in_pool(pop_users_recent)
assert len(pop5_users) == K and len(pop5_secs) == K and len(pop5_recent) == K

def rec_random(seed):
    rng = np.random.default_rng(seed)
    return {u: [int(s) for s in rng.choice(cand_pool, size=K, replace=False)] for u in EVAL_USERS}

def check_recs(recs, name):
    ok = True
    for u, r in recs.items():
        if len(r) != K or len(set(r)) != K or 0 in set(r) or not set(r) <= cand_set:
            ok = False
            print(f"  ASSERT FAILED {name} user={u}: {r}")
    return ok

all_ok = True
rows = []

def eval_method(name, recs):
    global all_ok
    all_ok &= check_recs(recs, name)
    m1, e1 = map_at5(recs, gt1_eval)
    m1z, _ = map_at5(recs, gt1_eval, count_empty_as_zero=True)
    m2, e2 = map_at5(recs, gt2_eval)
    m2z, _ = map_at5(recs, gt2_eval, count_empty_as_zero=True)
    rows.append((name, m1, m2, m1z, m2z, len(EVAL_USERS) - e1, len(EVAL_USERS) - e2))
    print(f"{name:34s} | GT-1: {m1:.6f} ({len(EVAL_USERS)-e1:,} users) | empty=0: {m1z:.6f} | GT-2: {m2:.6f} ({len(EVAL_USERS)-e2:,} users) | empty=0: {m2z:.6f}")

t0 = time.time()

# --- Random: average of 5 fixed seeds ---
r1 = r1z = r2 = r2z = 0.0
for s in range(5):
    recs = rec_random(1000 + s)
    if s == 0:
        all_ok &= check_recs(recs, "Random seed 1000")
    a, _ = map_at5(recs, gt1_eval); r1 += a
    b, _ = map_at5(recs, gt1_eval, count_empty_as_zero=True); r1z += b
    c, _ = map_at5(recs, gt2_eval); r2 += c
    d, _ = map_at5(recs, gt2_eval, count_empty_as_zero=True); r2z += d
rows.append(("Random (avg 5 seeds)", r1 / 5, r2 / 5, r1z / 5, r2z / 5, len(EVAL_USERS), len(EVAL_USERS)))
print(f"{'Random (avg 5 seeds)':34s} | GT-1: {r1/5:.6f} ({len(EVAL_USERS):,} users) | empty=0: {r1z/5:.6f} | GT-2: {r2/5:.6f} ({len(EVAL_USERS):,} users) | empty=0: {r2z/5:.6f}")

# --- Popularity-by-users ---
recs_pop_users = {u: pop5_users for u in EVAL_USERS}
eval_method("Popularity-by-users", recs_pop_users)

# --- Popularity-by-seconds ---
recs_pop_secs = {u: pop5_secs for u in EVAL_USERS}
eval_method("Popularity-by-seconds", recs_pop_secs)

# --- Recency popularity (last 2 weeks of history: weeks 19-20) ---
recs_pop_recent = {u: pop5_recent for u in EVAL_USERS}
eval_method("Recency-popularity (hist w19-20)", recs_pop_recent)

# --- Personal history: diagnostic only, NOT a counted method ---
u_sec = histA.groupby(["user_id", "tv_show_id"])["overlap_sec"].sum()
recs_personal = {}
for u in EVAL_USERS:
    try:
        own = [int(s) for s in u_sec.xs(u, level=0).sort_values(ascending=False).index if int(s) in cand_set]
    except KeyError:
        own = []
    own = own[:K]
    if len(own) < K:
        own += [int(s) for s in pop5_users if int(s) not in set(own)][:K - len(own)]
    recs_personal[u] = own
eval_method("Personal history [diagnostic, not counted]", recs_personal)

print(f"\nasserts: length {K}, no tv_show_id==0, no duplicates, all in candidate pool -> {'ALL PASSED' if all_ok else 'FAILED'}")
print(f"baselines runtime: {time.time() - t0:.2f} s")

print("\n--- summary table: baseline x MAP@5 ---")
print(f"{'method':34s} | {'MAP@5 GT-1':>10s} | {'MAP@5 GT-2':>10s} | {'MAP@5 GT-1 empty=0':>18s} | {'MAP@5 GT-2 empty=0':>18s}")
for name, m1, m2, m1z, m2z, nu1, nu2 in rows:
    print(f"{name:34s} | {m1:10.6f} | {m2:10.6f} | {m1z:18.6f} | {m2z:18.6f}")


Random (avg 5 seeds)               | GT-1: 0.000506 (1,260 users) | empty=0: 0.000501 | GT-2: 0.003538 (1,260 users) | empty=0: 0.003493
Popularity-by-users                | GT-1: 0.177103 (1,246 users) | empty=0: 0.175135 | GT-2: 0.553070 (1,244 users) | empty=0: 0.546047
Popularity-by-seconds              | GT-1: 0.180193 (1,246 users) | empty=0: 0.178190 | GT-2: 0.541744 (1,244 users) | empty=0: 0.534865
Recency-popularity (hist w19-20)   | GT-1: 0.178793 (1,246 users) | empty=0: 0.176807 | GT-2: 0.542471 (1,244 users) | empty=0: 0.535582
Personal history [diagnostic, not counted] | GT-1: 0.301359 (1,246 users) | empty=0: 0.298011 | GT-2: 0.510529 (1,244 users) | empty=0: 0.504046

asserts: length 5, no tv_show_id==0, no duplicates, all in candidate pool -> ALL PASSED
baselines runtime: 1.05 s

--- summary table: baseline x MAP@5 ---
method                             | MAP@5 GT-1 | MAP@5 GT-2 | MAP@5 GT-1 empty=0 | MAP@5 GT-2 empty=0
Random (avg 5 seeds)               |   0.000506 

### Part 0 — loose ends from step 4

Items 1–3 below close out step 4. Item 3 re-runs the step-6 baselines on Fold B (history weeks 11–18 → target weeks 19–28) for a later robustness check.


### Part 0 — item 1: cross-check

The naive-vs-vectorized cross-check is executed and printed inside the matching cell (section 3, Part B). It is not repeated here; no numbers are restated in this note.


In [61]:
# --- Part 0 item 2: interaction filter note (all numbers recomputed here) ---
print("interactions.parquet filter: labelled matched pairs only (tv_show_id != 0),")
print("min_overlap_sec=0, min_overlap_frac=0.0 applied on top (no further filtering).")
print(f"Rows: {len(inter):,} | users: {inter['user_id'].nunique():,} | shows: {inter['tv_show_id'].nunique():,}")
zero_share = float((schedule_nd["tv_show_id"] == 0).mean()) * 100.0
print(f"Why this differs from the sensitivity-table '>0 s' row: that table counted ALL matched pairs,")
print(f"including airings with tv_show_id == 0 ({zero_share:.1f}% of schedule rows).")
n_u_all, n_u_lab = pairs_all["user_id"].nunique(), lab_all["user_id"].nunique()
n_s_all, n_s_lab = pairs_all["tv_show_id"].nunique(), lab_all["tv_show_id"].nunique()
print(f"Excluding id-0 airings changes: users {n_u_all:,} -> {n_u_lab:,} | shows {n_s_all:,} -> {n_s_lab:,}")
print("Going forward: interactions are rebuilt from data/interim/matched_pairs.parquet with")
print("configurable min_overlap_sec / min_overlap_frac (defaults 0 / 0.0).")


interactions.parquet filter: labelled matched pairs only (tv_show_id != 0),
min_overlap_sec=0, min_overlap_frac=0.0 applied on top (no further filtering).
Rows: 1,371,685 | users: 4,895 | shows: 4,363
Why this differs from the sensitivity-table '>0 s' row: that table counted ALL matched pairs,
including airings with tv_show_id == 0 (88.9% of schedule rows).
Excluding id-0 airings changes: users 4,912 -> 4,895 | shows 4,364 -> 4,363
Going forward: interactions are rebuilt from data/interim/matched_pairs.parquet with
configurable min_overlap_sec / min_overlap_frac (defaults 0 / 0.0).


In [62]:
# --- Part 0 item 3: baselines on Fold B (robustness) ---
import time
import statistics

K = 5
candB_pool = [int(s) for s in candB]
candB_set = set(candB_pool)

def gt1_from_target_B(target_df):
    g = target_df.groupby(["user_id", "tv_show_id"]).agg(
        total_seconds=("overlap_sec", "sum"), n_airings=("overlap_sec", "size")
    ).reset_index()
    g = g.sort_values(["user_id", "total_seconds", "n_airings", "tv_show_id"],
                      ascending=[True, False, False, True])
    out = {}
    for u, df in g.groupby("user_id"):
        out[u] = set(df.head(5)["tv_show_id"].tolist())
    return out

def gt2_from_target_B(target_df, frac=0.5):
    q = target_df[target_df["overlap_frac"] >= frac]
    return q.groupby("user_id")["tv_show_id"].apply(set).to_dict()

gt1_B_raw = gt1_from_target_B(targetB)
gt2_B_raw = gt2_from_target_B(targetB)
gt1_B = {u: gt1_B_raw.get(u, set()) for u in EVAL_USERS}
gt2_B = {u: gt2_B_raw.get(u, set()) for u in EVAL_USERS}

nz1B = [len(s) for s in gt1_B.values() if s]
nz2B = [len(s) for s in gt2_B.values() if s]
print(f"Fold B GT-1: non-empty={len(nz1B):,} empty={len(EVAL_USERS)-len(nz1B):,} |R_u| min={min(nz1B)} median={statistics.median(nz1B)} max={max(nz1B)}")
print(f"Fold B GT-2: non-empty={len(nz2B):,} empty={len(EVAL_USERS)-len(nz2B):,} |R_u| min={min(nz2B)} median={statistics.median(nz2B)} max={max(nz2B)}")

pop_users_B = histB.groupby("tv_show_id")["user_id"].nunique().sort_values(ascending=False)
pop_secs_B = histB.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False)
hist_recent_B = histB[histB["iso_week"].isin([17, 18])]
pop_users_recent_B = hist_recent_B.groupby("tv_show_id")["user_id"].nunique().sort_values(ascending=False)

def top5_in_pool_B(series):
    return [int(s) for s in series.index if int(s) in candB_set][:K]

pop5_users_B = top5_in_pool_B(pop_users_B)
pop5_secs_B = top5_in_pool_B(pop_secs_B)
pop5_recent_B = top5_in_pool_B(pop_users_recent_B)
assert len(pop5_users_B) == K and len(pop5_secs_B) == K and len(pop5_recent_B) == K

def rec_random_B(seed):
    rng = np.random.default_rng(seed)
    return {u: [int(s) for s in rng.choice(candB_pool, size=K, replace=False)] for u in EVAL_USERS}

def check_recs_B(recs, name):
    ok = True
    for u, r in recs.items():
        if len(r) != K or len(set(r)) != K or 0 in set(r) or not set(r) <= candB_set:
            ok = False
            print(f"  ASSERT FAILED {name} user={u}: {r}")
    return ok

all_ok_B = True
rows_B = []

def eval_method_B(name, recs):
    global all_ok_B
    all_ok_B &= check_recs_B(recs, name)
    m1, e1 = map_at5(recs, gt1_B)
    m1z, _ = map_at5(recs, gt1_B, count_empty_as_zero=True)
    m2, e2 = map_at5(recs, gt2_B)
    m2z, _ = map_at5(recs, gt2_B, count_empty_as_zero=True)
    rows_B.append((name, m1, m2, m1z, m2z, len(EVAL_USERS) - e1, len(EVAL_USERS) - e2))
    print(f"{name:34s} | GT-1: {m1:.6f} ({len(EVAL_USERS)-e1:,} users) | empty=0: {m1z:.6f} | GT-2: {m2:.6f} ({len(EVAL_USERS)-e2:,} users) | empty=0: {m2z:.6f}")

t0 = time.time()

r1 = r1z = r2 = r2z = 0.0
for s in range(5):
    recs = rec_random_B(1000 + s)
    if s == 0:
        all_ok_B &= check_recs_B(recs, "Random seed 1000")
    a, _ = map_at5(recs, gt1_B); r1 += a
    b, _ = map_at5(recs, gt1_B, count_empty_as_zero=True); r1z += b
    c, _ = map_at5(recs, gt2_B); r2 += c
    d, _ = map_at5(recs, gt2_B, count_empty_as_zero=True); r2z += d
rows_B.append(("Random (avg 5 seeds)", r1 / 5, r2 / 5, r1z / 5, r2z / 5,
               len(EVAL_USERS), len(EVAL_USERS)))
print(f"{'Random (avg 5 seeds)':34s} | GT-1: {r1/5:.6f} ({len(EVAL_USERS):,} users) | empty=0: {r1z/5:.6f} | GT-2: {r2/5:.6f} ({len(EVAL_USERS):,} users) | empty=0: {r2z/5:.6f}")

recs_pop_users_B = {u: pop5_users_B for u in EVAL_USERS}
eval_method_B("Popularity-by-users", recs_pop_users_B)
recs_pop_secs_B = {u: pop5_secs_B for u in EVAL_USERS}
eval_method_B("Popularity-by-seconds", recs_pop_secs_B)
recs_pop_recent_B = {u: pop5_recent_B for u in EVAL_USERS}
eval_method_B("Recency-popularity (hist w17-18)", recs_pop_recent_B)

u_sec_B = histB.groupby(["user_id", "tv_show_id"])["overlap_sec"].sum()
recs_personal_B = {}
for u in EVAL_USERS:
    try:
        own = [int(s) for s in u_sec_B.xs(u, level=0).sort_values(ascending=False).index if int(s) in candB_set]
    except KeyError:
        own = []
    own = own[:K]
    if len(own) < K:
        own += [int(s) for s in pop5_users_B if int(s) not in set(own)][:K - len(own)]
    recs_personal_B[u] = own
eval_method_B("Personal history [diagnostic, not counted]", recs_personal_B)

print(f"\nFold B asserts: length {K}, no tv_show_id==0, no duplicates, all in candidate pool -> {'ALL PASSED' if all_ok_B else 'FAILED'}")
print(f"Fold B baselines runtime: {time.time() - t0:.2f} s")
print("\n--- Fold B summary table ---")
print(f"{'method':34s} | {'MAP@5 GT-1':>10s} | {'MAP@5 GT-2':>10s} | {'MAP@5 GT-1 empty=0':>18s} | {'MAP@5 GT-2 empty=0':>18s}")
for name, m1, m2, m1z, m2z, nu1, nu2 in rows_B:
    print(f"{name:34s} | {m1:10.6f} | {m2:10.6f} | {m1z:18.6f} | {m2z:18.6f}")


Fold B GT-1: non-empty=1,246 empty=14 |R_u| min=3 median=5.0 max=5
Fold B GT-2: non-empty=1,240 empty=20 |R_u| min=1 median=32.0 max=322
Random (avg 5 seeds)               | GT-1: 0.000410 (1,260 users) | empty=0: 0.000405 | GT-2: 0.003508 (1,260 users) | empty=0: 0.003452
Popularity-by-users                | GT-1: 0.188676 (1,246 users) | empty=0: 0.186579 | GT-2: 0.552938 (1,240 users) | empty=0: 0.544161
Popularity-by-seconds              | GT-1: 0.192483 (1,246 users) | empty=0: 0.190344 | GT-2: 0.541780 (1,240 users) | empty=0: 0.533180
Recency-popularity (hist w17-18)   | GT-1: 0.165599 (1,246 users) | empty=0: 0.163759 | GT-2: 0.554387 (1,240 users) | empty=0: 0.545587
Personal history [diagnostic, not counted] | GT-1: 0.303652 (1,246 users) | empty=0: 0.300278 | GT-2: 0.504927 (1,240 users) | empty=0: 0.496912

Fold B asserts: length 5, no tv_show_id==0, no duplicates, all in candidate pool -> ALL PASSED
Fold B baselines runtime: 0.99 s

--- Fold B summary table ---
method     

## 7. Method A


### Section 7 — implicit ALS (Method A)

Approach: implicit matrix factorization (alternating least squares) on user–show confidence derived from watched seconds. Confidence for a (user, show) pair is C = 1 + alpha * log1p(total watched seconds in the history window). The model learns user and show factor vectors; a recommendation score is the dot product user_vector · show_vector, computed only for shows in the candidate pool (tv_show_id != 0, airing in the target window). Shows in the pool with no embedding (never watched in history) receive no score. Users with no history fall back to popularity-by-seconds. Already-watched shows are NOT removed from the ranking (repeat consumption is the norm in TV; 69% of GT-1 shows were watched before); an "exclude already watched" variant is also scored for comparison.

Alternatives considered and why not:
- Content-based on show title/genre: titles are free-form Russian text with heavy missingness (genres missing for 57-97% of rows); not reliable enough for a first method.
- Deep sequence models (GRU4Rec, SASRec): no GPU guaranteed here, long training, overkill for a baseline-strength first method.
- Item-item CF on co-watch: views are sparse per user; user-based signal is weaker but still usable via implicit feedback.

Objective (Hu, Koren, Volinsky, 'Collaborative Filtering for Implicit Feedback Datasets', ICDM 2008):
min_{x,y} sum_{u,i} c_ui (p_ui - x_u^T y_i)^2 + lambda_reg (sum_u ||x_u||^2 + sum_i ||y_i||^2),
with p_ui in {0,1} (watched / not watched) and confidence c_ui = 1 + alpha * log1p(watched seconds).
Reference: I am confident about this title/authors/venue/year (Hu, Koren, Volinsky; ICDM 2008).


In [63]:
# --- Section 7 setup: implicit ALS via the `implicit` library ---
import time
import inspect
import numpy as np
import pandas as pd
import scipy.sparse as sp

import implicit
print(f"implicit version: {implicit.__version__}")
from implicit.cpu.als import AlternatingLeastSquares

# --- verify how `implicit` applies `alpha` (do not guess) ---
src_lines = inspect.getsource(AlternatingLeastSquares.fit).splitlines()
print("---- implicit.cpu.als.AlternatingLeastSquares.fit source (head) ----")
for line in src_lines[:14]:
    print(line)
print("---------------------------------------------------------------")

RNG_SEED = 42
MIN_OVERLAP_SEC = 60
MIN_OVERLAP_FRAC = 0.0
ALPHA = 40.0
# NOTE (verified in implicit source): implicit multiplies the passed confidence matrix by model.alpha.
# We therefore pass C = 1 + alpha_t*log1p(sec) directly as matrix values and set model.alpha=1.0.

def build_conf_matrix(lab_df, alpha):
    """Confidence = 1 + alpha * log1p(total_seconds) per (user, show)."""
    conf = (
        lab_df.groupby(["user_id", "tv_show_id"])["overlap_sec"]
        .sum()
        .reset_index(name="total_seconds")
    )
    conf["confidence"] = 1.0 + alpha * np.log1p(conf["total_seconds"].astype("float64"))
    users = sorted(conf["user_id"].unique())
    items = sorted(conf["tv_show_id"].unique())
    u_pos = {u: i for i, u in enumerate(users)}
    i_pos = {i: j for j, i in enumerate(items)}
    mat = sp.csr_matrix(
        (
            conf["confidence"].astype(np.float32),
            (
                conf["user_id"].map(u_pos).values.astype(np.int32),
                conf["tv_show_id"].map(i_pos).values.astype(np.int32),
            ),
        ),
        shape=(len(users), len(items)),
    )
    return mat, u_pos, i_pos

# Labelled interactions, history window only (Fold A: weeks 11-20)
labA = pairs[
    (pairs["tv_show_id"] != 0)
    & (pairs["overlap_sec"] >= MIN_OVERLAP_SEC)
    & (pairs["overlap_frac"] >= MIN_OVERLAP_FRAC)
    & (pairs["iso_week"].between(11, 20))
].copy()
print(f"Fold A labelled interactions (min_overlap_sec={MIN_OVERLAP_SEC}): {len(labA):,} pairs")
print(f"  users: {labA['user_id'].nunique():,} | shows: {labA['tv_show_id'].nunique():,}")

user_items_A, u_posA, i_posA = build_conf_matrix(labA, ALPHA)
print(f"Fold A history matrix: {user_items_A.shape[0]:,} users x {user_items_A.shape[1]:,} items, nnz={user_items_A.nnz:,}")
print(f"eval users with history: {sum(u in u_posA for u in EVAL_USERS):,} / {len(EVAL_USERS):,}")
print(f"candidate-pool shows with embedding: {sum(int(s) in i_posA for s in candA):,} / {len(candA):,}")


implicit version: 0.7.3
---- implicit.cpu.als.AlternatingLeastSquares.fit source (head) ----
    def fit(self, user_items, show_progress=True, callback=None):
        """Factorizes the user_items matrix.

        After calling this method, the members 'user_factors' and 'item_factors' will be
        initialized with a latent factor model of the input data.

        The user_items matrix does double duty here. It defines which items are liked by which
        users (P_ui in the original paper), as well as how much confidence we have that the user
        liked the item (C_ui).

        The negative items are implicitly defined: This code assumes that positive items in the
        user_items matrix means that the user liked the item. The negatives are left unset in this
        sparse matrix: the library will assume that means Piu = 0 and Ciu = 1 for all these items.
        Negative items can also be passed with a higher confidence value by passing a negative
--------------------------

In [64]:
# --- Section 7 base helpers (fit + scoring harness) ---
import time

def fit_als(mat, factors, reg, model_alpha, iters, seed, cb=None):
    """Fit implicit ALS. NOTE: implicit multiplies the passed confidence matrix by model.alpha
    (verified in implicit/cpu/als.py source), so we pass C = 1 + alpha_t*log1p(sec) directly
    as matrix values and set model.alpha=1.0, making effective confidence exactly C and reg
    equal to the requested value."""
    m = AlternatingLeastSquares(
        factors=factors,
        regularization=reg,
        alpha=model_alpha,
        iterations=iters,
        random_state=seed,
        calculate_training_loss=True,
        num_threads=1,
    )
    t0 = time.time()
    m.fit(mat, show_progress=False, callback=cb)
    return m, time.time() - t0

def eval_als_on(model, cand_pool, u_pos, i_pos, item_factors, user_factors, popularity_fallback, K=5):
    """Dot-product scores restricted to candidate pool; missing embeddings -> no score."""
    pool_idx = np.array([i_pos[s] for s in cand_pool if s in i_pos], dtype=np.int64)
    pool_shows = np.array([int(s) for s in cand_pool if s in i_pos], dtype=np.int64)
    pool_no_emb = [int(s) for s in cand_pool if s not in i_pos]
    col_to_show = {int(c): int(s) for c, s in zip(pool_idx, pool_shows)}
    all_scores = user_factors @ item_factors.T  # (n_hist_users, n_items_hist)
    scores_pool = np.full(all_scores.shape, -np.inf, dtype=np.float32)
    scores_pool[:, pool_idx] = all_scores[:, pool_idx]
    recs = {}
    n_fallback_users = 0
    for u in EVAL_USERS:
        if u not in u_pos:
            recs[u] = popularity_fallback[:K]
            n_fallback_users += 1
            continue
        row = scores_pool[u_pos[u]]
        if not np.isfinite(row).any():
            recs[u] = popularity_fallback[:K]
            n_fallback_users += 1
            continue
        topk = np.argpartition(-row, kth=K - 1)[:K]
        topk = topk[np.argsort(-row[topk])]
        recs[u] = [col_to_show[int(j)] for j in topk]
    return recs, pool_no_emb, n_fallback_users

print("c_s7_base: helpers defined (fit_als, eval_als_on); no model fit here.")


c_s7_base: helpers defined (fit_als, eval_als_on); no model fit here.


### Part 2 — root cause and fix for Method A

Evidence (Part 3 diagnostics, committed): in-sample AUC of ALS factors = 0.8916 (factors separate watched from unwatched), and TruncatedSVD(k=64) on the same log1p(total_seconds) matrix reaches MAP@5 GT-1 = 0.2104 vs ALS 0.0029. The data and the matrix are fine; the problem is how implicit ALS is called.

Root cause: `implicit.cpu.als.fit` multiplies the passed confidence matrix by `model.alpha` (verified in the installed source at implicit/cpu/als.py). The first implementation encoded `C = 1 + alpha*log1p(sec)` in the matrix AND passed `alpha=alpha` to the constructor, so effective confidence was `alpha * (1 + alpha*log1p(sec))` and regularization behaved as `reg/alpha`. That was fixed (`model_alpha=1.0`), but ALS still underperforms because the confidence scale (alpha_t=40 -> confidences ~166-433) is large relative to reg=0.01: item factor norms collapse (~0.08) vs user factor norms (~50). The `model_alpha` sweep proves the sensitivity: 0.01 -> MAP 0.0815, 1.0 -> 0.0029, 100 -> 0.00024.

Wrong lines (first implementation):
    m = AlternatingLeastSquares(factors=factors, regularization=reg, alpha=alpha, ...)
    m.fit(mat, show_progress=True)

Corrected lines:
    m = AlternatingLeastSquares(factors=factors, regularization=reg, alpha=1.0, ..., use_cg=use_cg)
    m.fit(mat, show_progress=False, callback=cb)

Matrix values already encode the full confidence C = 1 + alpha_t*log1p(sec); constructor alpha stays 1.0 so implicit does not re-scale it. alpha_t, reg and use_cg are now swept as hyperparameters.


### Initial runs (before debugging)

Run table from the first Method A implementation (commit ca2cc3b, before the confidence-scale fix). Kept for reference; not part of the current best-config results.
First implementation bug: `C = 1 + alpha*log1p(sec)` encoded in the matrix AND `alpha=40` passed to the constructor, so `implicit` re-scaled by alpha (effective `40*(1+40*log1p(sec))`; reg behaved as `reg/40`).

| run | config | MAP@5 GT-1 | MAP@5 GT-2 | GT-1 empty=0 | fit(s) |
| 1 | base (old): f=64, reg=0.01, alpha_t=40, min_os=60, iters=15, cg | 0.002911 | 0.017425 | 0.002878 | 2.50 |
| 2 | alpha_t=10 | 0.005902 | 0.025366 | 0.005836 | 2.47 |
| 3 | alpha_t=100 | 0.002640 | 0.017070 | 0.016854 | 3.07 |
| 4 | factors=32 | 0.003620 | 0.018842 | 0.003579 | 1.88 |
| 5 | factors=128 | 0.004278 | 0.021468 | 0.004230 | 5.08 |
| 6 | reg=0.001 | 0.002426 | 0.016206 | 0.002399 | 3.03 |
| 7 | reg=0.1 | 0.003007 | 0.019175 | 0.002974 | 3.08 |
| 8 | iters=10 | 0.003216 | 0.018072 | 0.003180 | 2.41 |
| 9 | iters=30 | 0.003034 | 0.017481 | 0.003000 | 6.38 |
| 10 | min_overlap_sec=0 | 0.002911 | 0.017425 | 0.002878 | 3.06 |
| 11 | min_overlap_sec=300 | 0.004967 | 0.024049 | 0.004912 | 2.17 |

Seed stability (pre-fix best config alpha_t=10, 3 seeds): [0.005902, 0.004561, 0.006330], mean=0.005597, std=0.000753.
Pre-fix Fold B (no tuning): MAP@5 GT-1=0.008221, GT-2=0.029797.

All numbers above are real printed outputs from commit ca2cc3b.


In [65]:
# --- Section 7 experiments (FIXED): one variable at a time (<= 15 runs total) ---
import time

als_runs = []
base_model_A = None
base_u_pos = None
base_i_pos = None
best_model_A = None
best_recs_A = None
best_u_pos = None
best_i_pos = None
best_row = None

def fit_als2(mat, factors, reg, model_alpha, iters, seed, use_cg=True, cb=None):
    m = AlternatingLeastSquares(
        factors=factors,
        regularization=reg,
        alpha=model_alpha,
        iterations=iters,
        random_state=seed,
        calculate_training_loss=True,
        num_threads=1,
        use_cg=use_cg,
    )
    t0 = time.time()
    m.fit(mat, show_progress=False, callback=cb)
    return m, time.time() - t0

def run_fit_eval2(label, alpha_t, factors, reg, iters, min_os, seed, use_cg,
                  cand_pool, gt1, gt2, pop_fallback, pairs_hist):
    lab = pairs_hist[
        (pairs_hist["tv_show_id"] != 0)
        & (pairs_hist["overlap_sec"] >= min_os)
        & (pairs_hist["overlap_frac"] >= 0.0)
    ].copy()
    mat, u_pos, i_pos = build_conf_matrix(lab, alpha_t)
    model, fit_t = fit_als2(mat, factors=factors, reg=reg, model_alpha=1.0,
                            iters=iters, seed=seed, use_cg=use_cg)
    item_f = model.item_factors
    user_f = model.user_factors
    t0 = time.time()
    recs, pool_no_emb, n_fb = eval_als_on(model, cand_pool, u_pos, i_pos, item_f, user_f, pop_fallback, K=K)
    score_t = time.time() - t0
    m1, e1 = map_at5(recs, gt1)
    m1z, _ = map_at5(recs, gt1, count_empty_as_zero=True)
    m2, e2 = map_at5(recs, gt2)
    m2z, _ = map_at5(recs, gt2, count_empty_as_zero=True)
    row = dict(run=len(als_runs) + 1, config=label, alpha_t=alpha_t, factors=factors,
               reg=reg, iters=iters, min_os=min_os, seed=seed, use_cg=use_cg,
               map_gt1=m1, map_gt2=m2, map_gt1_empty0=m1z, map_gt2_empty0=m2z,
               fit_time=fit_t, score_time=score_t, n_pool_no_emb=len(pool_no_emb),
               n_fallback=n_fb)
    als_runs.append(row)
    print(f"run {row['run']:2d}: {label:42s} | GT-1 {m1:.6f} (e0 {m1z:.6f}) | GT-2 {m2:.6f} (e0 {m2z:.6f}) | fit {fit_t:.2f}s")
    return model, recs, u_pos, i_pos

pop_fallback_A = [int(s) for s in histA.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).index if int(s) in set(candA)][:K]
pop5_secs_A = pop5_secs

configs = [
    ("base (old): f=64, reg=0.01, alpha_t=40, min_os=60, iters=15, cg", 40, 64, 0.01, 15, 60, 42, True),
    ("alpha_t=10", 10, 64, 0.01, 15, 60, 42, True),
    ("alpha_t=4", 4, 64, 0.01, 15, 60, 42, True),
    ("alpha_t=1", 1, 64, 0.01, 15, 60, 42, True),
    ("alpha_t=0.1", 0.1, 64, 0.01, 15, 60, 42, True),
    ("alpha_t=40, reg=0.1", 40, 64, 0.1, 15, 60, 42, True),
    ("alpha_t=10, reg=0.1", 10, 64, 0.1, 15, 60, 42, True),
    ("alpha_t=1, reg=0.1", 1, 64, 0.1, 15, 60, 42, True),
    ("alpha_t=40, cg=False", 40, 64, 0.01, 15, 60, 42, False),
    ("alpha_t=10, cg=False", 10, 64, 0.01, 15, 60, 42, False),
    ("alpha_t=1, cg=False", 1, 64, 0.01, 15, 60, 42, False),
    ("alpha_t=10, factors=32", 10, 32, 0.01, 15, 60, 42, True),
    ("alpha_t=10, reg=0.001", 10, 64, 0.001, 15, 60, 42, True),
    ("alpha_t=10, min_os=300", 10, 64, 0.01, 15, 300, 42, True),
    ("alpha_t=10, min_os=0", 10, 64, 0.01, 15, 0, 42, True),
]
print(f"planned ALS runs on Fold A: {len(configs)} (<= 15)")
for (label, alpha_t, factors, reg, iters, min_os, seed, use_cg) in configs:
    model, recs_tmp, u_pos_tmp, i_pos_tmp = run_fit_eval2(
        label, alpha_t, factors, reg, iters, min_os, seed, use_cg,
        candA, gt1_eval, gt2_eval, pop_fallback_A, labA,
    )
    if label.startswith("base (old)"):
        base_model_A = model
        base_u_pos = u_pos_tmp
        base_i_pos = i_pos_tmp

best_row = max(als_runs, key=lambda r: r["map_gt1"])
print(f"\nBEST FOLD A CONFIG: {best_row['config']} | MAP@5 GT-1 = {best_row['map_gt1']:.6f} | GT-2 = {best_row['map_gt2']:.6f}")

print("\n--- full ALS run log (Fold A) ---")
print(f"{'run':>3s} | {'config':44s} | {'MAP@5 GT-1':>10s} | {'MAP@5 GT-2':>10s} | {'GT-1 empty=0':>12s} | {'fit(s)':>6s}")
for r in als_runs:
    print(f"{r['run']:>3d} | {r['config']:44s} | {r['map_gt1']:10.6f} | {r['map_gt2']:10.6f} | {r['map_gt1_empty0']:12.6f} | {r['fit_time']:6.2f}")

# refit best config and keep objects for verification / qualitative sections
best_model_A, best_recs_A, best_u_pos, best_i_pos = run_fit_eval2(
    f"BEST (refit): {best_row['config']}",
    best_row["alpha_t"], best_row["factors"], best_row["reg"], best_row["iters"],
    best_row["min_os"], best_row["seed"], best_row["use_cg"],
    candA, gt1_eval, gt2_eval, pop_fallback_A, labA,
)

# 3-seed stability
seed_scores = []
for s in [42, 123, 777]:
    _, recs_s, _, _ = run_fit_eval2(
        f"seed-stab s={s}", best_row["alpha_t"], best_row["factors"], best_row["reg"],
        best_row["iters"], best_row["min_os"], s, best_row["use_cg"],
        candA, gt1_eval, gt2_eval, pop_fallback_A, labA,
    )
    m1_s, _ = map_at5(recs_s, gt1_eval)
    seed_scores.append(m1_s)
print(f"\nseed stability (best config, 3 seeds): {[round(v, 6) for v in seed_scores]} mean={np.mean(seed_scores):.6f} std={np.std(seed_scores):.6f}")


planned ALS runs on Fold A: 15 (<= 15)
run  1: base (old): f=64, reg=0.01, alpha_t=40, min_os=60, iters=15, cg | GT-1 0.002911 (e0 0.002878) | GT-2 0.017425 (e0 0.017204) | fit 2.50s
run  2: alpha_t=10                                 | GT-1 0.005902 (e0 0.005836) | GT-2 0.025366 (e0 0.025044) | fit 2.50s
run  3: alpha_t=4                                  | GT-1 0.011431 (e0 0.011304) | GT-2 0.042511 (e0 0.041971) | fit 2.54s
run  4: alpha_t=1                                  | GT-1 0.042220 (e0 0.041751) | GT-2 0.139794 (e0 0.138019) | fit 2.49s
run  5: alpha_t=0.1                                | GT-1 0.095944 (e0 0.094878) | GT-2 0.293433 (e0 0.289707) | fit 2.49s
run  6: alpha_t=40, reg=0.1                        | GT-1 0.003007 (e0 0.002974) | GT-2 0.019175 (e0 0.018931) | fit 2.50s
run  7: alpha_t=10, reg=0.1                        | GT-1 0.005682 (e0 0.005619) | GT-2 0.025487 (e0 0.025163) | fit 2.49s
run  8: alpha_t=1, reg=0.1                         | GT-1 0.042022 (e0 0.041556

In [66]:
# --- Section 7 variants (Fold A): keep watched vs exclude watched (best config) ---
best_alpha_t = best_row["alpha_t"]
best_factors = best_row["factors"]
best_reg = best_row["reg"]
best_iters = best_row["iters"]
best_min_os = best_row["min_os"]
best_use_cg = best_row["use_cg"]

mat_best, u_pos_best, i_pos_best = build_conf_matrix(labA, best_alpha_t)
model_best, fit_t_best = fit_als2(mat_best, factors=best_factors, reg=best_reg, model_alpha=1.0,
                                  iters=best_iters, seed=42, use_cg=best_use_cg)
U_best = model_best.user_factors
V_best = model_best.item_factors
all_scores_best = (U_best @ V_best.T).astype(np.float64)
print(f"\nbest model factor norms: user min={np.linalg.norm(U_best, axis=1).min():.3f} med={np.median(np.linalg.norm(U_best, axis=1)):.3f} max={np.linalg.norm(U_best, axis=1).max():.3f} | item min={np.linalg.norm(V_best, axis=1).min():.4f} med={np.median(np.linalg.norm(V_best, axis=1)):.4f} max={np.linalg.norm(V_best, axis=1).max():.4f}")

def score_top5(u, exclude_watched=False):
    if u not in u_pos_best:
        return pop_fallback_A[:K]
    row_scores = all_scores_best[u_pos_best[u]].copy()
    if exclude_watched:
        for s in labA[labA["user_id"] == u]["tv_show_id"].unique():
            si = int(s)
            if si in i_pos_best:
                row_scores[i_pos_best[si]] = -np.inf
    pool_idx = np.array([i_pos_best[s] for s in candA if s in i_pos_best], dtype=np.int64)
    pool_shows = np.array([int(s) for s in candA if s in i_pos_best], dtype=np.int64)
    sub = row_scores[pool_idx]
    finite = np.isfinite(sub)
    n_avail = int(finite.sum())
    if n_avail == 0:
        return pop_fallback_A[:K]
    k = min(K, n_avail)
    sub_f = sub[finite]
    shows_f = pool_shows[finite]
    top = np.argpartition(-sub_f, kth=k - 1)[:k]
    top = top[np.argsort(-sub_f[top])]
    rec = [int(shows_f[int(c)]) for c in top]
    if len(rec) < K:
        rec += [int(s) for s in pop5_secs_A if int(s) not in set(rec)][:K - len(rec)]
    return rec

recs_keep = {u: score_top5(u, exclude_watched=False) for u in EVAL_USERS}
recs_excl = {u: score_top5(u, exclude_watched=True) for u in EVAL_USERS}

def check_recs_dict(recs, name):
    ok = True
    cset = set(candA)
    for u, r in recs.items():
        if len(r) != K or len(set(r)) != K or 0 in set(r) or not set(r) <= cset:
            ok = False
            print(f"  ASSERT FAILED {name} user={u}: {r}")
    return ok

ok_keep = check_recs_dict(recs_keep, "keep-watched")
ok_excl = check_recs_dict(recs_excl, "exclude-watched")

m1_keep, e1_keep = map_at5(recs_keep, gt1_eval)
m1z_keep, _ = map_at5(recs_keep, gt1_eval, count_empty_as_zero=True)
m2_keep, e2_keep = map_at5(recs_keep, gt2_eval)
m2z_keep, _ = map_at5(recs_keep, gt2_eval, count_empty_as_zero=True)
m1_excl, e1_excl = map_at5(recs_excl, gt1_eval)
m1z_excl, _ = map_at5(recs_excl, gt1_eval, count_empty_as_zero=True)
m2_excl, e2_excl = map_at5(recs_excl, gt2_eval)
m2z_excl, _ = map_at5(recs_excl, gt2_eval, count_empty_as_zero=True)

print(f"\nvariants (best config: {best_row['config']}):")
print(f"  keep-watched : MAP@5 GT-1 {m1_keep:.6f} ({len(EVAL_USERS)-e1_keep} users) | empty=0 {m1z_keep:.6f} | GT-2 {m2_keep:.6f} ({len(EVAL_USERS)-e2_keep} users) | empty=0 {m2z_keep:.6f}")
print(f"  exclude-watch: MAP@5 GT-1 {m1_excl:.6f} ({len(EVAL_USERS)-e1_excl} users) | empty=0 {m1z_excl:.6f} | GT-2 {m2_excl:.6f} ({len(EVAL_USERS)-e2_excl} users) | empty=0 {m2z_excl:.6f}")
print(f"  asserts keep={ok_keep} exclude={ok_excl} (length {K}, no id 0, no dups, all in pool)")

# honesty check: MAP by user activity group
labelled_counts_all = labA.groupby("user_id").size()
counts_arr = np.array([int(labelled_counts_all.get(u, 0)) for u in EVAL_USERS])
q33 = float(np.percentile(counts_arr, 33))
q66 = float(np.percentile(counts_arr, 66))
def activity_group(u):
    c = int(labelled_counts_all.get(u, 0))
    if c <= q33:
        return "light"
    if c <= q66:
        return "medium"
    return "heavy"
groups = {"light": [], "medium": [], "heavy": []}
for u in EVAL_USERS:
    groups[activity_group(u)].append(u)
for gname in ["light", "medium", "heavy"]:
    gusers = groups[gname]
    recs_g = {u: recs_keep[u] for u in gusers}
    gt_g = {u: gt1_eval.get(u, set()) for u in gusers}
    mg, eg = map_at5(recs_g, gt_g)
    mgz, _ = map_at5(recs_g, gt_g, count_empty_as_zero=True)
    print(f"  activity {gname:6s}: n_users={len(gusers):4d} MAP@5 GT-1={mg:.6f} empty=0={mgz:.6f} median_airings={np.median(counts_arr[[EVAL_USERS.index(u) for u in gusers]]):.0f}")

share_emb = sum(int(s) in i_pos_best for s in candA) / len(candA)
n_emb = sum(int(s) in i_pos_best for s in candA)
print(f"\ncandidate-pool shows with embedding: {n_emb} / {len(candA)} = {share_emb:.4f}")

print("\n--- comparison table Fold A (baselines from step 6 output) ---")
print(f"{'method':34s} | {'MAP@5 GT-1':>10s} | {'MAP@5 GT-2':>10s} | {'GT-1 empty=0':>12s} | {'GT-2 empty=0':>12s}")
def _m4(recs, gt1, gt2):
    m1, _ = map_at5(recs, gt1)
    m2, _ = map_at5(recs, gt2)
    m1z, _ = map_at5(recs, gt1, count_empty_as_zero=True)
    m2z, _ = map_at5(recs, gt2, count_empty_as_zero=True)
    return m1, m2, m1z, m2z

def _print4(name, vals):
    print(f"{name:34s} | {vals[0]:10.6f} | {vals[1]:10.6f} | {vals[2]:12.6f} | {vals[3]:12.6f}")

_rr = np.array([_m4(rec_random(1000 + s), gt1_eval, gt2_eval) for s in range(5)]).mean(axis=0)
_print4("Random (avg 5 seeds)", _rr)
_print4("Popularity-by-users", _m4(recs_pop_users, gt1_eval, gt2_eval))
_print4("Popularity-by-seconds", _m4(recs_pop_secs, gt1_eval, gt2_eval))
_print4("Recency-popularity (w19-20)", _m4(recs_pop_recent, gt1_eval, gt2_eval))
_print4("Personal history [diagnostic]", _m4(recs_personal, gt1_eval, gt2_eval))
print(f"{'ALS keep-watched (best)':34s} | {m1_keep:10.6f} | {m2_keep:10.6f} | {m1z_keep:12.6f} | {m2z_keep:12.6f}")
print(f"{'ALS exclude-watched':34s} | {m1_excl:10.6f} | {m2_excl:10.6f} | {m1z_excl:12.6f} | {m2z_excl:12.6f}")



best model factor norms: user min=0.073 med=10.248 max=24.881 | item min=0.0003 med=0.0687 max=0.2948

variants (best config: alpha_t=0.1):
  keep-watched : MAP@5 GT-1 0.095944 (1246 users) | empty=0 0.094878 | GT-2 0.293433 (1244 users) | empty=0 0.289707
  exclude-watch: MAP@5 GT-1 0.007100 (1246 users) | empty=0 0.007021 | GT-2 0.056383 (1244 users) | empty=0 0.055667
  asserts keep=True exclude=True (length 5, no id 0, no dups, all in pool)
  activity light : n_users= 416 MAP@5 GT-1=0.112731 empty=0=0.109479 median_airings=228
  activity medium: n_users= 416 MAP@5 GT-1=0.114058 empty=0=0.113510 median_airings=514
  activity heavy : n_users= 428 MAP@5 GT-1=0.062578 empty=0=0.062578 median_airings=976

candidate-pool shows with embedding: 2206 / 6376 = 0.3460

--- comparison table Fold A (baselines from step 6 output) ---
method                             | MAP@5 GT-1 | MAP@5 GT-2 | GT-1 empty=0 | GT-2 empty=0
Random (avg 5 seeds)               |   0.000506 |   0.003538 |     0.000

In [67]:
# --- Section 7: best Fold-A config on Fold B (no tuning on Fold B) ---
pop_fallback_B = [int(s) for s in histB.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).index if int(s) in set(candB)][:K]

labB_best = pairs[
    (pairs["tv_show_id"] != 0)
    & (pairs["iso_week"].between(11, 18))
    & (pairs["overlap_sec"] >= best_min_os)
    & (pairs["overlap_frac"] >= 0.0)
].copy()
matB_best, u_posB_best, i_posB_best = build_conf_matrix(labB_best, best_alpha_t)
modelB_best, fit_tB_best = fit_als2(matB_best, factors=best_factors, reg=best_reg, model_alpha=1.0,
                                    iters=best_iters, seed=42, use_cg=best_use_cg)
recsB_best, _, _ = eval_als_on(modelB_best, candB, u_posB_best, i_posB_best,
                               modelB_best.item_factors, modelB_best.user_factors,
                               pop_fallback_B, K=K)
m1B, e1B = map_at5(recsB_best, gt1_B)
m1zB, _ = map_at5(recsB_best, gt1_B, count_empty_as_zero=True)
m2B, e2B = map_at5(recsB_best, gt2_B)
m2zB, _ = map_at5(recsB_best, gt2_B, count_empty_as_zero=True)

print(f"\nFold B with best Fold-A config (no Fold-B tuning), fit={fit_tB_best:.2f}s:")
print(f"  MAP@5 GT-1: {m1B:.6f} ({len(EVAL_USERS)-e1B} users) | empty=0: {m1zB:.6f}")
print(f"  MAP@5 GT-2: {m2B:.6f} ({len(EVAL_USERS)-e2B} users) | empty=0: {m2zB:.6f}")

print("\n--- comparison table Fold B ---")
print(f"{'method':34s} | {'MAP@5 GT-1':>10s} | {'MAP@5 GT-2':>10s} | {'GT-1 empty=0':>12s} | {'GT-2 empty=0':>12s}")
def _m4(recs, gt1, gt2):
    m1, _ = map_at5(recs, gt1)
    m2, _ = map_at5(recs, gt2)
    m1z, _ = map_at5(recs, gt1, count_empty_as_zero=True)
    m2z, _ = map_at5(recs, gt2, count_empty_as_zero=True)
    return m1, m2, m1z, m2z

def _print4(name, vals):
    print(f"{name:34s} | {vals[0]:10.6f} | {vals[1]:10.6f} | {vals[2]:12.6f} | {vals[3]:12.6f}")

_rrB = np.array([_m4(rec_random_B(1000 + s), gt1_B, gt2_B) for s in range(5)]).mean(axis=0)
_print4("Random (avg 5 seeds)", _rrB)
_print4("Popularity-by-users", _m4(recs_pop_users_B, gt1_B, gt2_B))
_print4("Popularity-by-seconds", _m4(recs_pop_secs_B, gt1_B, gt2_B))
_print4("Recency-popularity (w17-18)", _m4(recs_pop_recent_B, gt1_B, gt2_B))
_print4("Personal history [diagnostic]", _m4(recs_personal_B, gt1_B, gt2_B))
print(f"{'ALS keep-watched (best Fold-A config)':34s} | {m1B:10.6f} | {m2B:10.6f} | {m1zB:12.6f} | {m2zB:12.6f}")



Fold B with best Fold-A config (no Fold-B tuning), fit=2.13s:
  MAP@5 GT-1: 0.095080 (1246 users) | empty=0: 0.094024
  MAP@5 GT-2: 0.274747 (1240 users) | empty=0: 0.270386

--- comparison table Fold B ---
method                             | MAP@5 GT-1 | MAP@5 GT-2 | GT-1 empty=0 | GT-2 empty=0
Random (avg 5 seeds)               |   0.000410 |   0.003508 |     0.000405 |     0.003452
Popularity-by-users                |   0.188676 |   0.552938 |     0.186579 |     0.544161
Popularity-by-seconds              |   0.192483 |   0.541780 |     0.190344 |     0.533180
Recency-popularity (w17-18)        |   0.165599 |   0.554387 |     0.163759 |     0.545587
Personal history [diagnostic]      |   0.303652 |   0.504927 |     0.300278 |     0.496912
ALS keep-watched (best Fold-A config) |   0.095080 |   0.274747 |     0.094024 |     0.270386


In [68]:
# --- Section 7 verification (redo with fixed best model) ---
import inspect

print("=" * 70)
print("VERIFICATION 1 — synthetic block structure")
print("=" * 70)
rng = np.random.default_rng(0)
n_u, n_i = 10, 6
C = np.zeros((n_u, n_i), dtype=np.float32)
C[:5, :3] = 100.0
C[5:, 3:] = 100.0
C += rng.uniform(0, 1, size=C.shape).astype(np.float32)
mat_block = sp.csr_matrix(C)
m_block, _ = fit_als2(mat_block, factors=4, reg=0.01, model_alpha=1.0, iters=20, seed=0, use_cg=True)
n_correct = 0
for u in range(n_u):
    ids, scores = m_block.recommend(u, mat_block[u], N=3, filter_already_liked_items=False)
    ids = [int(x) for x in ids]
    expected = list(range(0, 3)) if u < 5 else list(range(3, 6))
    hit = bool(set(ids) & set(expected))
    n_correct += hit
    print(f"synthetic user {u}: recommended item indices {ids} | expected block {expected} | hit={hit}")
print(f"block-test users with >=1 own-block recommendation: {n_correct}/{n_u}")

print("\n" + "=" * 70)
print("VERIFICATION 2 — hand-check of one (user, show) dot-product score")
print("=" * 70)
u_star = next(u for u in EVAL_USERS if u in best_u_pos)
recs_u = best_recs_A[u_star]
s_star = recs_u[0]
print(f"hand-check user={u_star}, show={s_star} (first in its ALS top-5, best config)")
j_star = best_i_pos[s_star]
manual = float(model_best.user_factors[best_u_pos[u_star]] @ model_best.item_factors[j_star])
mat_trace, _, _ = build_conf_matrix(labA, best_alpha_t)
ids_lib, scores_lib = model_best.recommend(
    best_u_pos[u_star], mat_trace[best_u_pos[u_star]], N=5, filter_already_liked_items=False,
)
ids_lib = [int(x) for x in ids_lib]
if j_star in ids_lib:
    lib_score = float(scores_lib[ids_lib.index(j_star)])
else:
    lib_score = None
print(f"  manual numpy dot product: {manual:.6f}")
print(f"  model.recommend score for same pair: {lib_score}")
print(f"  difference: {(manual - lib_score) if lib_score is not None else 'n/a (not in top-5)'}")

print("\n" + "=" * 70)
print("VERIFICATION 3 — training objective per iteration")
print("=" * 70)
fit_src = inspect.getsource(AlternatingLeastSquares.fit)
print("callback usage in fit source:")
for line in fit_src.splitlines():
    if "callback" in line:
        print("   ", line.strip())

loss_hist = []
def _cb(it, elapsed, loss):
    loss_hist.append((it, loss))

model_trace, fit_t_trace = fit_als2(
    mat_best, factors=best_factors, reg=best_reg, model_alpha=1.0, iters=15, seed=42, cb=_cb,
)
print(f"loss per iteration (best config, {fit_t_trace:.2f}s fit):")
for it, loss in loss_hist:
    print(f"  iter {it:2d}: loss = {loss:.4f}")
print(f"final objective: {loss_hist[-1][1]:.4f} (ALS decreases it monotonically by construction)")


VERIFICATION 1 — synthetic block structure
synthetic user 0: recommended item indices [0, 1, 2] | expected block [0, 1, 2] | hit=True
synthetic user 1: recommended item indices [1, 0, 2] | expected block [0, 1, 2] | hit=True
synthetic user 2: recommended item indices [0, 1, 2] | expected block [0, 1, 2] | hit=True
synthetic user 3: recommended item indices [1, 0, 2] | expected block [0, 1, 2] | hit=True
synthetic user 4: recommended item indices [1, 0, 2] | expected block [0, 1, 2] | hit=True
synthetic user 5: recommended item indices [3, 4, 5] | expected block [3, 4, 5] | hit=True
synthetic user 6: recommended item indices [5, 4, 3] | expected block [3, 4, 5] | hit=True
synthetic user 7: recommended item indices [3, 5, 4] | expected block [3, 4, 5] | hit=True
synthetic user 8: recommended item indices [5, 4, 3] | expected block [3, 4, 5] | hit=True
synthetic user 9: recommended item indices [3, 5, 4] | expected block [3, 4, 5] | hit=True
block-test users with >=1 own-block recommendat

In [69]:
# --- Section 7 qualitative: 2 fixed case users (kept for all later methods) ---
labelled_counts = labA.groupby("user_id").size()
eligible = [u for u in EVAL_USERS if gt1_eval.get(u)]
case_heavy = max(eligible, key=lambda u: int(labelled_counts.get(u, 0)))
light_candidates = [u for u in eligible if int(labelled_counts.get(u, 0)) > 0]
case_light = min(light_candidates, key=lambda u: int(labelled_counts.get(u, 0))) if light_candidates else case_heavy
CASE_USERS = [case_heavy, case_light]
print(f"CASE_USERS = {CASE_USERS}  (heavy={case_heavy}, light={case_light}; saved for all later methods)")

sched_map = schedule_nd.dropna(subset=["tv_show_title"]).groupby("tv_show_id").first()

def show_info(sid):
    sid = int(sid)
    if sid not in sched_map.index:
        return sid, "", "", ""
    r = sched_map.loc[sid]
    genres = " | ".join(str(r[c]) for c in ["tv_show_genre_1", "tv_show_genre_2", "tv_show_genre_3"]
                        if c in r.index and pd.notna(r[c]))
    return sid, str(r["tv_show_title"]), str(r["tv_show_category"]), genres

for u in CASE_USERS:
    print(f"\n=== case user {u} (labelled airings in history: {int(labelled_counts.get(u, 0))}) ===")
    hist_u = labA[labA["user_id"] == u].groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).head(5)
    print("top-5 most-watched shows in history:")
    for sid, sec in hist_u.items():
        sid_i, title, cat, genres = show_info(sid)
        print(f"  {sid_i:6d} | {sec:7.0f}s | {cat:12s} | {genres:30s} | {title}")
    print("GT-1 top-5 (target window):", sorted(int(s) for s in gt1_eval[u]))
    als_top5 = recs_keep.get(u, [])
    print(f"ALS top-5 (keep-watched): {als_top5}")
    hits = sorted(int(s) for s in set(als_top5) & gt1_eval[u])
    print(f"hits in GT-1: {hits} ({len(hits)}/5)")

readme_path = Path("README.md")
readme_text = readme_path.read_text()
if "## Case study users" not in readme_text:
    with open(readme_path, "a") as f:
        f.write("\n## Case study users (section 7)\n")
        f.write(f"Fixed users used for qualitative comparison across all methods: {CASE_USERS[0]} (heavy user) and {CASE_USERS[1]} (light user).\n")
print(f"README case-users block present after run: {'## Case study users' in Path('README.md').read_text()}")


CASE_USERS = ['13571199046623737884', '8150816946177670084']  (heavy=13571199046623737884, light=8150816946177670084; saved for all later methods)

=== case user 13571199046623737884 (labelled airings in history: 9614) ===
top-5 most-watched shows in history:
  2400480 |  892243s | Инфо         | общество и политика            | 'ТСН. Неделя'.
   20088 |  350704s | Развлечения  | информация (комплексная)       | 'Завтрак с 1+1'. Информационно-развлекательная программа.
  240081 |  253744s | Инфо         | общество и политика            | Факты. День.
  2400508 |  247527s | Инфо         | информация (комплексная)       | Сегодня. Итоги с Олегом Панютой.
  6500479 |  244989s | Познавательное | путешествия                    | 'Мир наизнанку. Бразилия', 36 эп.
GT-1 top-5 (target window): [200352, 240081, 700369, 2400480, 90074629]
ALS top-5 (keep-watched): [10003012, 90039256, 90068680, 90074481, 90050977]
hits in GT-1: [] (0/5)

=== case user 8150816946177670084 (labelled airings in hist

### Part 1 — debugging Method A hypotheses (pre-fix evidence)

In [70]:
# --- Part 1: hypothesis tests (Fold A; read installed docs, never guess) ---
import inspect
import numpy as np

print("=" * 72)
print("PART 1 — debugging hypotheses, Fold A base config")
print("  config: factors=64, reg=0.01, alpha_t=40, min_overlap_sec=60, seed=42")
def _m1_A(recs):
    return map_at5(recs, gt1_eval)[0]
print(f"reference: Random MAP@5 GT-1 = {float(np.mean([_m1_A(rec_random(1000 + s)) for s in range(5)])):.6f} | Popularity-by-seconds = {_m1_A(recs_pop_secs):.6f}")
print(f"reference: Personal-history diagnostic GT-1 = {_m1_A(recs_personal):.6f} | ALS base (run 1): see the section 7 initial-runs table above")
print("=" * 72)

model = base_model_A
U = model.user_factors
V = model.item_factors

# ---------- H4: matrix orientation (from installed docstring) ----------
fit_src = inspect.getsource(AlternatingLeastSquares.fit)
H4_doc_users_rows = "rows of the matrix are the users" in fit_src
print(f"\n[H4] fit() docstring: rows of user_items matrix are the users -> {H4_doc_users_rows}")
print(f"     user_items_A shape: {user_items_A.shape[0]} users x {user_items_A.shape[1]} items")
print(f"     user_factors rows: {U.shape[0]} | item_factors rows: {V.shape[0]}")
H4_ok = H4_doc_users_rows and user_items_A.shape[0] == U.shape[0] and user_items_A.shape[1] == V.shape[0]
print(f"     verdict H4: {'OK (users x items consistent)' if H4_ok else 'MISMATCH'}")

# ---------- H1: recommend default + manual rescore == helper ----------
rec_sig = inspect.signature(AlternatingLeastSquares.recommend)
rec_doc = AlternatingLeastSquares.recommend.__doc__ or ""
H1_default_true = "filter_already_liked_items=True" in rec_doc
print(f"\n[H1] model.recommend signature: {rec_sig}")
print(f"[H1] docstring states default filter_already_liked_items=True: {H1_default_true}")
print(f"[H1] eval_als_on uses manual dot product, NOT model.recommend -> already-watched NOT filtered")

i_pos_inv = {int(j): int(s) for s, j in i_posA.items()}
pool_idx = np.array([i_posA[s] for s in candA if s in i_posA], dtype=np.int64)
pool_shows = np.array([int(s) for s in candA if s in i_posA], dtype=np.int64)
col_to_show = {int(c): int(s) for c, s in zip(pool_idx, pool_shows)}
all_scores = (U @ V.T).astype(np.float64)
scores_pool = np.full(all_scores.shape, -np.inf, dtype=np.float64)
scores_pool[:, pool_idx] = all_scores[:, pool_idx]

recs_manual = {}
for u in EVAL_USERS:
    if u not in u_posA:
        recs_manual[u] = pop_fallback_A[:K]
        continue
    row = scores_pool[u_posA[u]]
    if not np.isfinite(row).any():
        recs_manual[u] = pop_fallback_A[:K]
        continue
    topk = np.argpartition(-row, kth=K - 1)[:K]
    topk = topk[np.argsort(-row[topk])]
    recs_manual[u] = [col_to_show[int(j)] for j in topk]

recs_helper, _, _ = eval_als_on(base_model_A, candA, u_posA, i_posA, V, U, pop_fallback_A, K=K)
identical = all(recs_manual[u] == recs_helper[u] for u in EVAL_USERS)
m1_help, _ = map_at5(recs_helper, gt1_eval)
print(f"[H1] manual numpy rescore == eval_als_on helper: {identical} | MAP@5 GT-1 (helper/base model): {m1_help:.6f}")
print(f"     verdict H1: {'REJECTED (filtering not the cause; neither path filters)' if identical else 'CONFIRMED (differs)'}")

# ---------- H2: id mapping round-trip ----------
print(f"\n[H2] user_id dtype in labA: {labA['user_id'].dtype}; sample: {labA['user_id'].iloc[0]!r} (type {type(labA['user_id'].iloc[0]).__name__})")
try:
    as_i64 = labA["user_id"].astype("int64")
    over_i64 = bool((as_i64 > 2**63 - 1).any())
    print(f"[H2] cast to int64 succeeds; any id > int64 max: {over_i64}")
except Exception as ex:
    print(f"[H2] cast to int64 RAISED {type(ex).__name__}: {ex}")
print(f"[H2] u_posA keys are str: {all(isinstance(k, str) for k in list(u_posA.keys())[:5])}")
i_pos_keys_str = all(isinstance(k, str) for k in list(i_posA.keys())[:5])
print(f"[H2] i_posA keys are str: {i_pos_keys_str}")
test_users = list(u_posA.keys())[:5]
ok = True
for u in test_users:
    idx = u_posA[u]
    row = user_items_A[idx]
    nz_ids = sorted(i_pos_inv[int(c)] for c in row.indices)
    lab_ids = sorted(int(s) for s in labA.loc[labA["user_id"] == u, "tv_show_id"].unique())
    match = nz_ids == lab_ids
    ok = ok and match
    print(f"     user {str(u)[:14]}...: matrix nonzeros={len(nz_ids)} labA items={len(lab_ids)} equal={match}")
s_probe = int(candA[0]) if int(candA[0]) in i_posA else int([s for s in candA if s in i_posA][0])
some_user = next(u for u in EVAL_USERS if u in u_posA and int(s_probe) in set(labA.loc[labA["user_id"] == u, "tv_show_id"].astype(int)))
row_probe = user_items_A[u_posA[some_user]]
conf_at_col = float(row_probe[0, i_posA[s_probe]]) if row_probe.shape[0] == 1 else float(row_probe[0, i_posA[s_probe]])
print(f"     probe show {s_probe}: train column i_posA={i_posA[s_probe]}, confidence for {str(some_user)[:8]}... = {conf_at_col:.2f} (>1 means observed in training)")
print(f"     verdict H2: {'OK' if ok and i_pos_keys_str else 'ISSUE'}")

# ---------- H3: candidate filtering order ----------
lens = [len(recs_manual[u]) for u in EVAL_USERS]
n_five = sum(1 for l in lens if l == 5)
print(f"\n[H3] rec list lengths over {len(EVAL_USERS)} users: min={min(lens)} max={max(lens)} all-5={n_five}")
print(f"     verdict H3: {'OK (every list length 5)' if n_five == len(EVAL_USERS) else 'ISSUE'}")

# ---------- H5: own-history sanity ----------
recs_hist = {}
for u in EVAL_USERS:
    if u not in u_posA:
        recs_hist[u] = pop_fallback_A[:K]
        continue
    items_u = [int(s) for s in labA.loc[labA["user_id"] == u, "tv_show_id"].unique() if int(s) in i_posA]
    if not items_u:
        recs_hist[u] = pop_fallback_A[:K]
        continue
    cols = np.array([i_posA[s] for s in items_u], dtype=np.int64)
    scores = all_scores[u_posA[u]][cols]
    top_cols = cols[np.argsort(-scores)][:K]
    recs_hist[u] = [int(i_pos_inv[int(c)]) for c in top_cols]
m1_hist, e1_hist = map_at5(recs_hist, gt1_eval)
print(f"\n[H5] MAP@5 GT-1 ranking ONLY own-history shows by ALS score: {m1_hist:.6f} ({len(EVAL_USERS)-e1_hist} users with truth)")
print(f"     personal-history diagnostic baseline GT-1: {_m1_A(recs_personal):.6f}")
print(f"     verdict H5: {'CONFIRMED (model ranks own history well -> cause is candidate pool/restriction)' if m1_hist > 0.10 else 'REJECTED (model itself is broken)'}")

h5_result = f"{m1_hist:.6f} vs {_m1_A(recs_personal):.6f}"
h5_verdict = "CONFIRMED" if m1_hist > 0.10 else "REJECTED"
print(f"\n--- SUMMARY TABLE ---")
print(f"{'hypothesis':14s} | {'test':38s} | {'result':20s} | {'verdict':16s}")
print(f"{'H1 filtering':14s} | {'manual==helper rescore':38s} | {'identical' if identical else 'DIFFERS':20s} | {'REJECTED' if identical else 'CONFIRMED':16s}")
print(f"{'H2 id mapping':14s} | {'round-trip 5 users + probe show':38s} | {'OK' if ok and i_pos_keys_str else 'ISSUE':20s} | {'OK' if ok and i_pos_keys_str else 'ISSUE':16s}")
print(f"{'H3 cand order':14s} | {'all lists length 5':38s} | {'yes' if n_five==len(EVAL_USERS) else 'no':20s} | {'OK' if n_five==len(EVAL_USERS) else 'ISSUE':16s}")
print(f"{'H4 orientation':14s} | {'rows=users per docstring+shapes':38s} | {'OK' if H4_ok else 'MISMATCH':20s} | {'OK' if H4_ok else 'ISSUE':16s}")
print(f"{'H5 own-history':14s} | {'MAP@5 GT-1 ranking own history':38s} | {h5_result:20s} | {h5_verdict:16s}")


PART 1 — debugging hypotheses, Fold A base config
  config: factors=64, reg=0.01, alpha_t=40, min_overlap_sec=60, seed=42
reference: Random MAP@5 GT-1 = 0.000506 | Popularity-by-seconds = 0.180193
reference: Personal-history diagnostic GT-1 = 0.301359 | ALS base (run 1): see the section 7 initial-runs table above

[H4] fit() docstring: rows of user_items matrix are the users -> True
     user_items_A shape: 4841 users x 2975 items
     user_factors rows: 4841 | item_factors rows: 2975
     verdict H4: OK (users x items consistent)

[H1] model.recommend signature: (self, userid, user_items, N=10, filter_already_liked_items=True, filter_items=None, recalculate_user=False, items=None)
[H1] docstring states default filter_already_liked_items=True: False
[H1] eval_als_on uses manual dot product, NOT model.recommend -> already-watched NOT filtered
[H1] manual numpy rescore == eval_als_on helper: True | MAP@5 GT-1 (helper/base model): 0.002911
     verdict H1: REJECTED (filtering not the caus

### Part 2 — diagnosing why ALS factors are degenerate


In [71]:
# --- Part 2: ALS failure diagnostics ---
import scipy.sparse as sp
import numpy as np
import time

print("=" * 72)
print("PART 2 — why do ALS factors fail even on own-history ranking?")
print("=" * 72)

# D1: factor dispersion
print(f"\n[D1] user_factors: mean={U.mean():.6f} std={U.std():.6f} min={U.min():.6f} max={U.max():.6f}")
print(f"[D1] item_factors: mean={V.mean():.6f} std={V.std():.6f} min={V.min():.6f} max={V.max():.6f}")
if EVAL_USERS[0] in u_posA:
    s0 = all_scores[u_posA[EVAL_USERS[0]]][pool_idx]
    print(f"[D1] user {str(EVAL_USERS[0])[:14]}... pool score std={s0.std():.6f} mean={s0.mean():.6f} (std near 0 = degenerate)")

# D2: implicit constructor alpha multiplies the confidence matrix
for a in [0.01, 0.1, 1.0, 10.0, 100.0]:
    m, ft = fit_als(user_items_A, factors=64, reg=0.01, model_alpha=a, iters=15, seed=42)
    recs, _, _ = eval_als_on(m, candA, u_posA, i_posA, m.item_factors, m.user_factors, pop_fallback_A, K=K)
    mm, _ = map_at5(recs, gt1_eval)
    print(f"[D2] model_alpha={a:<6g} MAP@5 GT-1={mm:.6f} fit={ft:.2f}s")

# D3: confidence transform variants (rebuild matrix each time)
def conf_mat_from(lab_df, transform):
    g = lab_df.groupby(["user_id", "tv_show_id"])["overlap_sec"].sum().reset_index(name="sec")
    users = sorted(g["user_id"].unique())
    items = sorted(g["tv_show_id"].unique())
    up = {u: i for i, u in enumerate(users)}
    ip = {i: j for j, i in enumerate(items)}
    sec = g["sec"].astype("float64").values
    if transform == "raw":
        c = sec
    elif transform == "1+sec":
        c = 1.0 + sec
    elif transform == "1+10log1p":
        c = 1.0 + 10.0 * np.log1p(sec)
    elif transform == "1+100log1p":
        c = 1.0 + 100.0 * np.log1p(sec)
    rows_d4 = g["user_id"].map(up).astype(np.int32).values
    cols = g["tv_show_id"].map(ip).astype(np.int32).values
    mat = sp.csr_matrix((c.astype(np.float32), (rows_d4, cols)), shape=(len(users), len(items)))
    return mat, up, ip

for t in ["raw", "1+sec", "1+10log1p", "1+100log1p"]:
    Ct, upt, ipt = conf_mat_from(labA, t)
    m, ft = fit_als(Ct, factors=64, reg=0.01, model_alpha=1.0, iters=15, seed=42)
    recs, _, _ = eval_als_on(m, candA, upt, ipt, m.item_factors, m.user_factors, pop_fallback_A, K=K)
    mm, _ = map_at5(recs, gt1_eval)
    print(f"[D3] C={t:<12s} MAP@5 GT-1={mm:.6f} fit={ft:.2f}s")

# D4: exact weighted-ALS in numpy on a subsample (50 users, all items) — one full cycle
print(f"\n[D4] exact weighted-ALS (numpy) on first 50 users, 1 user-item cycle, reg=0.01")
rng = np.random.default_rng(0)
n_samp = 50
samp_users = sorted(list(u_posA.keys()))[:n_samp]
samp_idx = np.array([u_posA[u] for u in samp_users], dtype=np.int32)
C_sub = user_items_A[samp_idx].tocsr()  # (50, 2975)
n_items = C_sub.shape[1]
Y = rng.normal(0, 0.1, size=(n_items, 64)).astype(np.float64)
X = rng.normal(0, 0.1, size=(n_samp, 64)).astype(np.float64)
reg = 0.01
YtY = Y.T @ Y
# solve users given Y
for uu in range(n_samp):
    Cu = C_sub[uu].tocoo()
    cols = Cu.col
    w = Cu.data - 1.0
    A = YtY + reg * np.eye(64) + (Y[cols] * w[:, None]).T @ Y[cols]
    b = Y[cols].T @ (Cu.data)  # Y^T C p with p=1 on observed
    X[uu] = np.linalg.solve(A, b)
# solve items given X
XtX = X.T @ X
for ii in range(n_items):
    Ci = C_sub[:, ii].tocoo()
    rows_d4 = Ci.row
    w = Ci.data - 1.0
    A = XtX + reg * np.eye(64) + (X[rows_d4] * w[:, None]).T @ X[rows_d4]
    b = X[rows_d4].T @ (Ci.data)
    Y[ii] = np.linalg.solve(A, b)
# evaluate on subsample users: rank their own history items
hits = 0
total = 0
for uu, u in enumerate(samp_users):
    lab_items = sorted(int(s) for s in labA.loc[labA["user_id"] == u, "tv_show_id"].unique() if int(s) in i_posA)
    gt = gt1_eval.get(u, set())
    if not gt:
        continue
    scores_all = X[uu] @ Y.T  # over all 2975 items (column j = item index)
    cols = np.array([i_posA[s] for s in lab_items], dtype=np.int64)
    scores = scores_all[cols]
    top_cols = cols[np.argsort(-scores)][:5]
    top_ids = [int(i_pos_inv[int(c)]) for c in top_cols]
    hits += len(set(top_ids) & gt)
    total += len(gt)
print(f"[D4] numpy exact ALS subsample MAP@5-ish (hits/total GT items): {hits}/{total} = {hits/total if total else 0:.4f}")
print("[D4] if this is high but implicit ALS stays near zero, implicit's solver/config is the cause")


PART 2 — why do ALS factors fail even on own-history ranking?

[D1] user_factors: mean=2.588511 std=4.805438 min=-20.388668 max=32.641754
[D1] item_factors: mean=0.003956 std=0.008420 min=-0.035292 max=0.045855
[D1] user 10008352999832... pool score std=0.427686 mean=0.343111 (std near 0 = degenerate)
[D2] model_alpha=0.01   MAP@5 GT-1=0.081541 fit=2.56s
[D2] model_alpha=0.1    MAP@5 GT-1=0.011771 fit=2.53s
[D2] model_alpha=1      MAP@5 GT-1=0.002911 fit=2.50s
[D2] model_alpha=10     MAP@5 GT-1=0.001332 fit=2.50s
[D2] model_alpha=100    MAP@5 GT-1=0.000241 fit=2.53s
[D3] C=raw          MAP@5 GT-1=0.001811 fit=2.50s
[D3] C=1+sec        MAP@5 GT-1=0.001557 fit=2.49s
[D3] C=1+10log1p    MAP@5 GT-1=0.005902 fit=2.50s
[D3] C=1+100log1p   MAP@5 GT-1=0.002640 fit=2.50s

[D4] exact weighted-ALS (numpy) on first 50 users, 1 user-item cycle, reg=0.01
[D4] numpy exact ALS subsample MAP@5-ish (hits/total GT items): 3/85 = 0.0353
[D4] if this is high but implicit ALS stays near zero, implicit's sol

### Part 3 — independent in-sample and SVD checks


In [72]:
# --- Part 3: independent checks (Fold A, base ALS config) ---
import numpy as np
import scipy.sparse as sp

print("=" * 72)
print("PART 3 — independent checks of the ALS failure")
print("  base config: factors=64, reg=0.01, alpha_t=40, min_overlap_sec=60, seed=42")
print(f"reference: Random MAP@5 GT-1 = {float(np.mean([_m1_A(rec_random(1000 + s)) for s in range(5)])):.6f} | Popularity-by-seconds = {_m1_A(recs_pop_secs):.6f}")
print(f"reference: Personal-history diagnostic GT-1 = {_m1_A(recs_personal):.6f} | ALS base: see the section 7 initial-runs table above")
print("=" * 72)

model = base_model_A
U = model.user_factors
V = model.item_factors
all_scores = (U @ V.T).astype(np.float64)  # base model factors, user_items_A column space

# ---- probe user: row nnz, confidence stats, factor norms ----
probe_u = next(u for u in EVAL_USERS if u in u_posA)
row = user_items_A[u_posA[probe_u]].tocsr()
nnz = int(row.nnz)
conf_vals = row.data
un = float(np.linalg.norm(U[u_posA[probe_u]]))
item_norms = np.linalg.norm(V, axis=1)
user_norms = np.linalg.norm(U, axis=1)
print(f"\nprobe user {str(probe_u)[:14]}...:")
print(f"  matrix row nnz={nnz}, confidence min={conf_vals.min():.3f} max={conf_vals.max():.3f} mean={conf_vals.mean():.3f}")
print(f"  probe user factor norm: {un:.6f}")
print(f"  item factor norms: min={item_norms.min():.6f} median={np.median(item_norms):.6f} max={item_norms.max():.6f}")
print(f"  user factor norms: min={user_norms.min():.6f} median={np.median(user_norms):.6f} max={user_norms.max():.6f}")

# ---- TEST 1: in-sample AUC (score watched > score unwatched) on training matrix ----
rng = np.random.default_rng(20261030)
n_test = 200
cand_pool_idx = np.array([i_posA[s] for s in candA if s in i_posA], dtype=np.int64)
test_users = [u for u in EVAL_USERS if u in u_posA]
test_users = rng.choice(test_users, size=min(n_test, len(test_users)), replace=False)

def auc_user(u):
    r = user_items_A[u_posA[u]].tocsr()
    watched = set(int(c) for c in r.indices.tolist())
    if len(watched) == 0:
        return None
    unwatched_idx = [int(c) for c in cand_pool_idx if int(c) not in watched]
    if len(unwatched_idx) == 0:
        return None
    n_pairs = min(len(watched) * len(unwatched_idx), 5000)
    w_list = np.array(sorted(watched))[rng.integers(0, len(watched), size=n_pairs)]
    uw_list = np.array(unwatched_idx)[rng.integers(0, len(unwatched_idx), size=n_pairs)]
    sw = all_scores[u_posA[u]][w_list]
    su = all_scores[u_posA[u]][uw_list]
    return float((sw > su).mean())

aucs = []
for u in test_users:
    a = auc_user(u)
    if a is not None:
        aucs.append(a)
if aucs:
    print(f"\n[TEST1] in-sample AUC over {len(aucs)} sampled users: mean={np.mean(aucs):.4f} median={np.median(aucs):.4f} min={min(aucs):.4f} max={max(aucs):.4f}")
else:
    print("\n[TEST1] no computable AUCs")
print("      far above 0.5 => factors separate watched from unwatched; near 0.5 => degenerate")

# ---- TEST 2: independent TruncatedSVD on log1p(total_seconds) matrix ----
print(f"\n[TEST2] rank-k factorization (TruncatedSVD k=64) on log1p(total_seconds) matrix")

def build_log_matrix():
    g = labA.groupby(["user_id", "tv_show_id"])["overlap_sec"].sum().reset_index(name="sec")
    users = sorted(g["user_id"].unique())
    items = sorted(g["tv_show_id"].unique())
    up = {u: i for i, u in enumerate(users)}
    ip = {i: j for j, i in enumerate(items)}
    sec = np.log1p(g["sec"].astype("float64").values)
    mat = sp.csr_matrix(
        (sec, (g["user_id"].map(up).astype(np.int32).values,
               g["tv_show_id"].map(ip).astype(np.int32).values)),
        shape=(len(users), len(items)),
    )
    return mat, up, ip

Mlog, up_log, ip_log = build_log_matrix()
print(f"     log matrix: {Mlog.shape[0]} users x {Mlog.shape[1]} items, nnz={Mlog.nnz:,}")
used_svd = None
try:
    from sklearn.decomposition import TruncatedSVD
    svd = TruncatedSVD(n_components=64, random_state=42)
    Us = svd.fit_transform(Mlog)
    Vs = svd.components_
    used_svd = "sklearn.decomposition.TruncatedSVD"
    print(f"     used: sklearn.decomposition.TruncatedSVD")
except Exception as ex:
    from scipy.sparse.linalg import svds
    print(f"     sklearn unavailable ({type(ex).__name__}: {ex}); using scipy svds")
    Ut, st, Vt = svds(Mlog, k=64, random_state=42)
    Us = Ut * st
    Vs = Vt
    used_svd = "scipy.sparse.linalg.svds"
    print(f"     used: scipy.sparse.linalg.svds")

scores_all = Us @ Vs  # (n_log_users, n_log_items) in log-matrix index space
recs_svd = {}
for u in EVAL_USERS:
    if u not in up_log:
        recs_svd[u] = pop_fallback_A[:K]
        continue
    ui = up_log[u]
    row_scores = scores_all[ui]
    cand_cols = np.array([ip_log[s] for s in candA if s in ip_log], dtype=np.int64)
    cand_shows = np.array([int(s) for s in candA if s in ip_log], dtype=np.int64)
    sub = row_scores[cand_cols]
    top = np.argpartition(-sub, kth=K - 1)[:K]
    top = top[np.argsort(-sub[top])]
    recs_svd[u] = [int(cand_shows[int(c)]) for c in top]

m1_svd, e1_svd = map_at5(recs_svd, gt1_eval)
print(f"[TEST2] SVD MAP@5 GT-1 (candidate pool, keep watched): {m1_svd:.6f} ({len(EVAL_USERS)-e1_svd} users with truth)")
print(f"     implicit ALS base: see the section 7 initial-runs table | popularity-by-seconds: {_m1_A(recs_pop_secs):.6f}")

t1_result = f"mean={np.mean(aucs):.4f}" if aucs else "n/a"
t1_verdict = "DEGENERATE (~0.5)" if aucs and np.mean(aucs) < 0.55 else ("OK (>0.55)" if aucs else "n/a")
t2_result = f"{m1_svd:.6f}"
print(f"\n--- SUMMARY ---")
print(f"{'test':16s} | {'result':22s} | {'verdict':16s}")
print(f"{'TEST1 in-sample AUC':16s} | {t1_result:22s} | {t1_verdict:16s}")
print(f"{'TEST2 SVD MAP@5 GT-1':16s} | {t2_result:22s} | {'works' if m1_svd >= 0.10 else 'also fails':16s}")


PART 3 — independent checks of the ALS failure
  base config: factors=64, reg=0.01, alpha_t=40, min_overlap_sec=60, seed=42
reference: Random MAP@5 GT-1 = 0.000506 | Popularity-by-seconds = 0.180193
reference: Personal-history diagnostic GT-1 = 0.301359 | ALS base: see the section 7 initial-runs table above

probe user 10008352999832...:
  matrix row nnz=130, confidence min=166.085 max=432.783 mean=275.782
  probe user factor norm: 50.023361
  item factor norms: min=0.016418 median=0.079047 max=0.102592
  user factor norms: min=10.625981 median=44.668114 max=59.953476

[TEST1] in-sample AUC over 200 sampled users: mean=0.8916 median=0.8983 min=0.7532 max=0.9974
      far above 0.5 => factors separate watched from unwatched; near 0.5 => degenerate

[TEST2] rank-k factorization (TruncatedSVD k=64) on log1p(total_seconds) matrix
     log matrix: 4841 users x 2975 items, nnz=842,560
     used: sklearn.decomposition.TruncatedSVD
[TEST2] SVD MAP@5 GT-1 (candidate pool, keep watched): 0.21040

### Part 1 — attribution: double alpha vs confidence scale

Fold A, base: factors=64, reg=0.01, iterations=15, seed=42, min_overlap_sec=60, keep already-watched, GT-1 + GT-2.
(a) original bug reproduced: matrix C = 1 + 40*log1p(sec) AND constructor alpha=40;
(b) single application: matrix C = 1 + 40*log1p(sec), model alpha=1;
(c) single application alpha_t = 10;
(d) single application alpha_t = 0.1;
(e) sweep alpha_t in {0.01, 0.03, 0.1, 0.3, 1, 3, 10, 40}, model alpha=1;
(f) binary matrix (all observed entries = 2) for reference.
Conclusion stated in the markdown cell AFTER this code cell once results are known.


In [73]:
# --- Part 1: attribution table (Fold A, keep-watched) ---
import numpy as np
import scipy.sparse as sp
import time

print("=" * 72)
print("PART 1 — attribution table (Fold A, keep-watched, GT-1 + GT-2)")
print("=" * 72)

def run_attr(label, C_matrix, u_pos, i_pos, model_alpha, factors=64, reg=0.01, iters=15, seed=42):
    m, fit_t = fit_als2(C_matrix, factors=factors, reg=reg, model_alpha=model_alpha,
                        iters=iters, seed=seed)
    recs, _, _ = eval_als_on(m, candA, u_pos, i_pos, m.item_factors, m.user_factors,
                             pop_fallback_A, K=K)
    m1, _ = map_at5(recs, gt1_eval)
    m2, _ = map_at5(recs, gt2_eval)
    print(f"{label:38s} | GT-1 {m1:.6f} | GT-2 {m2:.6f} | fit {fit_t:.2f}s")
    return m1, m2

def conf_stats(alpha_t):
    g = labA.groupby(["user_id", "tv_show_id"])["overlap_sec"].sum().reset_index(name="sec")
    sec = g["sec"].astype("float64").values
    C = 1.0 + alpha_t * np.log1p(sec)
    qs = np.quantile(C, [0.0, 0.5, 0.9, 1.0])
    return qs

print("\n[Quantiles of confidence C = 1 + alpha_t*log1p(sec), labelled interactions]")
qs_lines = []
for at in [40.0, 10.0, 0.1]:
    qs = conf_stats(at)
    qs_lines.append(f"  alpha_t={at:<6g} | min={qs[0]:.2f} median={qs[1]:.2f} p90={qs[2]:.2f} max={qs[3]:.2f}")
for line in qs_lines:
    print(line)

print("\n[Attribution table]")
print(f"{'config':38s} | {'MAP@5 GT-1':>10s} | {'MAP@5 GT-2':>10s} | {'fit(s)':>6s}")

# (a) original bug reproduced
mat_a, u_pos_a, i_pos_a = build_conf_matrix(labA, 40.0)
run_attr("(a) double: C=1+40log1p, alpha=40", mat_a, u_pos_a, i_pos_a, model_alpha=40.0)

# (b) single application alpha_t=40
mat_b, u_pos_b, i_pos_b = build_conf_matrix(labA, 40.0)
run_attr("(b) single: C=1+40log1p, alpha=1 ", mat_b, u_pos_b, i_pos_b, model_alpha=1.0)

# (c) single application alpha_t=10
mat_c, u_pos_c, i_pos_c = build_conf_matrix(labA, 10.0)
run_attr("(c) single: C=1+10log1p, alpha=1 ", mat_c, u_pos_c, i_pos_c, model_alpha=1.0)

# (d) single application alpha_t=0.1
mat_d, u_pos_d, i_pos_d = build_conf_matrix(labA, 0.1)
run_attr("(d) single: C=1+0.1log1p, alpha=1", mat_d, u_pos_d, i_pos_d, model_alpha=1.0)

# (e) sweep
print("\n[Sweep alpha_t, single application, model alpha=1]")
sweep_results = []
for at in [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 40.0]:
    mat, up, ip = build_conf_matrix(labA, at)
    m1, m2 = run_attr(f"  alpha_t={at:g}", mat, up, ip, model_alpha=1.0)
    sweep_results.append((at, m1, m2))

# (f) binary matrix
print("\n[Binary matrix reference]")
g = labA.groupby(["user_id", "tv_show_id"])["overlap_sec"].sum().reset_index(name="sec")
users_bin = sorted(g["user_id"].unique())
items_bin = sorted(g["tv_show_id"].unique())
up_bin = {u: i for i, u in enumerate(users_bin)}
ip_bin = {i: j for j, i in enumerate(items_bin)}
Cbin = np.ones(len(g), dtype=np.float32) * 2.0
mat_bin = sp.csr_matrix((Cbin, (g["user_id"].map(up_bin).astype(np.int32).values,
                                g["tv_show_id"].map(ip_bin).astype(np.int32).values)),
                        shape=(len(users_bin), len(items_bin)))
run_attr("(f) binary: all observed = 2", mat_bin, up_bin, ip_bin, model_alpha=1.0)


PART 1 — attribution table (Fold A, keep-watched, GT-1 + GT-2)

[Quantiles of confidence C = 1 + alpha_t*log1p(sec), labelled interactions]
  alpha_t=40     | min=165.43 median=271.18 p90=350.94 max=581.03
  alpha_t=10     | min=42.11 median=68.55 p90=88.48 max=146.01
  alpha_t=0.1    | min=1.41 median=1.68 p90=1.87 max=2.45

[Attribution table]
config                                 | MAP@5 GT-1 | MAP@5 GT-2 | fit(s)
(a) double: C=1+40log1p, alpha=40      | GT-1 0.000546 | GT-2 0.008213 | fit 2.51s
(b) single: C=1+40log1p, alpha=1       | GT-1 0.002911 | GT-2 0.017425 | fit 2.51s
(c) single: C=1+10log1p, alpha=1       | GT-1 0.005902 | GT-2 0.025366 | fit 2.54s
(d) single: C=1+0.1log1p, alpha=1      | GT-1 0.095944 | GT-2 0.293433 | fit 2.52s

[Sweep alpha_t, single application, model alpha=1]
  alpha_t=0.01                         | GT-1 0.098475 | GT-2 0.290533 | fit 2.51s
  alpha_t=0.03                         | GT-1 0.098381 | GT-2 0.294831 | fit 2.50s
  alpha_t=0.1               

(0.09112092027822366, 0.2841601911396927)

### Part 1 — conclusion (attribution)

The numbers isolate the cause. Removing the double alpha application ((a) → (b)) improves MAP@5 GT-1 only 5x (0.000546 → 0.002911) — marginal.
Lowering the confidence scale ((b) → (d), alpha_t 40 → 0.1) improves it 33x (0.002911 → 0.095944) — dominant.
The sweep shows the optimum near alpha_t = 0.01 (GT-1 = 0.098475), i.e. confidences of order 1–2 (median 1.68), not 100–500. The binary matrix (all confidences = 2) reaches 0.091121, close to the best weighted run — so log weighting is not the key factor; small confidence values are.
The earlier explanation 'reg behaves like reg/40' is NOT supported: if it were, fixing alpha to 1 (single application) while keeping C=1+40log1p(sec) would have recovered performance, but it does not (0.002911 vs random 0.000506). The real cause is that confidences ~166–581 swamp regularization (reg=0.01), collapsing item factor norms (~0.08) vs user norms (~50).
The best alpha_t = 0.01 lies on the EDGE of the swept grid {0.01..40}. Part 2 extends the grid downward to confirm the optimum is inside.


In [74]:
# --- Part 2: ALS re-tuning with best alpha_t (Fold A, keep-watched) ---
import numpy as np
import time

print("=" * 72)
print("PART 2 — ALS re-tuning (Fold A, keep-watched, GT-1 + GT-2)")
print("=" * 72)
print("  NOTE: each row rebuilds interactions from matched_pairs with that row's own min_overlap_sec.")

# Extend grid downward — best was alpha_t=0.01 at edge
print("\n[Extend grid downward]")
for at in [0.001, 0.003]:
    mat, up, ip = build_conf_matrix(labA, at)
    m1, m2 = run_attr(f"  alpha_t={at:g}", mat, up, ip, model_alpha=1.0)
    sweep_results.append((at, m1, m2))

best_entry = max(sweep_results, key=lambda x: x[1])
best_alpha_t = best_entry[0]
print(f"\nBest alpha_t after grid extension: {best_alpha_t:g} | GT-1 = {best_entry[1]:.6f} | GT-2 = {best_entry[2]:.6f}")

# One-variable-at-a-time re-tune (max 12 runs total incl. grid extension)
als_run_log = []
base_config = dict(alpha_t=best_alpha_t, factors=64, reg=0.01, iters=15, min_os=60, seed=42)

def run_tune(label, **overrides):
    cfg = dict(base_config)
    cfg.update(overrides)
    lab_cfg = pairs[(pairs["tv_show_id"] != 0)
                    & (pairs["iso_week"].between(11, 20))
                    & (pairs["overlap_sec"] >= cfg["min_os"])
                    & (pairs["overlap_frac"] >= 0.0)].copy()
    mat, up, ip = build_conf_matrix(lab_cfg, cfg["alpha_t"])
    m, fit_t = fit_als2(mat, factors=cfg["factors"], reg=cfg["reg"], model_alpha=1.0,
                        iters=cfg["iters"], seed=cfg["seed"])
    recs, _, _ = eval_als_on(m, candA, up, ip, m.item_factors, m.user_factors, pop_fallback_A, K=K)
    m1, _ = map_at5(recs, gt1_eval)
    m2, _ = map_at5(recs, gt2_eval)
    als_run_log.append(dict(config=label, alpha_t=cfg["alpha_t"], factors=cfg["factors"],
                            reg=cfg["reg"], iters=cfg["iters"], min_os=cfg["min_os"],
                            map_gt1=m1, map_gt2=m2, fit_time=fit_t))
    print(f"{label:42s} | GT-1 {m1:.6f} | GT-2 {m2:.6f} | fit {fit_t:.2f}s")

print("\n[One-variable-at-a-time re-tune from base]")
run_tune("base: f=64 reg=0.01 it=15 min_os=60")
for f in [32, 128, 256]:
    run_tune(f"factors={f}", factors=f)
for r in [0.1, 1.0]:
    run_tune(f"reg={r}", reg=r)
for it in [5, 30]:
    run_tune(f"iters={it}", iters=it)
for mo in [0, 300]:
    run_tune(f"min_os={mo}", min_os=mo)

best_tune = max(als_run_log, key=lambda r: r["map_gt1"])
print(f"\nBest tuned config: {best_tune['config']} | GT-1={best_tune['map_gt1']:.6f} | GT-2={best_tune['map_gt2']:.6f}")

print("\n--- Full ALS tuning run log ---")
print(f"{'config':42s} | {'GT-1':>8s} | {'GT-2':>8s} | {'fit(s)':>6s}")
for r in als_run_log:
    print(f"{r['config']:42s} | {r['map_gt1']:8.6f} | {r['map_gt2']:8.6f} | {r['fit_time']:6.2f}")

# 3-seed stability for best config
seed_scores = []
for s in [42, 123, 777]:
    lab_seed = pairs[(pairs["tv_show_id"] != 0)
                     & (pairs["iso_week"].between(11, 20))
                     & (pairs["overlap_sec"] >= best_tune["min_os"])
                     & (pairs["overlap_frac"] >= 0.0)].copy()
    mat, up, ip = build_conf_matrix(lab_seed, best_tune["alpha_t"])
    m, _ = fit_als2(mat, factors=best_tune["factors"], reg=best_tune["reg"], model_alpha=1.0,
                    iters=best_tune["iters"], seed=s)
    recs, _, _ = eval_als_on(m, candA, up, ip, m.item_factors, m.user_factors, pop_fallback_A, K=K)
    m1, _ = map_at5(recs, gt1_eval)
    seed_scores.append(m1)
print(f"\nSeed stability (best config, 3 seeds): {[round(v,6) for v in seed_scores]} mean={np.mean(seed_scores):.6f} std={np.std(seed_scores):.6f}")

# Store best model + recs for later parts
best_als_alpha_t = best_tune["alpha_t"]
best_als_factors = best_tune["factors"]
best_als_reg = best_tune["reg"]
best_als_iters = best_tune["iters"]
best_als_min_os = best_tune["min_os"]

lab_best_als = pairs[(pairs["tv_show_id"] != 0)
                     & (pairs["iso_week"].between(11, 20))
                     & (pairs["overlap_sec"] >= best_als_min_os)
                     & (pairs["overlap_frac"] >= 0.0)].copy()
mat_best_als, u_pos_best_als, i_pos_best_als = build_conf_matrix(lab_best_als, best_als_alpha_t)
best_model_als, fit_t_best_als = fit_als2(mat_best_als, factors=best_als_factors, reg=best_als_reg,
                                          model_alpha=1.0, iters=best_als_iters, seed=42)
best_recs_als, _, _ = eval_als_on(best_model_als, candA, u_pos_best_als, i_pos_best_als,
                                   best_model_als.item_factors, best_model_als.user_factors,
                                   pop_fallback_A, K=K)

# Fold B with best Fold-A config (no tuning on Fold B)
print("\n[Fold B with best Fold-A config, no tuning]")
labB = pairs[(pairs["tv_show_id"] != 0) & (pairs["iso_week"].between(11, 18))
             & (pairs["overlap_sec"] >= best_als_min_os) & (pairs["overlap_frac"] >= 0.0)].copy()
pop_fallback_B = [int(s) for s in histB.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).index if int(s) in set(candB)][:K]
matB, u_posB, i_posB = build_conf_matrix(labB, best_als_alpha_t)
modelB_als, fit_tB_als = fit_als2(matB, factors=best_als_factors, reg=best_als_reg, model_alpha=1.0,
                                  iters=best_als_iters, seed=42)
recsB_als, _, _ = eval_als_on(modelB_als, candB, u_posB, i_posB,
                               modelB_als.item_factors, modelB_als.user_factors, pop_fallback_B, K=K)
m1B, e1B = map_at5(recsB_als, gt1_B)
m1zB, _ = map_at5(recsB_als, gt1_B, count_empty_as_zero=True)
m2B, e2B = map_at5(recsB_als, gt2_B)
m2zB, _ = map_at5(recsB_als, gt2_B, count_empty_as_zero=True)
print(f"  MAP@5 GT-1: {m1B:.6f} ({len(EVAL_USERS)-e1B} users) | empty=0: {m1zB:.6f}")
print(f"  MAP@5 GT-2: {m2B:.6f} ({len(EVAL_USERS)-e2B} users) | empty=0: {m2zB:.6f}")


PART 2 — ALS re-tuning (Fold A, keep-watched, GT-1 + GT-2)
  NOTE: each row rebuilds interactions from matched_pairs with that row's own min_overlap_sec.

[Extend grid downward]
  alpha_t=0.001                        | GT-1 0.097044 | GT-2 0.287954 | fit 2.50s
  alpha_t=0.003                        | GT-1 0.097509 | GT-2 0.288894 | fit 2.50s

Best alpha_t after grid extension: 0.01 | GT-1 = 0.098475 | GT-2 = 0.290533

[One-variable-at-a-time re-tune from base]
base: f=64 reg=0.01 it=15 min_os=60        | GT-1 0.098475 | GT-2 0.290533 | fit 2.50s
factors=32                                 | GT-1 0.122533 | GT-2 0.368034 | fit 1.21s
factors=128                                | GT-1 0.071038 | GT-2 0.223945 | fit 3.95s
factors=256                                | GT-1 0.053333 | GT-2 0.178805 | fit 7.56s
reg=0.1                                    | GT-1 0.098890 | GT-2 0.291192 | fit 2.52s
reg=1.0                                    | GT-1 0.099869 | GT-2 0.298861 | fit 2.50s
iters=5      

#### Correction — invalid min_overlap_sec rows

In the first version of the table above, the rows `min_os=0` and `min_os=300` were identical to the base row: the matrix was always built from `labA`, which had already been filtered at `overlap_sec >= 60`, so the override never reached the matrix. Those two rows were invalid and are replaced by the current run, where `run_tune` rebuilds interactions from `matched_pairs` with each row's own `min_overlap_sec`.


### Part 3 — note: SVD vs implicit ALS

TruncatedSVD treats unobserved entries as 0 (no confidence weighting, no implicit-feedback formulation). ALS with confidence C = 1 + alpha_t*log1p(sec) weights observed entries higher and treats unobserved as confidence=1. SVD minimizes plain squared error on observed log1p(seconds) values; ALS minimizes confidence-weighted squared error with p_ui in {0,1}. The difference in MAP comes from this treatment: SVD distributes error uniformly across all entries, while ALS focuses on correctly ranking observed (watched) items. Both use the same user x show matrix and the same candidate pool.


In [75]:
# --- Part 3: SVD variant (Fold A, keep-watched) ---
from sklearn.decomposition import TruncatedSVD

print("=" * 72)
print("PART 3 — SVD variant (Fold A, keep-watched, GT-1 + GT-2)")
print("  TruncatedSVD on log1p(total_seconds) matrix, deterministic (random_state=42)")
print("=" * 72)

# Build log1p matrix (same structure as c_ind_code TEST2)
g = labA.groupby(["user_id", "tv_show_id"])["overlap_sec"].sum().reset_index(name="sec")
users_svd = sorted(g["user_id"].unique())
items_svd = sorted(g["tv_show_id"].unique())
up_svd = {u: i for i, u in enumerate(users_svd)}
ip_svd = {i: j for j, i in enumerate(items_svd)}
sec_svd = np.log1p(g["sec"].astype("float64").values)
Mlog = sp.csr_matrix((sec_svd, (g["user_id"].map(up_svd).astype(np.int32).values,
                                g["tv_show_id"].map(ip_svd).astype(np.int32).values)),
                     shape=(len(users_svd), len(items_svd)))
print(f"log1p matrix: {Mlog.shape[0]} users x {Mlog.shape[1]} items, nnz={Mlog.nnz:,}")

cand_cols_svd = np.array([ip_svd[s] for s in candA if s in ip_svd], dtype=np.int64)
cand_shows_svd = np.array([int(s) for s in candA if s in ip_svd], dtype=np.int64)
pop_fallback_A = [int(s) for s in histA.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).index if int(s) in set(candA)][:K]

def score_svd(Us, Vs, up_svd, ip_svd, cand_cols, cand_shows, pop_fallback):
    recs = {}
    scores_all = Us @ Vs
    for u in EVAL_USERS:
        if u not in up_svd:
            recs[u] = pop_fallback[:K]
            continue
        ui = up_svd[u]
        sub = scores_all[ui][cand_cols]
        top = np.argpartition(-sub, kth=K - 1)[:K]
        top = top[np.argsort(-sub[top])]
        recs[u] = [int(cand_shows[int(c)]) for c in top]
    return recs

svd_results = []
for k in [16, 32, 64, 128]:
    svd = TruncatedSVD(n_components=k, random_state=42)
    Us = svd.fit_transform(Mlog)
    Vs = svd.components_
    recs = score_svd(Us, Vs, up_svd, ip_svd, cand_cols_svd, cand_shows_svd, pop_fallback_A)
    m1, e1 = map_at5(recs, gt1_eval)
    m2, e2 = map_at5(recs, gt2_eval)
    svd_results.append((k, m1, m2, Us, Vs))
    print(f"SVD k={k:4d} | GT-1 {m1:.6f} ({len(EVAL_USERS)-e1} users) | GT-2 {m2:.6f} ({len(EVAL_USERS)-e2} users)")

# Binary matrix SVD reference
Mbin = (Mlog > 0).astype(np.float64)  # binary matrix, same sparsity pattern
svd_bin = TruncatedSVD(n_components=64, random_state=42)
Us_bin = svd_bin.fit_transform(Mbin)
Vs_bin = svd_bin.components_
recs_bin = score_svd(Us_bin, Vs_bin, up_svd, ip_svd, cand_cols_svd, cand_shows_svd, pop_fallback_A)
m1_bin, _ = map_at5(recs_bin, gt1_eval)
m2_bin, _ = map_at5(recs_bin, gt2_eval)
print(f"SVD k=64 binary | GT-1 {m1_bin:.6f} | GT-2 {m2_bin:.6f}")

best_k = max(svd_results, key=lambda x: x[1])[0]
best_svd = max(svd_results, key=lambda x: x[1])
print(f"\nBest SVD: k={best_k} | GT-1={best_svd[1]:.6f} | GT-2={best_svd[2]:.6f}")

# Store best SVD for Parts 4-5
best_svd_k = best_k
best_svd_Us = best_svd[3]
best_svd_Vs = best_svd[4]
best_recs_svd = score_svd(best_svd_Us, best_svd_Vs, up_svd, ip_svd, cand_cols_svd, cand_shows_svd, pop_fallback_A)

# Hand-check: recompute one (user, show) score by hand from factors
probe_u_svd = next(u for u in EVAL_USERS if u in up_svd)
probe_s = int(candA[0]) if int(candA[0]) in ip_svd else int([s for s in candA if s in ip_svd][0])
ui = up_svd[probe_u_svd]
ij = ip_svd[probe_s]
hand_score = float(np.dot(best_svd_Us[ui], best_svd_Vs[:, ij]))
scores_all = best_svd_Us @ best_svd_Vs
full_score = float(scores_all[ui][ij])
print(f"\nHand-check user={probe_u_svd}, show={probe_s}:")
print(f"  hand dot product: {hand_score:.6f}")
print(f"  full matrix score: {full_score:.6f}")
print(f"  difference: {abs(hand_score - full_score):.2e}")

# Fold B with best k
print("\n[Fold B with best SVD k, no tuning]")
labB_svd = pairs[(pairs["tv_show_id"] != 0) & (pairs["iso_week"].between(11, 18))
                 & (pairs["overlap_sec"] >= best_als_min_os) & (pairs["overlap_frac"] >= 0.0)].copy()
gB = labB_svd.groupby(["user_id", "tv_show_id"])["overlap_sec"].sum().reset_index(name="sec")
usersB = sorted(gB["user_id"].unique())
itemsB = sorted(gB["tv_show_id"].unique())
upB = {u: i for i, u in enumerate(usersB)}
ipB = {i: j for j, i in enumerate(itemsB)}
secB = np.log1p(gB["sec"].astype("float64").values)
MB = sp.csr_matrix((secB, (gB["user_id"].map(upB).astype(np.int32).values,
                           gB["tv_show_id"].map(ipB).astype(np.int32).values)),
                   shape=(len(usersB), len(itemsB)))
svdB = TruncatedSVD(n_components=best_svd_k, random_state=42)
UsB = svdB.fit_transform(MB)
VsB = svdB.components_
scoresB = UsB @ VsB
cand_colsB = np.array([ipB[s] for s in candB if s in ipB], dtype=np.int64)
cand_showsB = np.array([int(s) for s in candB if s in ipB], dtype=np.int64)
pop_fallback_B = [int(s) for s in histB.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).index if int(s) in set(candB)][:K]
recsB_svd = {}
for u in EVAL_USERS:
    if u not in upB:
        recsB_svd[u] = pop_fallback_B[:K]
        continue
    ui = upB[u]
    sub = scoresB[ui][cand_colsB]
    top = np.argpartition(-sub, kth=K - 1)[:K]
    top = top[np.argsort(-sub[top])]
    recsB_svd[u] = [int(cand_showsB[int(c)]) for c in top]
m1B_svd, e1B_svd = map_at5(recsB_svd, gt1_B)
m2B_svd, e2B_svd = map_at5(recsB_svd, gt2_B)
print(f"  MAP@5 GT-1: {m1B_svd:.6f} ({len(EVAL_USERS)-e1B_svd} users)")
print(f"  MAP@5 GT-2: {m2B_svd:.6f} ({len(EVAL_USERS)-e2B_svd} users)")


PART 3 — SVD variant (Fold A, keep-watched, GT-1 + GT-2)
  TruncatedSVD on log1p(total_seconds) matrix, deterministic (random_state=42)
log1p matrix: 4841 users x 2975 items, nnz=842,560
SVD k=  16 | GT-1 0.213451 (1246 users) | GT-2 0.533531 (1244 users)
SVD k=  32 | GT-1 0.221413 (1246 users) | GT-2 0.507973 (1244 users)
SVD k=  64 | GT-1 0.210407 (1246 users) | GT-2 0.466198 (1244 users)
SVD k= 128 | GT-1 0.201758 (1246 users) | GT-2 0.434396 (1244 users)
SVD k=64 binary | GT-1 0.094358 | GT-2 0.292129

Best SVD: k=32 | GT-1=0.221413 | GT-2=0.507973

Hand-check user=10008352999832730058, show=20088:
  hand dot product: 9.379401
  full matrix score: 9.379401
  difference: 1.78e-15

[Fold B with best SVD k, no tuning]
  MAP@5 GT-1: 0.225958 (1246 users)
  MAP@5 GT-2: 0.509013 (1240 users)


### Part 4 — comparison table (Fold A and Fold B)

Same 1,260 evaluation users in both folds. GT-1 = top-5 by watched seconds in the target window; GT-2 = all shows with an airing watched >= 50%. `empty=0` counts users with empty truth as 0. ALS and SVD fit/scoring times are measured in this cell (deterministic models, same data as above).


In [76]:
# --- Part 4: comparison table (Fold A + Fold B) ---
import time

print("=" * 72)
print("PART 4 — comparison table (Fold A and Fold B, same users, GT-1 + GT-2)")
print("=" * 72)

def metrics(recs, gt1, gt2):
    m1, e1 = map_at5(recs, gt1)
    m1z, _ = map_at5(recs, gt1, count_empty_as_zero=True)
    m2, e2 = map_at5(recs, gt2)
    m2z, _ = map_at5(recs, gt2, count_empty_as_zero=True)
    return m1, m1z, m2, m2z

alsA = metrics(best_recs_als, gt1_eval, gt2_eval)
svdA = metrics(best_recs_svd, gt1_eval, gt2_eval)
alsB = metrics(recsB_als, gt1_B, gt2_B)
svdB = metrics(recsB_svd, gt1_B, gt2_B)

# timing: rescore ALS on Fold A, refit+rescore SVD on Fold A and Fold B
t0 = time.time()
_ = eval_als_on(best_model_als, candA, u_pos_best_als, i_pos_best_als,
                best_model_als.item_factors, best_model_als.user_factors, pop_fallback_A, K=K)
als_score_t = time.time() - t0
svd_tmp = TruncatedSVD(n_components=best_svd_k, random_state=42)
t0 = time.time()
_ = svd_tmp.fit_transform(Mlog)
svd_fit_t = time.time() - t0
t0 = time.time()
_ = score_svd(best_svd_Us, best_svd_Vs, up_svd, ip_svd, cand_cols_svd, cand_shows_svd, pop_fallback_A)
svd_score_t = time.time() - t0
svdB_tmp = TruncatedSVD(n_components=best_svd_k, random_state=42)
t0 = time.time()
UsB_tmp = svdB_tmp.fit_transform(MB)
svdB_fit_t = time.time() - t0
cand_colsB_t = np.array([ipB[s] for s in candB if s in ipB], dtype=np.int64) if "ipB" in globals() else None
# NOTE: Fold A baseline values are rebuilt from the stored recs dicts (same seeds) so this table is independent of the section 6 print loop.
baseA = {
    "Random (avg 5 seeds)": tuple(float(np.mean([vals[j] for vals in
        [metrics(rec_random(1000 + s), gt1_eval, gt2_eval) for s in range(5)]])) for j in range(4)),
    "Popularity-by-users": metrics(recs_pop_users, gt1_eval, gt2_eval),
    "Popularity-by-seconds": metrics(recs_pop_secs, gt1_eval, gt2_eval),
    "Personal history [diagnostic, not counted]": metrics(recs_personal, gt1_eval, gt2_eval),
}
baseB = {name: (m1, m1z, m2, m2z) for (name, m1, m2, m1z, m2z, nu1, nu2) in rows_B}

def fmt_row(name, vals, fit_s, score_s):
    m1, m1z, m2, m2z = vals
    f = f"{fit_s:7.2f}" if fit_s is not None else "      -"
    s = f"{score_s:7.2f}" if score_s is not None else "      -"
    return f"{name:34s} | {m1:10.6f} | {m2:10.6f} | {m1z:12.6f} | {m2z:12.6f} | {f} | {s}"

def show_fold(title, base, als_m, als_fit, als_score, svd_m, svd_fit, svd_score):
    print(f"\n--- {title} ---")
    print(f"{'method':34s} | {'GT-1':>10s} | {'GT-2':>10s} | {'GT-1 e=0':>12s} | {'GT-2 e=0':>12s} | {'fit(s)':>7s} | {'score(s)':>7s}")
    print(fmt_row("Random (avg 5 seeds)", base["Random (avg 5 seeds)"], None, None))
    print(fmt_row("Popularity-by-users", base["Popularity-by-users"], None, None))
    print(fmt_row("Popularity-by-seconds", base["Popularity-by-seconds"], None, None))
    print(fmt_row("Personal history [diagnostic]", base["Personal history [diagnostic, not counted]"], None, None))
    als_name = f"Best ALS (alpha_t={best_als_alpha_t:g}, f={best_als_factors})"
    svd_name = f"Best SVD (k={best_svd_k})"
    print(fmt_row(als_name, als_m, als_fit, als_score))
    print(fmt_row(svd_name, svd_m, svd_fit, svd_score))

show_fold("Fold A: history w11-20 -> target w21-30", baseA, alsA, fit_t_best_als, als_score_t, svdA, svd_fit_t, svd_score_t)
show_fold("Fold B: history w11-18 -> target w19-28", baseB, alsB, fit_tB_als, None, svdB, svdB_fit_t, None)


PART 4 — comparison table (Fold A and Fold B, same users, GT-1 + GT-2)

--- Fold A: history w11-20 -> target w21-30 ---
method                             |       GT-1 |       GT-2 |     GT-1 e=0 |     GT-2 e=0 |  fit(s) | score(s)
Random (avg 5 seeds)               |   0.000506 |   0.003538 |     0.000501 |     0.003493 |       - |       -
Popularity-by-users                |   0.177103 |   0.553070 |     0.175135 |     0.546047 |       - |       -
Popularity-by-seconds              |   0.180193 |   0.541744 |     0.178190 |     0.534865 |       - |       -
Personal history [diagnostic]      |   0.301359 |   0.510529 |     0.298011 |     0.504046 |       - |       -
Best ALS (alpha_t=0.01, f=32)      |   0.122533 |   0.368034 |     0.121172 |     0.363360 |    1.22 |    0.09
Best SVD (k=32)                    |   0.221413 |   0.507973 |     0.218952 |     0.501523 |    0.10 |    0.23

--- Fold B: history w11-18 -> target w19-28 ---
method                             |       GT-1 |    

### Part 5 — diagnostics: activity terciles, repeat vs new hits, SVD case-study lists

Terciles are computed on the number of labelled airings per user in the Fold-A history (same groups as in section 7). 'Repeat' means a GT-1 hit that is already in the user's history; 'new' means a GT-1 hit absent from that user's history. SVD top-5 uses the best k chosen on Fold A.


In [77]:
# --- Part 5: diagnostics (Fold A, GT-1) ---
print("=" * 72)
print("PART 5 — diagnostics (Fold A, GT-1)")
print("=" * 72)

methods = {
    "Popularity-by-seconds": recs_pop_secs,
    "Personal history [diagnostic]": recs_personal,
    "Best ALS": best_recs_als,
    "Best SVD": best_recs_svd,
}

print("\n[MAP@5 GT-1 by user activity tercile (labelled airings in history)]")
print(f"{'method':32s} | {'light':>8s} | {'medium':>8s} | {'heavy':>8s} | n(l/m/h)")
for mname, recs in methods.items():
    vals = []
    for g in ["light", "medium", "heavy"]:
        gusers = groups[g]
        mg, _ = map_at5({u: recs.get(u, []) for u in gusers},
                        {u: gt1_eval.get(u, set()) for u in gusers})
        vals.append(mg)
    print(f"{mname:32s} | {vals[0]:8.6f} | {vals[1]:8.6f} | {vals[2]:8.6f} | {len(groups['light'])}/{len(groups['medium'])}/{len(groups['heavy'])}")

hist_sets = labA.groupby("user_id")["tv_show_id"].apply(lambda s: set(int(x) for x in s)).to_dict()
print("\n[GT-1 hits: already in history (repeat) vs new shows]")
print(f"{'method':32s} | {'hits':>5s} | {'repeat':>6s} | {'new':>6s} | {'repeat_share':>12s}")
for mname, recs in methods.items():
    n_hits = 0
    n_rep = 0
    for u in EVAL_USERS:
        h = hist_sets.get(u, set())
        for s in recs.get(u, [])[:K]:
            if int(s) in gt1_eval.get(u, set()):
                n_hits += 1
                if int(s) in h:
                    n_rep += 1
    share = (n_rep / n_hits) if n_hits else 0.0
    print(f"{mname:32s} | {n_hits:5d} | {n_rep:6d} | {n_hits - n_rep:6d} | {share:12.4f}")

print("\n[SVD top-5 for the two case-study users]")
for u in CASE_USERS:
    print(f"\n=== case user {u} (labelled airings in history: {int(labelled_counts_all.get(u, 0))}) ===")
    hist_u = labA[labA["user_id"] == u].groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).head(5)
    print("top-5 most-watched in history:")
    for sid, sec in hist_u.items():
        sid_i, title, cat, genres = show_info(sid)
        print(f"  {sid_i:6d} | {sec:7.0f}s | {cat:12s} | {genres:30s} | {title}")
    print("GT-1 top-5:", sorted(int(s) for s in gt1_eval.get(u, set())))
    print("SVD top-5:")
    for rank, sid in enumerate(best_recs_svd.get(u, [])[:K], 1):
        sid_i, title, cat, genres = show_info(sid)
        hit = "HIT" if int(sid) in gt1_eval.get(u, set()) else "-"
        print(f"  {rank}. {sid_i:6d} | {cat:12s} | {genres:26s} | {str(title)[:60]} | {hit}")


PART 5 — diagnostics (Fold A, GT-1)

[MAP@5 GT-1 by user activity tercile (labelled airings in history)]
method                           |    light |   medium |    heavy | n(l/m/h)
Popularity-by-seconds            | 0.117772 | 0.179799 | 0.239494 | 416/416/428
Personal history [diagnostic]    | 0.181015 | 0.327424 | 0.389743 | 416/416/428
Best ALS                         | 0.130932 | 0.150709 | 0.087352 | 416/416/428
Best SVD                         | 0.167871 | 0.240153 | 0.253824 | 416/416/428

[GT-1 hits: already in history (repeat) vs new shows]
method                           |  hits | repeat |    new | repeat_share
Popularity-by-seconds            |  1545 |   1512 |     33 |       0.9786
Personal history [diagnostic]    |  2279 |   2276 |      3 |       0.9987
Best ALS                         |  1181 |   1170 |     11 |       0.9907
Best SVD                         |  1797 |   1792 |      5 |       0.9972

[SVD top-5 for the two case-study users]

=== case user 1357119904662373

## 8. Method B


### Method B — CBF-lite (TF-IDF over show metadata)

Item table: one row per tv_show_id taken from `schedule_nd` (first non-null airing). Text variants: (a) category + 3 genres; (b) + director and actors; (c) + title words. TF-IDF (word tokens, sublinear_tf, L2-normalized rows). User profile = sum of item vectors weighted by log1p(total watched seconds) in the history window only, then L2-normalized; score = cosine over candidate-pool shows. Already-watched shows are kept; users with no history get popularity-by-seconds. Best variant is chosen on Fold A only and applied unchanged to Fold B. Ablation variant (d) repeats the best text set with binary weights instead of log1p.


In [78]:
# --- Section 8: CBF-lite (TF-IDF over show metadata) ---
from sklearn.feature_extraction.text import TfidfVectorizer
import time

print("=" * 72)
print("SECTION 8 — CBF-lite (TF-IDF over show metadata)")
print("=" * 72)

META_COLS = ["tv_show_id", "tv_show_title", "tv_show_category",
             "tv_show_genre_1", "tv_show_genre_2", "tv_show_genre_3",
             "director", "actors"]
sched_meta = schedule_nd[META_COLS].dropna(subset=["tv_show_id"]).copy()
sched_meta["tv_show_id"] = sched_meta["tv_show_id"].astype(int)
sched_meta = sched_meta[sched_meta["tv_show_id"] != 0]
first_item = sched_meta.drop_duplicates("tv_show_id").set_index("tv_show_id")

n_shows = len(first_item)
print(f"shows with metadata in schedule: {n_shows:,}")

def n_distinct(s):
    return len(set(str(v) for v in s.dropna().unique()))

print("\n[metadata consistency across airings]")
print(f"{'field':22s} | {'shows with >1 value':>20s} | share")
for col in ["tv_show_title", "tv_show_category", "tv_show_genre_1", "director", "actors"]:
    nd = sched_meta.groupby("tv_show_id")[col].agg(n_distinct)
    bad = int((nd > 1).sum())
    print(f"{col:22s} | {bad:20,d} | {bad / n_shows:.4%}")

print("\n[candidate-pool missingness (Fold A pool)]")
candA_ids = [int(s) for s in candA]
poolA_meta = [s for s in candA_ids if s in first_item.index]
print(f"pool shows with any metadata row: {len(poolA_meta):,} / {len(candA_ids):,}")
for col in ["tv_show_category", "tv_show_genre_1", "tv_show_genre_2", "tv_show_genre_3", "director", "actors", "tv_show_title"]:
    miss = first_item.loc[poolA_meta, col].isna().mean()
    print(f"  missing {col:18s}: {miss:.2%}")

def _cat(row):
    return "" if pd.isna(row["tv_show_category"]) else str(row["tv_show_category"]).lower().replace(" ", "_")

def _genres(row):
    parts = []
    for c in ["tv_show_genre_1", "tv_show_genre_2", "tv_show_genre_3"]:
        if pd.notna(row[c]):
            parts.append(str(row[c]).lower().replace(" ", "_"))
    return " ".join(parts)

def _people(row):
    parts = []
    for c in ["director", "actors"]:
        if pd.notna(row[c]):
            parts.append(str(row[c]).lower().replace(",", " ").replace(" ", "_"))
    return " ".join(parts)

def _title(row):
    return "" if pd.isna(row["tv_show_title"]) else str(row["tv_show_title"]).lower()

def text_a(row):
    return _cat(row) + " " + _genres(row)

def text_b(row):
    return text_a(row) + " " + _people(row)

def text_c(row):
    return text_b(row) + " " + _title(row)

def cbf_fit_eval(text_fn, weight_mode, lab_hist, cand, gt1, gt2, pop_fb, tag):
    t0 = time.time()
    cand_ids = [int(s) for s in cand]
    hist_ids = set(int(s) for s in lab_hist["tv_show_id"].unique())
    ids = sorted(s for s in (set(cand_ids) | hist_ids) if s in first_item.index)
    docs = [text_fn(first_item.loc[s]) for s in ids]
    vec = TfidfVectorizer(token_pattern=r"[^\s]+", lowercase=True, sublinear_tf=True, norm="l2")
    X = vec.fit_transform(docs)
    idx = {s: i for i, s in enumerate(ids)}
    g = (lab_hist[lab_hist["tv_show_id"].isin(idx)]
         .groupby(["user_id", "tv_show_id"])["overlap_sec"].sum().reset_index())
    users = sorted(g["user_id"].unique())
    upos = {u: i for i, u in enumerate(users)}
    if weight_mode == "log":
        w = np.log1p(g["overlap_sec"].values.astype("float64"))
    else:
        w = np.ones(len(g), dtype="float64")
    Wmat = sp.csr_matrix((w, (g["user_id"].map(upos).astype(np.int32).values,
                              g["tv_show_id"].map(idx).astype(np.int32).values)),
                         shape=(len(users), len(ids)))
    P = Wmat @ X
    norms = np.sqrt(np.asarray(P.multiply(P).sum(axis=1)).ravel())
    norms[norms == 0] = 1.0
    P = sp.diags(1.0 / norms) @ P
    fit_s = time.time() - t0
    t0 = time.time()
    col_cand = np.array([idx[s] for s in cand_ids if s in idx], dtype=np.int64)
    shows_cand = np.array([s for s in cand_ids if s in idx], dtype=np.int64)
    S = (P @ X[col_cand].T).toarray()
    recs = {}
    for u in EVAL_USERS:
        if u not in upos:
            recs[u] = pop_fb[:K]
            continue
        row = S[upos[u]]
        if not np.isfinite(row).any():
            recs[u] = pop_fb[:K]
            continue
        top = np.argpartition(-row, kth=K - 1)[:K]
        top = top[np.argsort(-row[top])]
        recs[u] = [int(shows_cand[int(c)]) for c in top]
    score_s = time.time() - t0
    m1, e1 = map_at5(recs, gt1)
    m1z, _ = map_at5(recs, gt1, count_empty_as_zero=True)
    m2, e2 = map_at5(recs, gt2)
    m2z, _ = map_at5(recs, gt2, count_empty_as_zero=True)
    print(f"{tag:30s} | GT-1 {m1:.6f} ({len(EVAL_USERS)-e1}) | GT-2 {m2:.6f} ({len(EVAL_USERS)-e2}) | fit {fit_s:5.2f}s | score {score_s:5.2f}s")
    return dict(tag=tag, recs=recs, m1=m1, m2=m2, m1z=m1z, m2z=m2z,
                fit_s=fit_s, score_s=score_s, n_items=len(ids), X=X, P=P, idx=idx,
                shows_cand=shows_cand, upos=upos)

print("\n--- Fold A ablation (history w11-20 -> target w21-30) ---")
cbf_A = []
cbf_A.append(cbf_fit_eval(text_a, "log", labA, candA, gt1_eval, gt2_eval, pop_fallback_A, "(a) cat+genres, log1p"))
cbf_A.append(cbf_fit_eval(text_b, "log", labA, candA, gt1_eval, gt2_eval, pop_fallback_A, "(b) +director/actors"))
cbf_A.append(cbf_fit_eval(text_c, "log", labA, candA, gt1_eval, gt2_eval, pop_fallback_A, "(c) +title words"))
best_by_a = max(cbf_A, key=lambda r: r["m1"])
best_text = {"(a)": text_a, "(b)": text_b, "(c)": text_c}
bt = best_text.get(best_by_a["tag"][:3], text_c)
cbf_A.append(cbf_fit_eval(bt, "binary", labA, candA, gt1_eval, gt2_eval, pop_fallback_A, "(d) best text, binary"))

print("\n--- Fold B ablation (history w11-18 -> target w19-28) ---")
cbf_B = []
cbf_B.append(cbf_fit_eval(text_a, "log", labB, candB, gt1_B, gt2_B, pop_fallback_B, "(a) cat+genres, log1p"))
cbf_B.append(cbf_fit_eval(text_b, "log", labB, candB, gt1_B, gt2_B, pop_fallback_B, "(b) +director/actors"))
cbf_B.append(cbf_fit_eval(text_c, "log", labB, candB, gt1_B, gt2_B, pop_fallback_B, "(c) +title words"))
cbf_B.append(cbf_fit_eval(bt, "binary", labB, candB, gt1_B, gt2_B, pop_fallback_B, "(d) best text, binary"))

best_cbf_A = max(cbf_A, key=lambda r: r["m1"])
tag_best = best_cbf_A["tag"]
best_cbf_B = next(r for r in cbf_B if r["tag"] == tag_best)
best_cbf_recs_A = best_cbf_A["recs"]
best_cbf_recs_B = best_cbf_B["recs"]
print(f"\nbest CBF on Fold A: {tag_best} | GT-1 {best_cbf_A['m1']:.6f} | GT-2 {best_cbf_A['m2']:.6f}")
print(f"same config on Fold B: GT-1 {best_cbf_B['m1']:.6f} | GT-2 {best_cbf_B['m2']:.6f}")

# --- hand-check one cosine by hand ---
probe_u = next(u for u in EVAL_USERS if u in best_cbf_A["upos"])
probe_s = int(best_cbf_A["shows_cand"][0])
up_ = best_cbf_A["upos"][probe_u]
ip_ = best_cbf_A["idx"][probe_s]
prof = np.asarray(best_cbf_A["P"][up_].todense()).ravel()
item = np.asarray(best_cbf_A["X"][ip_].todense()).ravel()
hand_cos = float(np.dot(prof, item) / (np.linalg.norm(prof) * np.linalg.norm(item)))
col_ = int(np.where(best_cbf_A["shows_cand"] == probe_s)[0][0])
mat_cos = None
S_chk = (best_cbf_A["P"] @ best_cbf_A["X"][np.array([best_cbf_A["idx"][probe_s]])].T)
mat_cos = float(np.asarray(S_chk[up_, 0]).ravel()[0])
print(f"\nhand-check user={probe_u}, show={probe_s}:")
print(f"  cosine by hand: {hand_cos:.6f}")
print(f"  matrix value  : {mat_cos:.6f}")
print(f"  difference    : {abs(hand_cos - mat_cos):.2e}")

# --- new-show share of GT-1 hits: CBF vs best MF ---
global_hist_A = set(int(s) for s in labA["tv_show_id"].unique())
hist_sets_A = labA.groupby("user_id")["tv_show_id"].apply(lambda s: set(int(x) for x in s)).to_dict()

def new_show_stats(recs, gt, hist_global):
    hits = 0
    new = 0
    rep = 0
    for u in EVAL_USERS:
        for s in recs.get(u, [])[:K]:
            if int(s) in gt.get(u, set()):
                hits += 1
                if int(s) in hist_global:
                    rep += 1
                else:
                    new += 1
    return hits, rep, new

print("\n[GT-1 hits: shows never watched by anyone in history vs repeats]")
for name, recs in [("best CBF", best_cbf_recs_A), ("best SVD", best_recs_svd)]:
    h, r, n = new_show_stats(recs, gt1_eval, global_hist_A)
    share = n / h if h else 0.0
    print(f"  {name:10s} | hits {h:5d} | repeat {r:5d} | new {n:4d} | new_share {share:.4f}")

cbf_cost = dict(fit_s=best_cbf_A["fit_s"], score_s=best_cbf_A["score_s"], n_items=best_cbf_A["n_items"])
print(f"\nbest CBF cost (Fold A): fit {cbf_cost['fit_s']:.2f}s | score {cbf_cost['score_s']:.2f}s | items {cbf_cost['n_items']:,}")

SECTION 8 — CBF-lite (TF-IDF over show metadata)
shows with metadata in schedule: 9,671

[metadata consistency across airings]
field                  |  shows with >1 value | share
tv_show_title          |                4,351 | 44.9902%
tv_show_category       |                    0 | 0.0000%
tv_show_genre_1        |                   80 | 0.8272%
director               |                   48 | 0.4963%
actors                 |                  174 | 1.7992%

[candidate-pool missingness (Fold A pool)]
pool shows with any metadata row: 6,376 / 6,376
  missing tv_show_category  : 0.00%
  missing tv_show_genre_1   : 2.13%
  missing tv_show_genre_2   : 38.14%
  missing tv_show_genre_3   : 76.58%
  missing director          : 2.35%
  missing actors            : 1.96%
  missing tv_show_title     : 0.00%

--- Fold A ablation (history w11-20 -> target w21-30) ---
(a) cat+genres, log1p          | GT-1 0.001247 (1246) | GT-2 0.009469 (1244) | fit  0.65s | score  0.40s
(b) +director/actors        

## 9. Hybrid


### Hybrid — MF + CBF + personal history + popularity

Components: best MF (SVD k=32 scores), best CBF (TF-IDF variant chosen on Fold A), personal-history score (log1p watched seconds in the history window), popularity-by-seconds (log1p, same window). ONE normalization: per-user min-max over candidate-pool scores, applied to each component separately. Weights: step 0.25 on the 4-simplex, 15 combinations, selected on Fold A by GT-1 only, then fixed on Fold B. Every combination is printed for both folds. Caveat: Fold B is not fully independent of Fold A (Fold A history weeks 11-20 overlaps Fold B target weeks 19-28).
 Note: personal history is marked "diagnostic, not counted" in sections 6-7; here it is a weighted component and the tuner is allowed to select it.


In [79]:
# --- Section 9: hybrid-lite ---
import time

print("=" * 72)
print("SECTION 9 — hybrid-lite (MF + CBF + personal history + popularity)")
print("  normalization: per-user min-max over candidate-pool scores")
print("  grid: step 0.25 on the 4-simplex, 15 combinations, tuned on Fold A only")
print("=" * 72)

def per_user_minmax(S):
    S = np.array(S, dtype="float64", copy=True)
    mins = np.nanmin(S, axis=1)
    maxs = np.nanmax(S, axis=1)
    rng = maxs - mins
    rng[rng == 0] = 1.0
    out = (S - mins[:, None]) / rng[:, None]
    out[~np.isfinite(out)] = 0.0
    return out

def build_components(cand, lab_hist, svd_Us, svd_Vs, svd_up, svd_i_pos, cbf_entry):
    pool_shows = np.array([int(s) for s in cand], dtype=np.int64)
    n_pool = len(pool_shows)
    n_users = len(EVAL_USERS)
    scores_all = svd_Us @ svd_Vs
    cols = np.array([svd_i_pos.get(int(s), -1) for s in pool_shows], dtype=np.int64)
    valid = cols >= 0
    mf = np.zeros((n_users, n_pool), dtype="float64")
    for i, u in enumerate(EVAL_USERS):
        if u in svd_up and valid.any():
            mf[i, valid] = scores_all[svd_up[u]][cols[valid]]
    P, X, idx, shows_cand, upos = (cbf_entry["P"], cbf_entry["X"], cbf_entry["idx"],
                                   cbf_entry["shows_cand"], cbf_entry["upos"])
    col_cand = np.array([idx[s] for s in shows_cand], dtype=np.int64)
    S_cbf = (P @ X[col_cand].T).toarray()
    pos_map = {int(s): j for j, s in enumerate(shows_cand)}
    cand_pos = np.array([pos_map.get(int(s), -1) for s in pool_shows], dtype=np.int64)
    valid_c = cand_pos >= 0
    cbf = np.zeros((n_users, n_pool), dtype="float64")
    for i, u in enumerate(EVAL_USERS):
        if u in upos:
            cbf[i, valid_c] = S_cbf[upos[u]][cand_pos[valid_c]]
    g = lab_hist.groupby(["user_id", "tv_show_id"])["overlap_sec"].sum().reset_index()
    pool_pos = {int(s): j for j, s in enumerate(pool_shows)}
    u2i = {u: i for i, u in enumerate(EVAL_USERS)}
    g2 = g[g["tv_show_id"].isin(pool_pos) & g["user_id"].isin(u2i)]
    own = np.zeros((n_users, n_pool), dtype="float64")
    if len(g2):
        rr = g2["user_id"].map(u2i).values.astype(np.int64)
        cc = g2["tv_show_id"].map(pool_pos).values.astype(np.int64)
        own[rr, cc] = np.log1p(g2["overlap_sec"].values.astype("float64"))
    pop_s = lab_hist.groupby("tv_show_id")["overlap_sec"].sum()
    pop_v = np.array([np.log1p(float(pop_s.get(int(s), 0.0))) for s in pool_shows])
    pop = np.tile(pop_v, (n_users, 1))
    return dict(mf=mf, cbf=cbf, own=own, pop=pop, pool_shows=pool_shows)

compA = build_components(candA, labA, best_svd_Us, best_svd_Vs, up_svd, ip_svd, best_cbf_A)
compB = build_components(candB, labB, best_svd_Us, best_svd_Vs, up_svd, ip_svd, best_cbf_B)
NAMES = ["mf", "cbf", "own", "pop"]
normA = {n: per_user_minmax(compA[n]) for n in NAMES}
normB = {n: per_user_minmax(compB[n]) for n in NAMES}

GRID = [(1,0,0,0), (0,1,0,0), (0,0,1,0), (0,0,0,1),
        (0.5,0.5,0,0), (0.75,0.25,0,0), (0.25,0.75,0,0),
        (0.5,0,0.5,0), (0.5,0,0,0.5), (0,0.5,0.5,0),
        (0,0.5,0,0.5), (0,0,0.5,0.5), (0.25,0.25,0.25,0.25),
        (0.5,0.25,0.25,0), (0.25,0.5,0.25,0)]

def hybrid_recs(norm, w, pool_shows, pop_fb):
    S = np.zeros_like(norm["mf"])
    for wi, n in zip(w, NAMES):
        if wi:
            S += wi * norm[n]
    pop_list = [int(s) for s in list(pop_fb)[:K]]
    recs = {}
    n_fb = 0
    for i, u in enumerate(EVAL_USERS):
        row = S[i]
        if not np.isfinite(row.max()) or row.max() <= 0:
            recs[u] = list(pop_list)
            n_fb += 1
            continue
        top = np.argpartition(-row, kth=K - 1)[:K]
        top = top[np.argsort(-row[top])]
        recs[u] = [int(pool_shows[int(c)]) for c in top]
    if n_fb:
        print(f"  [hybrid_recs] popularity fallback rows: {n_fb}")
    return recs

def grid_eval(norm, pool_shows, gt1, gt2, pop_fb, tag):
    out = []
    print(f"\n--- {tag} ---")
    print(f"{'w(mf,cbf,own,pop)':26s} | {'GT-1':>9s} | {'GT-2':>9s} | {'GT-1 e0':>9s}")
    for w in GRID:
        recs = hybrid_recs(norm, w, pool_shows, pop_fb)
        m1, _ = map_at5(recs, gt1)
        m1z, _ = map_at5(recs, gt1, count_empty_as_zero=True)
        m2, _ = map_at5(recs, gt2)
        out.append(dict(w=w, recs=recs, m1=m1, m2=m2, m1z=m1z))
        ws = "(" + ",".join(f"{x:.2f}" for x in w) + ")"
        print(f"{ws:26s} | {m1:9.6f} | {m2:9.6f} | {m1z:9.6f}")
    return out

grid_A = grid_eval(normA, compA["pool_shows"], gt1_eval, gt2_eval, pop_fallback_A, "Fold A (tuning)")
grid_B = grid_eval(normB, compB["pool_shows"], gt1_B, gt2_B, pop_fallback_B, "Fold B (fixed weights from A)")

best_A = max(grid_A, key=lambda r: r["m1"])
best_w = best_A["w"]
print(f"\nchosen weights (Fold A, GT-1): {best_w}")
best_B = next(r for r in grid_B if r["w"] == best_w)
best_hybrid_recs_A = best_A["recs"]
best_hybrid_recs_B = best_B["recs"]
print(f"Fold A hybrid: GT-1 {best_A['m1']:.6f} | GT-2 {best_A['m2']:.6f} | GT-1 e0 {best_A['m1z']:.6f}")
print(f"Fold B hybrid: GT-1 {best_B['m1']:.6f} | GT-2 {best_B['m2']:.6f} | GT-1 e0 {best_B['m1z']:.6f}")

print("\n--- final comparison table (Fold A | Fold B, GT-1 / GT-2 / GT-1 e0) ---")
rowsA = []
rowsB = []
rng_rec = {u: rec_random(1000)[u] for u in EVAL_USERS}
rngB = np.random.default_rng(1000)
poolB_list = [int(s) for s in candB]
rng_rec_B = {u: [int(x) for x in rngB.choice(poolB_list, size=K, replace=False)] for u in EVAL_USERS}
for name, recsA, recsB in [
    ("Random (seed 1000)", rng_rec, rng_rec_B),
    ("Popularity-by-users", recs_pop_users, recs_pop_users_B),
    ("Popularity-by-seconds", recs_pop_secs, recs_pop_secs_B),
    ("Personal history [diagnostic]", recs_personal, recs_personal_B),
    ("Best ALS", best_recs_als, recsB_als),
    ("Best SVD", best_recs_svd, recsB_svd),
    ("Best CBF", best_cbf_recs_A, best_cbf_recs_B),
    ("Hybrid", best_hybrid_recs_A, best_hybrid_recs_B),
]:
    if recsB is None:
        mB1, mB2, mB1z = float("nan"), float("nan"), float("nan")
    else:
        mB1, mB1z, mB2, _ = metrics(recsB, gt1_B, gt2_B)
    if recsA is None:
        mA1, mA2, mA1z = float("nan"), float("nan"), float("nan")
    else:
        mA1, mA1z, mA2, _ = metrics(recsA, gt1_eval, gt2_eval)
    rowsA.append((name, mA1, mA2, mA1z))
    rowsB.append((name, mB1, mB2, mB1z))
print(f"{'method':32s} | {'A GT-1':>8s} | {'A GT-2':>8s} | {'A GT-1e0':>8s} | {'B GT-1':>8s} | {'B GT-2':>8s} | {'B GT-1e0':>8s}")
for (n, a1, a2, az), (_, b1, b2, bz) in zip(rowsA, rowsB):
    print(f"{n:32s} | {a1:8.6f} | {a2:8.6f} | {az:8.6f} | {b1:8.6f} | {b2:8.6f} | {bz:8.6f}")

print("\n--- cost table (Fold A) ---")
t0 = time.time()
svd_tmp2 = TruncatedSVD(n_components=best_svd_k, random_state=42)
Us_tmp2 = svd_tmp2.fit_transform(Mlog)
Vs_tmp2 = svd_tmp2.components_
mf_fit_t = time.time() - t0
t0 = time.time()
_ = Us_tmp2 @ Vs_tmp2
mf_score_t = time.time() - t0
t0 = time.time()
_ = per_user_minmax(compA["mf"]) + per_user_minmax(compA["cbf"]) + per_user_minmax(compA["own"]) + per_user_minmax(compA["pop"])
norm_t = time.time() - t0
t0 = time.time()
_ = hybrid_recs(normA, best_w, compA["pool_shows"], pop_fallback_A)
hyb_score_t = time.time() - t0
print(f"{'component':22s} | {'fit(s)':>7s} | {'score/normalize(s)':>19s}")
print(f"{'MF (SVD)':22s} | {mf_fit_t:7.2f} | {mf_score_t:19.2f}")
print(f"{'CBF (TF-IDF)':22s} | {best_cbf_A['fit_s']:7.2f} | {best_cbf_A['score_s']:19.2f}")
print(f"{'personal history':22s} | {0.0:7.2f} | {0.0:19.2f}")
print(f"{'popularity':22s} | {0.0:7.2f} | {0.0:19.2f}")
print(f"{'all normalizations':22s} | {'-':>7s} | {norm_t:19.2f}")
print(f"{'hybrid scoring':22s} | {'-':>7s} | {hyb_score_t:19.2f}")

print("\n[MAP@5 GT-1 by activity tercile: best MF, CBF, hybrid (Fold A)]")
for mname, recs in [("Best SVD", best_recs_svd), ("Best CBF", best_cbf_recs_A), ("Hybrid", best_hybrid_recs_A)]:
    vals = []
    for g in ["light", "medium", "heavy"]:
        gusers = groups[g]
        mg, _ = map_at5({u: recs.get(u, []) for u in gusers},
                        {u: gt1_eval.get(u, set()) for u in gusers})
        vals.append(mg)
    print(f"{mname:10s} | light {vals[0]:.6f} | medium {vals[1]:.6f} | heavy {vals[2]:.6f}")


SECTION 9 — hybrid-lite (MF + CBF + personal history + popularity)
  normalization: per-user min-max over candidate-pool scores
  grid: step 0.25 on the 4-simplex, 15 combinations, tuned on Fold A only

--- Fold A (tuning) ---
w(mf,cbf,own,pop)          |      GT-1 |      GT-2 |   GT-1 e0
  [hybrid_recs] popularity fallback rows: 10
(1.00,0.00,0.00,0.00)      |  0.221413 |  0.507973 |  0.218952
  [hybrid_recs] popularity fallback rows: 10
(0.00,1.00,0.00,0.00)      |  0.023807 |  0.095296 |  0.023542
  [hybrid_recs] popularity fallback rows: 10
(0.00,0.00,1.00,0.00)      |  0.301423 |  0.510269 |  0.298074
(0.00,0.00,0.00,1.00)      |  0.180193 |  0.541744 |  0.178190
  [hybrid_recs] popularity fallback rows: 10
(0.50,0.50,0.00,0.00)      |  0.113044 |  0.293214 |  0.111788
  [hybrid_recs] popularity fallback rows: 10
(0.75,0.25,0.00,0.00)      |  0.193154 |  0.450700 |  0.191008
  [hybrid_recs] popularity fallback rows: 10
(0.25,0.75,0.00,0.00)      |  0.051367 |  0.167876 |  0.050796

## 9b. Repeat ranker — improving the own-history score (step 7)

The tuned hybrid selected (mf, cbf, own, pop) = (0, 0, 1, 0), and 97.9-99.9% of GT-1 hits are programs already in the user's history, so only the ordering inside the user's own history matters. One variable is changed at a time on Fold A (tuning); the winner is then applied to Fold B without re-tuning. At most 15 runs, all logged.

Variables: (a) per-program value = total seconds, log1p(seconds), number of airings, number of distinct days, number of airings with `overlap_frac >= 0.5`. (b) recency weight `0.5 ** (week_age / half_life)` with half-life in {infinite, 8, 4, 2, 1} weeks. (c) multiply the score by `log1p(airings of the program in the target-window schedule)`; the schedule is published in advance, so this is not leakage. (d) fill rule for users with fewer than 5 of their own programs in the candidate pool: popularity-by-seconds vs SVD ranking.

Step 0 first checks the provenance of the three already-submitted files against the `pop_fb` fix, because two users with empty history used to receive arbitrary rows.


In [80]:
# --- Section 9b, step 0: provenance of submitted files + final-window setup ---
import hashlib
import subprocess
from pathlib import Path

print("=" * 72)
print("STEP 0 — provenance of output files vs the pop_fb fix")
print("=" * 72)

FIX_COMMIT = "b8afb93"
SUB_FILES = ["outputs/submission_mf.csv", "outputs/submission_cbf.csv", "outputs/submission_hybrid.csv"]

def file_commit(path):
    r = subprocess.run(["git", "log", "-1", "--format=%H|%h|%ad|%s", "--date=short", "--", path],
                       capture_output=True, text=True)
    return r.stdout.strip()

def after_fix(path):
    info = file_commit(path)
    if not info:
        return False
    c = info.split("|")[0]
    r = subprocess.run(["git", "merge-base", "--is-ancestor", FIX_COMMIT, c])
    return r.returncode == 0

prov_after_fix = {}
for f in SUB_FILES:
    sha_f = hashlib.sha256(Path(f).read_bytes()).hexdigest()[:8]
    prov_after_fix[f] = after_fix(f)
    print()
    print(f)
    print("  sha256[:8] " + sha_f)
    print("  last commit: " + file_commit(f))
    print(f"  generated after {FIX_COMMIT}: {prov_after_fix[f]}")

# final-window setup (weeks 11-30 -> 31-42), self-contained
histR = pairs[(pairs["tv_show_id"] != 0) & (pairs["iso_week"].between(11, 30))
              & (pairs["overlap_sec"] >= 60) & (pairs["overlap_frac"] >= 0.0)].copy()
wk_all = schedule_nd["start_dt"].dt.isocalendar().week.astype(int)
schedR = schedule_nd[wk_all.between(31, 42) & (schedule_nd["tv_show_id"] != 0)]
candR = sorted(int(s) for s in schedR["tv_show_id"].unique() if int(s) != 0)
candR_set = set(candR)
assert len(candR) == 6642, len(candR)
print()
print(f"final setup: history {len(histR):,} interactions | pool {len(candR):,} shows | schedule rows {len(schedR):,}")

SAMPLE_R = pd.read_csv("data/submission.csv", dtype={"user_id": str})
sub_users_r = SAMPLE_R["user_id"].tolist()
pop_R = [int(s) for s in histR.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).index if int(s) in candR_set][:K]
assert len(pop_R) == K, pop_R

gR = histR.groupby(["user_id", "tv_show_id"])["overlap_sec"].sum().reset_index(name="sec")
uR = sorted(gR["user_id"].unique())
iR = sorted(gR["tv_show_id"].unique())
upR = {u: i for i, u in enumerate(uR)}
ipR = {s: j for j, s in enumerate(iR)}
MR = sp.csr_matrix((np.log1p(gR["sec"].astype("float64")),
                    (gR["user_id"].map(upR).astype(np.int32), gR["tv_show_id"].map(ipR).astype(np.int32))),
                   shape=(len(uR), len(iR)))
svdR = TruncatedSVD(n_components=best_svd_k, random_state=42)
UR = svdR.fit_transform(MR)
VR = svdR.components_
scR = UR @ VR
cand_cols_R = np.array([ipR[s] for s in candR if s in ipR], dtype=np.int64)
cand_shows_R = np.array([int(s) for s in candR if s in ipR], dtype=np.int64)
print(f"SVD fill model: k={best_svd_k} | pool shows with factors {len(cand_shows_R):,} / {len(candR):,}")

recs_svd_R = {}
for u in sub_users_r:
    if u not in upR:
        recs_svd_R[u] = list(pop_R)
        continue
    sub_s = scR[upR[u]][cand_cols_R]
    if not np.isfinite(sub_s).any():
        recs_svd_R[u] = list(pop_R)
        continue
    top_s = np.argpartition(-sub_s, kth=K - 1)[:K]
    top_s = top_s[np.argsort(-sub_s[top_s])]
    recs_svd_R[u] = [int(cand_shows_R[int(c)]) for c in top_s]

# cold users and their rows in the already-submitted files
hist_users_R = set(histR["user_id"].unique())
cold_R = [u for u in sub_users_r if u not in hist_users_R]
pop_str = " ".join(str(s) for s in pop_R)
print(f"\nusers with no history in weeks 11-30: {len(cold_R)}")
for f in SUB_FILES:
    d = pd.read_csv(f, dtype={"user_id": str}).set_index("user_id")
    rows_c = [str(d.loc[u, "tv_show_id"]) for u in cold_R if u in d.index]
    ok = [r == pop_str for r in rows_c]
    print(f"  {f}: cold rows equal popularity top-5 -> {ok}")

# regenerate a pre-fix file under a NEW name and compare
MF_PATH = "outputs/submission_mf.csv"
if not prov_after_fix[MF_PATH]:
    out_chk = pd.DataFrame({"user_id": sub_users_r,
                            "tv_show_id": [" ".join(str(int(s)) for s in recs_svd_R[u]) for u in sub_users_r]})
    chk_path = Path("outputs/submission_mf_recheck.csv")
    out_chk.to_csv(chk_path, index=False)
    sha_old = hashlib.sha256(Path(MF_PATH).read_bytes()).hexdigest()
    sha_new = hashlib.sha256(chk_path.read_bytes()).hexdigest()
    old = pd.read_csv(MF_PATH, dtype={"user_id": str})
    new = pd.read_csv(chk_path, dtype={"user_id": str})
    diff_rows = int((old["tv_show_id"].astype(str) != new["tv_show_id"].astype(str)).sum())
    print()
    print(f"[regen] {chk_path} | sha256 identical: {sha_old == sha_new} | differing rows: {diff_rows}")
    print(f"  sha256[:8] old {sha_old[:8]} vs new {sha_new[:8]}")
else:
    print("\n[regen] not needed: file was generated after the fix")


STEP 0 — provenance of output files vs the pop_fb fix

outputs/submission_mf.csv
  sha256[:8] c9f84e23
  last commit: 62b82596b97e8244073265e503b13b5f3326a8ae|62b8259|2026-10-31|Section 11: safe MF submission (SVD k=32, full history)
  generated after b8afb93: False

outputs/submission_cbf.csv
  sha256[:8] 9c0c56fb
  last commit: b8afb93db0957665f9e2910793135dc4ae279f1c|b8afb93|2026-10-31|Section 9 fix: popularity fallback for history-less users, random Fold-B baseline, remove dead line
  generated after b8afb93: True

outputs/submission_hybrid.csv
  sha256[:8] 2550c033
  last commit: b8afb93db0957665f9e2910793135dc4ae279f1c|b8afb93|2026-10-31|Section 9 fix: popularity fallback for history-less users, random Fold-B baseline, remove dead line
  generated after b8afb93: True

final setup: history 3,742,246 interactions | pool 6,642 shows | schedule rows 144,784
SVD fill model: k=32 | pool shows with factors 3,114 / 6,642

users with no history in weeks 11-30: 2
  outputs/submission_mf.cs

In [81]:
# --- Section 9b: repeat-ranker experiments (one variable at a time, tuned on Fold A) ---
print("=" * 72)
print("SECTION 9b — repeat ranker: improving the own-history score")
print("=" * 72)

def sched_counts(weeks):
    w = schedule_nd["start_dt"].dt.isocalendar().week.astype(int)
    s = schedule_nd[w.isin(list(weeks)) & (schedule_nd["tv_show_id"] != 0)]
    return s.groupby("tv_show_id").size().to_dict()

sched_cnt_A = sched_counts(range(21, 31))
sched_cnt_B = sched_counts(range(19, 29))
sched_cnt_F = sched_counts(range(31, 43))
print(f"schedule airings per show: Fold A {len(sched_cnt_A):,} | Fold B {len(sched_cnt_B):,} | final {len(sched_cnt_F):,}")

def agg_scores(hist, last_week, value, hl):
    d = hist[["user_id", "tv_show_id", "iso_week", "overlap_sec", "overlap_frac", "view_start_dt"]].copy()
    d["iso_week"] = d["iso_week"].astype("int64")
    if value == "ndays":
        d["day"] = d["view_start_dt"].dt.normalize()
        dd = d.groupby(["user_id", "tv_show_id", "day"], as_index=False)["iso_week"].min()
        dd["w"] = 1.0 if hl is None else 0.5 ** ((last_week - dd["iso_week"]) / float(hl))
        return dd.groupby(["user_id", "tv_show_id"], as_index=False)["w"].sum().rename(columns={"w": "score"})
    d["w"] = 1.0 if hl is None else 0.5 ** ((last_week - d["iso_week"]) / float(hl))
    if value in ("sec", "logsec"):
        d["contrib"] = d["w"] * d["overlap_sec"].astype("float64")
        g = d.groupby(["user_id", "tv_show_id"], as_index=False)["contrib"].sum().rename(columns={"contrib": "score"})
        if value == "logsec":
            g["score"] = np.log1p(g["score"])
        return g
    if value == "nair":
        return d.groupby(["user_id", "tv_show_id"], as_index=False)["w"].sum().rename(columns={"w": "score"})
    if value == "nfrac":
        d["contrib"] = d["w"] * (d["overlap_frac"] >= 0.5).astype("float64")
        return d.groupby(["user_id", "tv_show_id"], as_index=False)["contrib"].sum().rename(columns={"contrib": "score"})
    raise ValueError(value)

def score_frame(hist, pool, last_week, cnt, value, hl, mult):
    g = agg_scores(hist, last_week, value, hl)
    pool_set = set(int(s) for s in pool)
    g = g[g["tv_show_id"].isin(pool_set)].copy()
    if mult:
        air = g["tv_show_id"].map(lambda s: float(cnt.get(int(s), 0)))
        g["score"] = g["score"] * np.log1p(air)
    return g

def rank_repeat(hist, pool, last_week, cnt, value, hl, mult, users, fill_kind, fill_pop, fill_svd=None):
    g = score_frame(hist, pool, last_week, cnt, value, hl, mult)
    pool_set = set(int(s) for s in pool)
    g = g.sort_values(["user_id", "score", "tv_show_id"], ascending=[True, False, True])
    top = g.groupby("user_id", sort=False).head(K)
    def fill_list(u):
        if fill_kind == "svd" and fill_svd is not None:
            return [int(s) for s in fill_svd.get(u, fill_pop)]
        return [int(s) for s in fill_pop]
    recs = {}
    for u, sub in top.groupby("user_id", sort=False):
        r = [int(s) for s in sub["tv_show_id"]]
        if len(r) < K:
            for s in fill_list(u):
                if len(r) >= K:
                    break
                if s not in r and s in pool_set:
                    r.append(s)
        recs[u] = r[:K]
    n_cold = 0
    for u in users:
        if u not in recs:
            n_cold += 1
            recs[u] = [s for s in fill_list(u) if s in pool_set][:K]
    return recs, n_cold

poolA_set = set(int(s) for s in candA)
poolB_set = set(int(s) for s in candB)
nuA = labA[labA["tv_show_id"].isin(poolA_set)].groupby("user_id")["tv_show_id"].nunique().to_dict()
nuB = labB[labB["tv_show_id"].isin(poolB_set)].groupby("user_id")["tv_show_id"].nunique().to_dict()
n_shortsA = sum(1 for u in EVAL_USERS if nuA.get(u, 0) < K)
n_shortsB = sum(1 for u in EVAL_USERS if nuB.get(u, 0) < K)
print(f"users needing a fill (<{K} own programs in pool): Fold A {n_shortsA} | Fold B {n_shortsB}")

BASE = dict(value="sec", hl=None, mult=False, fill="pop")
RUNS = [
    ("R1  (a) total_seconds (baseline)", dict()),
    ("R2  (a) log1p(seconds)", dict(value="logsec")),
    ("R3  (a) number of airings", dict(value="nair")),
    ("R4  (a) number of distinct days", dict(value="ndays")),
    ("R5  (a) airings with frac>=0.5", dict(value="nfrac")),
    ("R6  (b) half-life 8 weeks", dict(hl=8)),
    ("R7  (b) half-life 4 weeks", dict(hl=4)),
    ("R8  (b) half-life 2 weeks", dict(hl=2)),
    ("R9  (b) half-life 1 week", dict(hl=1)),
    ("R10 (c) x log1p(target-window airings)", dict(mult=True)),
    ("R11 (d) SVD fill instead of popularity", dict(fill="svd")),
]

def evaluate(cfg):
    kw = dict(BASE)
    kw.update(cfg)
    res = {}
    for tag, hist, pool, lw, cnt, gt1, gt2, popf, svdf in [
        ("A", labA, candA, 20, sched_cnt_A, gt1_eval, gt2_eval, pop_fallback_A, best_recs_svd),
        ("B", labB, candB, 18, sched_cnt_B, gt1_B, gt2_B, pop_fallback_B, recsB_svd),
    ]:
        recs, n_cold = rank_repeat(hist, pool, lw, cnt, kw["value"], kw["hl"], kw["mult"],
                                   EVAL_USERS, kw["fill"], popf, svdf)
        m1, _ = map_at5(recs, gt1)
        m2, _ = map_at5(recs, gt2)
        res[tag] = (m1, m2, recs, n_cold)
    return kw, res

print()
print(f"{'run':38s} | {'A GT-1':>8s} | {'A GT-2':>8s} | {'B GT-1':>8s} | {'B GT-2':>8s}")
RES = []
for label, cfg in RUNS:
    kw, res = evaluate(cfg)
    RES.append(dict(label=label, cfg=kw, A=res["A"][:2], B=res["B"][:2], recsA=res["A"][2]))
    print(f"{label:38s} | {res['A'][0]:8.6f} | {res['A'][1]:8.6f} | {res['B'][0]:8.6f} | {res['B'][1]:8.6f}")

def key_of(r):
    return (r["A"][0], r["A"][1])

best_value = max(RES[0:5], key=key_of)["cfg"]["value"]
best_hl = max([RES[0]] + RES[5:9], key=key_of)["cfg"]["hl"]
best_mult = max([RES[0], RES[9]], key=key_of)["cfg"]["mult"]
best_fill = max([RES[0], RES[10]], key=key_of)["cfg"]["fill"]
combo = dict(value=best_value, hl=best_hl, mult=best_mult, fill=best_fill)
print(f"\ncombined choice from the single-variable winners: {combo}")
kw_c, res_c = evaluate(combo)
RES.append(dict(label="R12 combined best of (a)-(d)", cfg=kw_c, A=res_c["A"][:2], B=res_c["B"][:2], recsA=res_c["A"][2]))
print(f"{'R12 combined best of (a)-(d)':38s} | {res_c['A'][0]:8.6f} | {res_c['A'][1]:8.6f} | {res_c['B'][0]:8.6f} | {res_c['B'][1]:8.6f}")

print("\n[delta vs R1 baseline, Fold A]")
for r in RES[1:]:
    print(f"  {r['label']:38s} GT-1 {r['A'][0] - RES[0]['A'][0]:+.6f} | GT-2 {r['A'][1] - RES[0]['A'][1]:+.6f}")

best_run = max(RES, key=key_of)
print(f"\nBEST (selected on Fold A GT-1): {best_run['label']}")
print(f"  config: {best_run['cfg']}")
print(f"  Fold A: GT-1 {best_run['A'][0]:.6f} | GT-2 {best_run['A'][1]:.6f}")
print(f"  Fold B: GT-1 {best_run['B'][0]:.6f} | GT-2 {best_run['B'][1]:.6f}")

# hand-check for one user, two programs, best configuration
probe = next(u for u in EVAL_USERS if nuA.get(u, 0) >= 3 and gt1_eval.get(u))
best_g = score_frame(labA, candA, 20, sched_cnt_A, best_run["cfg"]["value"], best_run["cfg"]["hl"], best_run["cfg"]["mult"])
rows_p = labA[labA["user_id"] == probe].copy()
rows_p["day"] = rows_p["view_start_dt"].dt.normalize()
two = [int(s) for s in best_g[best_g["user_id"] == probe].sort_values(["score", "tv_show_id"], ascending=[False, True]).head(2)["tv_show_id"]]
print(f"\n[hand-check] user {probe} | {len(rows_p)} Fold-A airings | programs {two}")
for s in two:
    rr = rows_p[rows_p["tv_show_id"] == s].sort_values("iso_week")
    print(f"\nprogram {s}: {len(rr)} airings in the history table")
    hand = 0.0
    parts = []
    if best_run["cfg"]["value"] == "ndays":
        day_ws = {}
        for _, r in rr.iterrows():
            w = 1.0 if best_run["cfg"]["hl"] is None else 0.5 ** ((20 - int(r["iso_week"])) / float(best_run["cfg"]["hl"]))
            day_ws[r["day"]] = w
            parts.append(f"day {r['day']} wk{int(r['iso_week'])} w={w:.6f}")
        hand = float(sum(day_ws.values()))
    else:
        for _, r in rr.iterrows():
            w = 1.0 if best_run["cfg"]["hl"] is None else 0.5 ** ((20 - int(r["iso_week"])) / float(best_run["cfg"]["hl"]))
            sec = float(r["overlap_sec"])
            frac = float(r["overlap_frac"])
            if best_run["cfg"]["value"] == "nair":
                c = w
            elif best_run["cfg"]["value"] == "nfrac":
                c = w * (1.0 if frac >= 0.5 else 0.0)
            else:
                c = w * sec
            hand += c
            parts.append(f"wk{int(r['iso_week'])} sec={int(sec)} frac={frac:.2f} w={w:.6f} -> {c:.6f}")
        if best_run["cfg"]["value"] == "logsec":
            hand = float(np.log1p(hand))
    if best_run["cfg"]["mult"]:
        k_air = sched_cnt_A.get(int(s), 0)
        print(f"  target-window airings {k_air} -> x log1p = {float(np.log1p(k_air)):.6f}")
        hand = hand * float(np.log1p(k_air))
    code_val = float(best_g[(best_g["user_id"] == probe) & (best_g["tv_show_id"] == s)]["score"].iloc[0])
    print("  " + " + ".join(parts))
    print(f"  hand = {hand:.10f} | code = {code_val:.10f} | abs diff = {abs(hand - code_val):.2e}")


SECTION 9b — repeat ranker: improving the own-history score
schedule airings per show: Fold A 6,376 | Fold B 6,477 | final 6,642
users needing a fill (<5 own programs in pool): Fold A 14 | Fold B 14

run                                    |   A GT-1 |   A GT-2 |   B GT-1 |   B GT-2
R1  (a) total_seconds (baseline)       | 0.301477 | 0.510443 | 0.303467 | 0.504754
R2  (a) log1p(seconds)                 | 0.301477 | 0.510443 | 0.303467 | 0.504754
R3  (a) number of airings              | 0.288093 | 0.575556 | 0.295407 | 0.573599
R4  (a) number of distinct days        | 0.289227 | 0.608476 | 0.296517 | 0.608478
R5  (a) airings with frac>=0.5         | 0.248753 | 0.616703 | 0.254101 | 0.620303
R6  (b) half-life 8 weeks              | 0.301204 | 0.506952 | 0.305219 | 0.506957
R7  (b) half-life 4 weeks              | 0.296736 | 0.491743 | 0.303277 | 0.500034
R8  (b) half-life 2 weeks              | 0.273598 | 0.449025 | 0.288007 | 0.473060
R9  (b) half-life 1 week               | 0.224403 | 0

### Repeat-ranker submission — best configuration, weeks 11-30 → weeks 31-42

Same users, order and format as `data/submission.csv`. No already-submitted file is touched; the result is a new file `outputs/submission_repeat.csv`.


In [82]:
# --- Section 9b: submission from the best repeat ranker ---
print("=" * 72)
print("SECTION 9b — submission: best repeat ranker (weeks 11-30 -> 31-42)")
print("=" * 72)

best_cfg = best_run["cfg"]
print(f"best config: value={best_cfg['value']} | half-life={best_cfg['hl']} | target-airing mult={best_cfg['mult']} | fill={best_cfg['fill']}")

recs_rep, n_cold_rep = rank_repeat(histR, candR, 30, sched_cnt_F, best_cfg["value"], best_cfg["hl"],
                                   best_cfg["mult"], sub_users_r, best_cfg["fill"], pop_R, recs_svd_R)
assert all(u in recs_rep for u in sub_users_r), "missing users"
recs_rep = {u: recs_rep[u] for u in sub_users_r}
assert len(recs_rep) == len(sub_users_r), (len(recs_rep), len(sub_users_r))
for u in sub_users_r:
    r = recs_rep[u]
    assert len(r) == K and len(set(r)) == K and 0 not in r, (u, r)
    assert set(r) <= candR_set, (u, r)
own_shows_R = histR.groupby("user_id")["tv_show_id"].apply(lambda s: set(int(x) for x in s)).to_dict()
n_short_R = sum(1 for u in sub_users_r if len(own_shows_R.get(u, set()) & candR_set) < K)
print(f"rows with cold-user fill: {n_cold_rep} | users needing <{K} fills: {n_short_R}")

out_rep = pd.DataFrame({"user_id": sub_users_r,
                        "tv_show_id": [" ".join(str(int(s)) for s in recs_rep[u]) for u in sub_users_r]})
assert len(out_rep) == len(SAMPLE_R)
assert out_rep["user_id"].tolist() == SAMPLE_R["user_id"].tolist()
REP_PATH = Path("outputs/submission_repeat.csv")
out_rep.to_csv(REP_PATH, index=False)
sha_rep = hashlib.sha256(REP_PATH.read_bytes()).hexdigest()
print(f"\n{REP_PATH}: rows {len(out_rep):,} | sha256[:8] {sha_rep[:8]}")
print(out_rep.head(3).to_string(index=False))


SECTION 9b — submission: best repeat ranker (weeks 11-30 -> 31-42)
best config: value=sec | half-life=None | target-airing mult=True | fill=pop
rows with cold-user fill: 2 | users needing <5 fills: 3

outputs/submission_repeat.csv: rows 1,260 | sha256[:8] cd9f833d
            user_id                                  tv_show_id
8377619604347126107         2400480 240081 20088 200337 2400508
8381667675275833309      2400480 12001682 240081 200432 6700482
8387147770138767246 12001682 2400467 12002856 12001732 12002355


## 9c. Exposure form and personalised exposure

R10 multiplies the own-history score by `log1p(airings in the target window)`. Two questions remain:

1. Is the log the right shape? Proposed forms: `none`, `log1p`, `sqrt`, `n^0.25`, `n^0.5`, `n^1`. `sqrt` and `n^0.5` are the same function, so one run covers both.
2. Is a global exposure good enough? Personalised exposure replaces the global count with how much the user watches a channel, and with a channel x hour-of-week profile, estimated from the history window only (add-one smoothing, normalised per user by its maximum).

Every run changes one thing relative to R10. Selection is on Fold A GT-1; Fold B is a check, not a tuning set. The upper-bound row is a diagnostic, not a method.


In [83]:
# --- Section 9c: exposure form + personalised exposure (one change at a time vs R10) ---
import time
import scipy.sparse as sp

print("=" * 72)
print("SECTION 9c — exposure form and personalised exposure")
print("=" * 72)

t0_9c = time.time()
CHANNELS = np.sort(schedule_nd["channel_id"].unique())
n_ch = len(CHANNELS)
ch_pos = {int(c): i for i, c in enumerate(CHANNELS)}
N_HW = 168
wk_all = schedule_nd["start_dt"].dt.isocalendar().week.astype(int)

def share_vectors(hist, users):
    h = hist[["user_id", "channel_id", "overlap_sec", "view_start_dt"]].copy()
    h["hw"] = (h["view_start_dt"].dt.weekday * 24 + h["view_start_dt"].dt.hour).astype(int)
    cm = h.groupby(["user_id", "channel_id"])["overlap_sec"].sum().unstack(fill_value=0.0)
    cm = cm.reindex(index=users, columns=CHANNELS, fill_value=0.0).to_numpy(dtype="float64")
    hm = h.groupby(["user_id", "hw"])["overlap_sec"].sum().unstack(fill_value=0.0)
    hm = hm.reindex(index=users, columns=range(N_HW), fill_value=0.0).to_numpy(dtype="float64")
    S = (cm + 1.0) / (cm.sum(axis=1, keepdims=True) + n_ch)
    Hs = (hm + 1.0) / (hm.sum(axis=1, keepdims=True) + float(N_HW))
    return S.astype("float32"), Hs.astype("float32")

def build_exposure(hist, pool, weeks, users):
    S, Hs = share_vectors(hist, users)
    s = schedule_nd[wk_all.isin(list(weeks)) & (schedule_nd["tv_show_id"] != 0)].copy()
    pool_arr = np.array(sorted(int(x) for x in pool), dtype=np.int64)
    p_pos = {int(v): i for i, v in enumerate(pool_arr)}
    s = s[s["tv_show_id"].isin(set(p_pos))]
    s["p"] = s["tv_show_id"].map(p_pos).astype(int)
    s["c"] = s["channel_id"].map(ch_pos).astype(int)
    s["hw"] = (s["start_dt"].dt.weekday * 24 + s["start_dt"].dt.hour).astype(int)
    n_pool = len(pool_arr)
    N1 = s.groupby(["p", "c"]).size().unstack(fill_value=0)
    N1 = N1.reindex(index=range(n_pool), columns=range(n_ch), fill_value=0).to_numpy(dtype="float32")
    P1 = (N1 @ S.T).T
    tr = s.groupby(["p", "c", "hw"]).size()
    ri = tr.index.get_level_values(0).to_numpy()
    ci = tr.index.get_level_values(1).to_numpy() * N_HW + tr.index.get_level_values(2).to_numpy()
    T = sp.csr_matrix((tr.to_numpy(dtype="float32"), (ri, ci)), shape=(n_pool, n_ch * N_HW))
    P2 = np.zeros((len(users), n_pool), dtype="float32")
    for a in range(0, len(users), 64):
        Wc = (S[a:a + 64][:, :, None] * Hs[a:a + 64][:, None, :]).reshape(-1, n_ch * N_HW)
        P2[a:a + 64] = (T @ Wc.T).T
    u_pos9 = {u: i for i, u in enumerate(users)}
    return {"ch": (P1, u_pos9, p_pos, P1.max(axis=1)),
            "chhw": (P2, u_pos9, p_pos, P2.max(axis=1))}

print("building personalised-exposure matrices (history-only shares x target-window airings)...")
PE9 = {"A": build_exposure(labA, candA, range(21, 31), EVAL_USERS),
       "B": build_exposure(labB, candB, range(19, 29), EVAL_USERS)}
for tag in ("A", "B"):
    for kind in ("ch", "chhw"):
        P = PE9[tag][kind][0]
        print(f"  Fold {tag} {kind:4s}: shape {P.shape} | non-zero share {float((P > 0).mean()):.3f} | median per-user max {float(np.median(PE9[tag][kind][3])):.4f}")

EXP_F = {"p25": lambda a: np.power(a, 0.25), "p5": lambda a: np.sqrt(a), "lin": lambda a: a}

def score9(hist, pool, lw, cnt, value, fkind, exposure, gamma, pe):
    g = score_frame(hist, pool, lw, cnt, value, None, False)
    if fkind is not None:
        air = g["tv_show_id"].map(lambda x: float(cnt.get(int(x), 0))).to_numpy(dtype="float64")
        mult = np.log1p(air) if fkind == "log1p" else EXP_F[fkind](air)
        g["score"] = g["score"].to_numpy() * mult
    if exposure is not None:
        P, upos, ppos, pmax = pe[exposure]
        keep9 = g["user_id"].isin(upos.keys()) & g["tv_show_id"].isin(ppos.keys())
        g = g[keep9].copy()
        ru = g["user_id"].map(upos).to_numpy(dtype="int64")
        raw = P[ru, g["tv_show_id"].map(ppos).to_numpy(dtype="int64")]
        den = np.where(pmax[ru] > 0, pmax[ru], 1.0)
        g["score"] = g["score"].to_numpy() * np.power(0.1 + raw / den, 1.0 if gamma is None else gamma)
    return g

def rank9(hist, pool, lw, cnt, users, fill_pop, value, fkind, exposure, gamma, pe):
    g = score9(hist, pool, lw, cnt, value, fkind, exposure, gamma, pe)
    pool_set = set(int(x) for x in pool)
    g = g.sort_values(["user_id", "score", "tv_show_id"], ascending=[True, False, True])
    top = g.groupby("user_id", sort=False).head(K)
    recs = {}
    for u, sub in top.groupby("user_id", sort=False):
        r = [int(x) for x in sub["tv_show_id"]]
        for v in fill_pop:
            if len(r) >= K:
                break
            if int(v) not in r and int(v) in pool_set:
                r.append(int(v))
        recs[u] = r[:K]
    n_cold = 0
    for u in users:
        if u not in recs:
            n_cold += 1
            recs[u] = [int(v) for v in fill_pop if int(v) in pool_set][:K]
    return recs, n_cold

RES9 = []

def eval9(label, cfg):
    recsA, coldA = rank9(labA, candA, 20, sched_cnt_A, EVAL_USERS, pop_fallback_A,
                         cfg.get("value", "sec"), cfg.get("fkind"), cfg.get("exposure"), cfg.get("gamma"), PE9["A"])
    recsB, coldB = rank9(labB, candB, 18, sched_cnt_B, EVAL_USERS, pop_fallback_B,
                         cfg.get("value", "sec"), cfg.get("fkind"), cfg.get("exposure"), cfg.get("gamma"), PE9["B"])
    m1A, _ = map_at5(recsA, gt1_eval)
    m2A, _ = map_at5(recsA, gt2_eval)
    m1B, _ = map_at5(recsB, gt1_B)
    m2B, _ = map_at5(recsB, gt2_B)
    res = dict(label=label, cfg=cfg, A=(m1A, m2A), B=(m1B, m2B), recsA=recsA, recsB=recsB, cold=(coldA, coldB))
    RES9.append(res)
    print(f"{label:46s} | {m1A:8.6f} | {m2A:8.6f} | {m1B:8.6f} | {m2B:8.6f}")
    return res

print()
print(f"{'run':46s} | {'A GT-1':>8s} | {'A GT-2':>8s} | {'B GT-1':>8s} | {'B GT-2':>8s}")
eval9("E0  ref: sec x log1p(airings)", dict(fkind="log1p"))
eval9("E1  f(n) = none (raw seconds)", dict(fkind=None))
eval9("E2  f(n) = n^0.25", dict(fkind="p25"))
eval9("E3  f(n) = sqrt = n^0.5 (one run)", dict(fkind="p5"))
eval9("E4  f(n) = n^1 (linear)", dict(fkind="lin"))
eval9("E5  personalised: channel, gamma=0.5", dict(fkind="log1p", exposure="ch", gamma=0.5))
eval9("E6  personalised: channel x hour, gamma=0.5", dict(fkind="log1p", exposure="chhw", gamma=0.5))

best_exp_kind = max([RES9[5], RES9[6]], key=lambda r: r["A"][0])["cfg"]["exposure"]
print(f"best personalised variant so far: {best_exp_kind}")
eval9(f"E7  best personalised ({best_exp_kind}) gamma=0.25", dict(fkind="log1p", exposure=best_exp_kind, gamma=0.25))
eval9(f"E8  best personalised ({best_exp_kind}) gamma=1.0", dict(fkind="log1p", exposure=best_exp_kind, gamma=1.0))
eval9("E9  ndays x log1p(airings)", dict(value="ndays", fkind="log1p"))
eval9("E10 frac>=0.5 count x log1p(airings)", dict(value="nfrac", fkind="log1p"))

best_f = max(RES9[0:9], key=lambda r: r["A"][0])
best_e = max([RES9[0], RES9[9], RES9[10]], key=lambda r: r["A"][0])
combo_cfg = dict(best_f["cfg"])
combo_cfg["value"] = best_e["cfg"].get("value", "sec")
eval9("E11 combo: best exposure form x best (e)", combo_cfg)

# --- upper bound: perfect ranking restricted to the user's own history ---
def upper_bound9(hist, pool, target, gt1, users):
    own = hist.groupby("user_id")["tv_show_id"].apply(lambda s: set(int(x) for x in s)).to_dict()
    pool_set = set(int(x) for x in pool)
    t = target[target["tv_show_id"].isin(pool_set) & target["user_id"].isin(set(users))]
    g = t.groupby(["user_id", "tv_show_id"], as_index=False)["overlap_sec"].sum()
    keep = np.array([int(s) in own.get(u, set()) for u, s in zip(g["user_id"], g["tv_show_id"])])
    g = g[keep]
    g = g.sort_values(["user_id", "overlap_sec", "tv_show_id"], ascending=[True, False, True])
    top = g.groupby("user_id", sort=False).head(K)
    recs = {u: [int(x) for x in sub["tv_show_id"]] for u, sub in top.groupby("user_id", sort=False)}
    m1, _ = map_at5(recs, gt1)
    n_gt = sum(1 for u in users if gt1.get(u))
    n_in = sum(1 for u in users if gt1.get(u) and set(int(x) for x in gt1[u]) <= own.get(u, set()))
    return m1, n_gt, n_in, own

ub_A, n_gt_A, n_in_A, ownA9 = upper_bound9(labA, candA, targetA, gt1_eval, EVAL_USERS)
ub_B, n_gt_B, n_in_B, ownB9 = upper_bound9(labB, candB, targetB, gt1_B, EVAL_USERS)
print()
print(f"{'UPPER (diagnostic, not a method)':46s} | {ub_A:8.6f} | {'-':>8s} | {ub_B:8.6f} | {'-':>8s}")
print(f"  users whose GT-1 lies entirely inside their own history: Fold A {n_in_A:,}/{n_gt_A:,} ({n_in_A / n_gt_A:.4f}) | Fold B {n_in_B:,}/{n_gt_B:,} ({n_in_B / n_gt_B:.4f})")

ref9 = RES9[0]
variants9 = [r for r in RES9 if not r["label"].startswith("E0")]
best9 = max(variants9, key=lambda r: (r["A"][0], r["B"][0]))
exp9c_adopted = bool(best9["A"][0] > ref9["A"][0] and best9["B"][0] >= ref9["B"][0])
exp9c_final_label = best9["label"] if exp9c_adopted else ref9["label"]
exp9c_final_cfg = dict(best9["cfg"] if exp9c_adopted else ref9["cfg"])
exp9c_final_recs_A = (best9 if exp9c_adopted else ref9)["recsA"]
exp9c_final_recs_B = (best9 if exp9c_adopted else ref9)["recsB"]
print(f"\nbest variant on Fold A GT-1: {best9['label']}  A {best9['A'][0]:.6f} / B {best9['B'][0]:.6f}")
print(f"R10 reference:              A {ref9['A'][0]:.6f} / B {ref9['B'][0]:.6f}")
print(f"decision: {'ADOPT ' + best9['label'] if exp9c_adopted else 'KEEP R10 (no variant improves Fold A GT-1 without dropping Fold B GT-1)'}")
print(f"final own-history ranker: {exp9c_final_label} | config {exp9c_final_cfg}")

# --- hand-check of the exposure factor for the final configuration ---
hc_kind = exp9c_final_cfg.get("exposure")
if hc_kind is None:
    print("\n[hand-check] the final configuration keeps a per-program multiplier and no personalised exposure, so there is no exposure factor to recompute by hand.")
else:
    u_hc = next(u for u in EVAL_USERS if nuA.get(u, 0) >= 4 and gt1_eval.get(u))
    g_u = score9(labA, candA, 20, sched_cnt_A, "sec", exp9c_final_cfg.get("fkind"), None, None, PE9["A"])
    g_u = g_u[g_u["user_id"] == u_hc].sort_values(["score", "tv_show_id"], ascending=[False, True])
    sched_h = schedule_nd[wk_all.isin(list(range(21, 31))) & (schedule_nd["tv_show_id"] != 0)].copy()
    sched_h["hw"] = (sched_h["start_dt"].dt.weekday * 24 + sched_h["start_dt"].dt.hour).astype(int)
    p_hc = None
    for s in g_u["tv_show_id"].tolist():
        n_air = int((sched_h["tv_show_id"] == int(s)).sum())
        if 2 <= n_air <= 5:
            p_hc = int(s)
            break
    if p_hc is None:
        p_hc = int(g_u["tv_show_id"].iloc[0])
    rows_h = sched_h[sched_h["tv_show_id"] == p_hc]
    hu = labA[labA["user_id"] == u_hc]
    csum = hu.groupby("channel_id")["overlap_sec"].sum()
    hsum = hu.groupby(hu["view_start_dt"].dt.weekday * 24 + hu["view_start_dt"].dt.hour)["overlap_sec"].sum()
    tot_u = float(hu["overlap_sec"].sum())
    hand = 0.0
    terms = []
    for _, r in rows_h.iterrows():
        c = int(r["channel_id"])
        hwv = int(r["hw"])
        sc = (float(csum.get(c, 0.0)) + 1.0) / (tot_u + n_ch)
        sh = (float(hsum.get(hwv, 0.0)) + 1.0) / (tot_u + float(N_HW))
        contrib = sc if hc_kind == "ch" else sc * sh
        hand += contrib
        terms.append(f"ch {c} hw {hwv:3d}: share_c={sc:.8f} share_h={sh:.8f} -> {contrib:.8f}")
    P, upos, ppos, pmax = PE9["A"][hc_kind]
    code_raw = float(P[upos[u_hc], ppos[p_hc]])
    mx = float(pmax[upos[u_hc]])
    gamma_h = 1.0 if exp9c_final_cfg.get("gamma") is None else float(exp9c_final_cfg["gamma"])
    print(f"\n[hand-check] user {u_hc} | program {p_hc} | {len(rows_h)} target-window airings | exposure={hc_kind} gamma={gamma_h}")
    for t in terms:
        print("  " + t)
    print(f"  hand raw = {hand:.10f} | matrix raw = {code_raw:.10f} | diff = {abs(hand - code_raw):.3e}")
    print(f"  user max raw = {mx:.10f} | normalised = {hand / mx:.10f} | factor = (0.1 + n)^gamma = {(0.1 + hand / mx) ** gamma_h:.10f}")

print(f"\n9c runtime: {time.time() - t0_9c:.1f} s")


SECTION 9c — exposure form and personalised exposure
building personalised-exposure matrices (history-only shares x target-window airings)...
  Fold A ch  : shape (1260, 6376) | non-zero share 1.000 | median per-user max 96.7205
  Fold A chhw: shape (1260, 6376) | non-zero share 1.000 | median per-user max 0.6321
  Fold B ch  : shape (1260, 6477) | non-zero share 1.000 | median per-user max 100.6737
  Fold B chhw: shape (1260, 6477) | non-zero share 1.000 | median per-user max 0.6608

run                                            |   A GT-1 |   A GT-2 |   B GT-1 |   B GT-2
E0  ref: sec x log1p(airings)                  | 0.314644 | 0.573441 | 0.320928 | 0.571963
E1  f(n) = none (raw seconds)                  | 0.301477 | 0.510443 | 0.303467 | 0.504754
E2  f(n) = n^0.25                              | 0.310556 | 0.579878 | 0.318114 | 0.579515
E3  f(n) = sqrt = n^0.5 (one run)              | 0.295851 | 0.605945 | 0.303020 | 0.606034
E4  f(n) = n^1 (linear)                        | 0.2511

### 9c submission files

`outputs/submission_probe_days.csv` and `outputs/submission_probe_frac.csv` are always written (probe variants for Kaggle). `outputs/submission_repeat2.csv` is written only when the adopted variant differs from the submitted repeat file; otherwise the submitted file stays `outputs/submission_repeat.csv`.


In [84]:
# --- Section 9c: probe submissions; new main file only if a variant was adopted ---
import hashlib

print("=" * 72)
print("SECTION 9c — submissions: probes + optional new main file")
print("=" * 72)
Path("outputs").mkdir(exist_ok=True)

def write_sub9(cfg, path, label, pe):
    recs, n_cold = rank9(histR, candR, 30, sched_cnt_F, sub_users_r, pop_R,
                         cfg.get("value", "sec"), cfg.get("fkind"), cfg.get("exposure"), cfg.get("gamma"), pe)
    recs = {u: recs[u] for u in sub_users_r}
    assert len(recs) == len(sub_users_r), (len(recs), len(sub_users_r))
    for u in sub_users_r:
        r = recs[u]
        assert len(r) == K and len(set(r)) == K and 0 not in r, (u, r)
        assert set(r) <= candR_set, (u, r)
    out = pd.DataFrame({"user_id": sub_users_r,
                        "tv_show_id": [" ".join(str(int(x)) for x in recs[u]) for u in sub_users_r]})
    assert len(out) == len(SAMPLE_R)
    assert out["user_id"].tolist() == SAMPLE_R["user_id"].tolist()
    out.to_csv(path, index=False)
    sha_ = hashlib.sha256(Path(path).read_bytes()).hexdigest()
    print(f"{label:46s} -> {path}: rows {len(out):,} | sha256[:8] {sha_[:8]} | cold fills {n_cold}")
    return recs

recs_days = write_sub9(dict(value="ndays", fkind="log1p"), "outputs/submission_probe_days.csv", "E9 ndays x log1p", PE9["A"])
recs_frac = write_sub9(dict(value="nfrac", fkind="log1p"), "outputs/submission_probe_frac.csv", "E10 frac>=0.5 x log1p", PE9["A"])

if exp9c_adopted:
    pe_F = None
    if exp9c_final_cfg.get("exposure") is not None:
        print("building personalised-exposure matrices for the final window (weeks 31-42)...")
        pe_F = build_exposure(histR, candR, range(31, 43), sub_users_r)
    _ = write_sub9(exp9c_final_cfg, "outputs/submission_repeat2.csv", exp9c_final_label, pe_F)
else:
    print("adopted variant: none -> outputs/submission_repeat2.csv NOT created (submitted repeat file unchanged)")


SECTION 9c — submissions: probes + optional new main file
E9 ndays x log1p                               -> outputs/submission_probe_days.csv: rows 1,260 | sha256[:8] cf246700 | cold fills 2
E10 frac>=0.5 x log1p                          -> outputs/submission_probe_frac.csv: rows 1,260 | sha256[:8] 273c8069 | cold fills 2
building personalised-exposure matrices for the final window (weeks 31-42)...
E7  best personalised (chhw) gamma=0.25        -> outputs/submission_repeat2.csv: rows 1,260 | sha256[:8] 3267a04f | cold fills 2


## 9d. Recalibrating the validation proxy against the returned Kaggle scores

Five files were scored by Kaggle (private scores, low to high): hybrid 0.24856, repeat 0.29642, repeat2 0.32890, probe_days 0.36849, probe_frac 0.41211. Our GT-1 (top-5 by watched seconds) ranks exactly these five validation analogs in the opposite order, so GT-1 is a bad proxy for this competition.

This section defines candidate ground truths on the target window for the same evaluation users and measures which one reproduces the Kaggle order:

- GT-3(theta): top-5 programs by the NUMBER of airings with `overlap_frac >= theta`; theta in {0.25, 0.5, 0.75, 0.9}.
- GT-4(theta): ALL programs with at least one airing at `overlap_frac >= theta` (this is GT-2 at theta = 0.5); theta in {0.25, 0.5, 0.75, 0.9}.
- GT-5: top-5 by the number of distinct days on which the program was watched with `overlap_frac >= 0.5`.
- GT-1: the existing top-5-by-seconds ground truth (reference).

For every candidate we print MAP@5 of the five analogs and the Spearman and Kendall rank correlation with the Kaggle private scores, on Fold A and on Fold B. Five points are very few: this is a calibration of the proxy, not a proof.


In [85]:
# --- Section 9d: proxy calibration against the returned Kaggle private scores ---
import statistics
import scipy
from scipy.stats import spearmanr, kendalltau
# scipy docs check (1.16.2): spearmanr ranks with average ranks for ties; kendalltau(variant="b")
# is the tie-corrected Kendall tau. Both are ties-aware.

print("=" * 78)
print("SECTION 9d — proxy calibration: which ground truth reproduces the Kaggle order?")
print("=" * 78)
print(f"scipy {scipy.__version__}")

def gt3_top5_by_hits(target_df, theta):
    q = target_df[target_df["overlap_frac"] >= theta]
    g = q.groupby(["user_id", "tv_show_id"]).agg(
        n_hit=("overlap_sec", "size"), secs=("overlap_sec", "sum")).reset_index()
    g = g.sort_values(["user_id", "n_hit", "secs", "tv_show_id"],
                      ascending=[True, False, False, True])
    return {u: set(int(s) for s in df.head(5)["tv_show_id"]) for u, df in g.groupby("user_id")}

def gt4_all_hits(target_df, theta):
    q = target_df[target_df["overlap_frac"] >= theta]
    out = {}
    for u, sset in q.groupby("user_id")["tv_show_id"].apply(set).items():
        out[u] = set(int(s) for s in sset)
    return out

def gt5_top5_days(target_df, frac=0.5):
    q = target_df[target_df["overlap_frac"] >= frac].copy()
    q["day"] = q["view_start_dt"].dt.normalize()
    g = q.groupby(["user_id", "tv_show_id"]).agg(
        n_days=("day", "nunique"), n_hit=("overlap_sec", "size"),
        secs=("overlap_sec", "sum")).reset_index()
    g = g.sort_values(["user_id", "n_days", "n_hit", "secs", "tv_show_id"],
                      ascending=[True, False, False, False, True])
    return {u: set(int(s) for s in df.head(5)["tv_show_id"]) for u, df in g.groupby("user_id")}

def restrict_gt(d):
    return {u: set(d.get(u, set())) for u in EVAL_USERS}

GT_BY_FOLD = {}
for tag, target, gt1_ref, gt2_ref in [("A", targetA, gt1_eval, gt2_eval),
                                      ("B", targetB, gt1_B, gt2_B)]:
    cands = {"GT-1 top-5 seconds": restrict_gt(gt1_ref)}
    for th in (0.25, 0.5, 0.75, 0.9):
        cands[f"GT-3 theta={th:g}"] = restrict_gt(gt3_top5_by_hits(target, th))
    for th in (0.25, 0.5, 0.75, 0.9):
        cands[f"GT-4 theta={th:g}"] = restrict_gt(gt4_all_hits(target, th))
    cands["GT-5 top-5 days"] = restrict_gt(gt5_top5_days(target, 0.5))
    GT_BY_FOLD[tag] = cands
    ident = sum(1 for u in EVAL_USERS if cands["GT-4 theta=0.5"][u] == set(gt2_ref.get(u, set())))
    print(f"[Fold {tag}] candidates {len(cands)} | sanity: GT-4(theta=0.5) equals GT-2 for {ident}/{len(EVAL_USERS)} users")

print("\n[ground-truth coverage, Fold A]")
print(f"{'ground truth':22s} | {'users':>6s} | {'median |R|':>10s} | {'mean |R|':>8s}")
for name, d in GT_BY_FOLD["A"].items():
    sizes = [len(s) for s in d.values() if s]
    print(f"{name:22s} | {len(sizes):6d} | {statistics.median(sizes):10.1f} | {float(np.mean(sizes)):8.2f}")

KAGGLE = {"E1": 0.24856, "E0": 0.29642, "E7": 0.32890, "E9": 0.36849, "E10": 0.41211}
ORDER = ("E1", "E0", "E7", "E9", "E10")
HEAD = ("hybrid", "repeat", "repeat2", "days", "frac")
R9 = {r["label"].split()[0]: r for r in RES9}
for c in ORDER:
    assert c in R9, c
KBY = [KAGGLE[c] for c in ORDER]

MAPT = {}
CORR = {}
for tag in ("A", "B"):
    MAPT[tag] = {}
    for gname, gtd in GT_BY_FOLD[tag].items():
        vals = {}
        for c in ORDER:
            recs = R9[c]["recsA"] if tag == "A" else R9[c]["recsB"]
            m, _ = map_at5(recs, gtd)
            vals[c] = m
        MAPT[tag][gname] = vals

for tag in ("A", "B"):
    print(f"\n[MAP@5, Fold {tag}] columns ordered by the Kaggle private score (low to high)")
    print(f"{'ground truth':22s} | " + " | ".join(f"{HEAD[i] + ' ' + f'{KBY[i]:.4f}':>15s}" for i in range(5)))
    for gname, vals in MAPT[tag].items():
        print(f"{gname:22s} | " + " | ".join(f"{vals[c]:15.6f}" for c in ORDER))

print("\n[rank correlation with the Kaggle private scores, ties-aware, n=5]")
for tag in ("A", "B"):
    CORR[tag] = {}
    for gname, vals in MAPT[tag].items():
        xs = [vals[c] for c in ORDER]
        spr = spearmanr(xs, KBY)
        kd = kendalltau(xs, KBY)
        CORR[tag][gname] = dict(spearman=float(spr.statistic), sp_p=float(spr.pvalue),
                                kendall=float(kd.statistic), kd_p=float(kd.pvalue))
    print(f"\nFold {tag}")
    print(f"{'ground truth':22s} | {'Spearman':>9s} | {'p':>7s} | {'Kendall':>9s} | {'p':>7s}")
    for gname, c in CORR[tag].items():
        print(f"{gname:22s} | {c['spearman']:9.4f} | {c['sp_p']:7.4f} | {c['kendall']:9.4f} | {c['kd_p']:7.4f}")

best_pair = sorted(CORR["A"].items(), key=lambda kv: (-kv[1]["spearman"], -kv[1]["kendall"]))[0]
GT_STAR_LABEL = best_pair[0]
GT_STAR = {"A": GT_BY_FOLD["A"][GT_STAR_LABEL], "B": GT_BY_FOLD["B"][GT_STAR_LABEL]}
print(f"\nGT* (chosen on Fold A by Spearman, tie-break Kendall): {GT_STAR_LABEL}")
print(f"  Fold A: Spearman {CORR['A'][GT_STAR_LABEL]['spearman']:.4f} | Kendall {CORR['A'][GT_STAR_LABEL]['kendall']:.4f}")
print(f"  Fold B: Spearman {CORR['B'][GT_STAR_LABEL]['spearman']:.4f} | Kendall {CORR['B'][GT_STAR_LABEL]['kendall']:.4f}")
print("  five points only: a calibration of the proxy, not a proof.")


SECTION 9d — proxy calibration: which ground truth reproduces the Kaggle order?
scipy 1.16.2
[Fold A] candidates 10 | sanity: GT-4(theta=0.5) equals GT-2 for 1260/1260 users
[Fold B] candidates 10 | sanity: GT-4(theta=0.5) equals GT-2 for 1260/1260 users

[ground-truth coverage, Fold A]
ground truth           |  users | median |R| | mean |R|
GT-1 top-5 seconds     |   1246 |        5.0 |     5.00
GT-3 theta=0.25        |   1246 |        5.0 |     4.98
GT-3 theta=0.5         |   1244 |        5.0 |     4.94
GT-3 theta=0.75        |   1239 |        5.0 |     4.86
GT-3 theta=0.9         |   1226 |        5.0 |     4.73
GT-4 theta=0.25        |   1246 |       58.0 |    74.25
GT-4 theta=0.5         |   1244 |       35.5 |    49.39
GT-4 theta=0.75        |   1239 |       21.0 |    34.51
GT-4 theta=0.9         |   1226 |       15.0 |    27.79
GT-5 top-5 days        |   1244 |        5.0 |     4.94

[MAP@5, Fold A] columns ordered by the Kaggle private score (low to high)
ground truth         

### 9d tuning against GT* only

Tuned on Fold A, confirmed on Fold B without re-tuning, one change at a time (13 runs, all logged with GT* and GT-1):

- (a) value = number of airings with `overlap_frac >= theta`, theta in {0.25, 0.5, 0.75, 0.9}, no exposure factor;
- (b) the best theta combined with global `log1p(airings)` and with `sqrt(airings)`;
- (c) the best of (b) combined with personalised exposure (channel x hour, normalised per user, `(0.1 + pe)^gamma`), gamma in {0.25, 0.5};
- (d) the best of (c) with value = `airings(frac >= theta) x (distinct days)^beta`, beta in {0.25, 0.5} (distinct days counted on the same `frac >= theta` airings);
- (e) `log1p(airings)` at theta 0.25 / 0.75 / 0.9, i.e. the exact configurations of the three new probe files, so the probe choice rests on the same table.

Selection: best Fold A GT*, then the Fold B guard "Fold B GT* not worse than the E0 analog's" (E0 is the currently submitted `repeat` file).


In [86]:
# --- Section 9d: tuning against GT* (Fold A tuned, Fold B confirmed, one change at a time) ---
import time

print("=" * 78)
print(f"SECTION 9d — tuning against GT* = {GT_STAR_LABEL}")
print("=" * 78)
t0_9e = time.time()

def frame9e(hist, pool, cnt, theta, fmult=None, exposure=None, gamma=None, pe=None, beta=None, tau=None):
    d = hist[["user_id", "tv_show_id", "overlap_sec", "overlap_frac", "view_start_dt"]].copy()
    d = d[d["overlap_frac"] >= theta]
    d["day"] = d["view_start_dt"].dt.normalize()
    if tau is None:
        g = d.groupby(["user_id", "tv_show_id"], as_index=False).agg(
            hits=("overlap_sec", "size"), ndays=("day", "nunique"))
    else:
        # recency weight exp(-age_weeks/tau); age measured from the last airing of the history window
        age_wk = (hist["view_start_dt"].max() - d["view_start_dt"]).dt.total_seconds() / 604800.0
        d["w"] = np.exp(-age_wk.to_numpy(dtype="float64") / float(tau))
        g = d.groupby(["user_id", "tv_show_id"], as_index=False).agg(
            hits=("w", "sum"), ndays=("day", "nunique"))
    pool_set = set(int(s) for s in pool)
    g = g[g["tv_show_id"].isin(pool_set)].copy()
    g["score"] = g["hits"].astype("float64")
    if beta is not None:
        g["score"] = g["score"].to_numpy(dtype="float64") * np.power(
            g["ndays"].to_numpy(dtype="float64"), beta)
    if fmult is not None:
        air = g["tv_show_id"].map(lambda s: float(cnt.get(int(s), 0))).to_numpy(dtype="float64")
        g["score"] = g["score"].to_numpy() * (np.log1p(air) if fmult == "log1p" else np.sqrt(air))
    if exposure is not None:
        P, upos, ppos, pmax = pe[exposure]
        keep = g["user_id"].isin(upos.keys()) & g["tv_show_id"].isin(ppos.keys())
        g = g[keep].copy()
        ru = g["user_id"].map(upos).to_numpy(dtype="int64")
        raw = P[ru, g["tv_show_id"].map(ppos).to_numpy(dtype="int64")]
        den = np.where(pmax[ru] > 0, pmax[ru], 1.0)
        g["score"] = g["score"].to_numpy() * np.power(0.1 + raw / den, gamma)
    return g

def rank9e(hist, pool, cnt, users, fill_pop, theta, fmult=None, exposure=None, gamma=None, pe=None, beta=None, tau=None, fill_scores=None):
    g = frame9e(hist, pool, cnt, theta, fmult, exposure, gamma, pe, beta, tau)
    pool_set = set(int(x) for x in pool)
    g = g.sort_values(["user_id", "score", "tv_show_id"], ascending=[True, False, True])
    top = g.groupby("user_id", sort=False).head(K)
    recs = {}
    for u, sub in top.groupby("user_id", sort=False):
        r = [int(x) for x in sub["tv_show_id"]]
        fill = fill_pop if fill_scores is None else fill_scores.get(u, fill_pop)
        for v in fill:
            if len(r) >= K:
                break
            if int(v) not in r and int(v) in pool_set:
                r.append(int(v))
        recs[u] = r[:K]
    n_cold = 0
    for u in users:
        if u not in recs:
            n_cold += 1
            fill = fill_pop if fill_scores is None else fill_scores.get(u, fill_pop)
            recs[u] = [int(v) for v in fill if int(v) in pool_set][:K]
    return recs, n_cold

RUN9E = []

def eval9e(label, theta, fmult=None, exposure=None, gamma=None, beta=None, tau=None, fill_scores=None):
    fsA = None if fill_scores is None else fill_scores.get("A")
    fsB = None if fill_scores is None else fill_scores.get("B")
    recsA, coldA = rank9e(labA, candA, sched_cnt_A, EVAL_USERS, pop_fallback_A,
                          theta, fmult, exposure, gamma, PE9["A"], beta, tau, fsA)
    recsB, coldB = rank9e(labB, candB, sched_cnt_B, EVAL_USERS, pop_fallback_B,
                          theta, fmult, exposure, gamma, PE9["B"], beta, tau, fsB)
    sA, _ = map_at5(recsA, GT_STAR["A"])
    tA, _ = map_at5(recsA, gt1_eval)
    sB, _ = map_at5(recsB, GT_STAR["B"])
    tB, _ = map_at5(recsB, gt1_B)
    res = dict(label=label, theta=theta, fmult=fmult, exposure=exposure, gamma=gamma, beta=beta, tau=tau,
               A=(sA, tA), B=(sB, tB), recsA=recsA, recsB=recsB, cold=(coldA, coldB))
    RUN9E.append(res)
    print(f"{label:42s} | {sA:8.6f} | {tA:8.6f} | {sB:8.6f} | {tB:8.6f}")
    return res

print()
print(f"{'run':42s} | {'A GT*':>8s} | {'A GT-1':>8s} | {'B GT*':>8s} | {'B GT-1':>8s}")
for th in (0.25, 0.5, 0.75, 0.9):
    eval9e(f"(a) hits(frac>={th:g}), no exposure", th)
th_a = max(RUN9E[-4:], key=lambda r: r["A"][0])["theta"]
print(f"  -> best theta in (a): {th_a:g}")
best_a = next(r for r in RUN9E if r["theta"] == th_a and r["fmult"] is None)
r_b1 = eval9e(f"(b) theta={th_a:g} x log1p(airings)", th_a, fmult="log1p")
r_b2 = eval9e(f"(b) theta={th_a:g} x sqrt(airings)", th_a, fmult="p5")
best_b = max([r_b1, r_b2], key=lambda r: r["A"][0])
fm_b = best_b["fmult"]
print(f"  -> best global multiplier in (b): {fm_b}")
r_c1 = eval9e("(c) + chhw exposure gamma=0.25", th_a, fmult=fm_b, exposure="chhw", gamma=0.25)
r_c2 = eval9e("(c) + chhw exposure gamma=0.5", th_a, fmult=fm_b, exposure="chhw", gamma=0.5)
best_c = max([r_c1, r_c2], key=lambda r: r["A"][0])
g_c = best_c["gamma"]
print(f"  -> best gamma in (c): {g_c}")
r_d1 = eval9e("(d) x days^0.25", th_a, fmult=fm_b, exposure="chhw", gamma=g_c, beta=0.25)
r_d2 = eval9e("(d) x days^0.5", th_a, fmult=fm_b, exposure="chhw", gamma=g_c, beta=0.5)
best_d = max([r_d1, r_d2], key=lambda r: r["A"][0])
print(f"  -> best beta in (d): {best_d['beta']}")
r_e = {}
for th in (0.25, 0.75, 0.9):
    r_e[th] = eval9e(f"(e) theta={th:g} x log1p (probe config)", th, fmult="log1p")

e0_A, _ = map_at5(R9["E0"]["recsA"], GT_STAR["A"])
e0_B, _ = map_at5(R9["E0"]["recsB"], GT_STAR["B"])
print(f"\nreference E0 (the submitted 'repeat' analog): A GT* {e0_A:.6f} | B GT* {e0_B:.6f}")
print("selection rule: best Fold A GT*, then the Fold B guard 'B GT* >= E0 B GT*'")
pick = None
for r in sorted(RUN9E, key=lambda x: -x["A"][0]):
    ok = bool(r["B"][0] >= e0_B)
    print(f"  {r['label']:42s} | A GT* {r['A'][0]:8.6f} | B GT* {r['B'][0]:8.6f} | guard {ok}")
    if ok and pick is None:
        pick = r
if pick is None:
    print("  -> no run passes the Fold B guard")
else:
    print(f"  -> selected for submission: {pick['label']}")

hc = pick if pick is not None else best_d
u_hc = next(u for u in EVAL_USERS if nuA.get(u, 0) >= 4 and GT_STAR["A"].get(u))
g_hc = frame9e(labA, candA, sched_cnt_A, hc["theta"], hc["fmult"], hc["exposure"], hc["gamma"], PE9["A"], hc["beta"])
g_u = g_hc[g_hc["user_id"] == u_hc].sort_values(["score", "tv_show_id"], ascending=[False, True])
p_hc = int(g_u["tv_show_id"].iloc[0])
rows_hc = labA[(labA["user_id"] == u_hc) & (labA["tv_show_id"] == p_hc)]
hits_hc = rows_hc[rows_hc["overlap_frac"] >= hc["theta"]]
n_hits = int(len(hits_hc))
days_hc = int(hits_hc["view_start_dt"].dt.normalize().nunique())
air_hc = int(sched_cnt_A.get(p_hc, 0))
val_pre = float(n_hits)
print(f"\n[hand-check] run {hc['label']}")
print(f"  user {u_hc} | program {p_hc} | history airings in weeks 11-20: {len(rows_hc)}")
print(f"  overlap_frac per airing (first 10 of {len(rows_hc)}): {[round(float(x), 3) for x in rows_hc['overlap_frac'][:10]]}")
print(f"  1) airings with frac>={hc['theta']:g}: {n_hits}")
if hc["beta"] is not None:
    val_pre *= float(days_hc ** hc["beta"])
    print(f"  2) distinct days of those airings {days_hc} -> days^{hc['beta']} = {float(days_hc ** hc['beta']):.6f}")
if hc["fmult"] is not None:
    mlt = float(np.log1p(air_hc)) if hc["fmult"] == "log1p" else float(np.sqrt(air_hc))
    val_pre *= mlt
    print(f"  3) target-window airings {air_hc} -> {hc['fmult']} = {mlt:.6f}")
print(f"  value before exposure = {val_pre:.10f}")

if hc["exposure"] is not None:
    sch_hc = schedule_nd[wk_all.isin(range(21, 31)) & (schedule_nd["tv_show_id"] == p_hc)].copy()
    sch_hc["hw"] = (sch_hc["start_dt"].dt.weekday * 24 + sch_hc["start_dt"].dt.hour).astype(int)
    hu = labA[labA["user_id"] == u_hc]
    csum = hu.groupby("channel_id")["overlap_sec"].sum()
    hsum = hu.groupby(hu["view_start_dt"].dt.weekday * 24 + hu["view_start_dt"].dt.hour)["overlap_sec"].sum()
    tot_u = float(hu["overlap_sec"].sum())
    raw_hand = 0.0
    terms = []
    for _, rr in sch_hc.iterrows():
        sc = (float(csum.get(int(rr["channel_id"]), 0.0)) + 1.0) / (tot_u + n_ch)
        sh = (float(hsum.get(int(rr["hw"]), 0.0)) + 1.0) / (tot_u + 168.0)
        raw_hand += sc * sh
        terms.append((int(rr["channel_id"]), int(rr["hw"]), sc, sh, sc * sh))
    P, upos, ppos, pmax = PE9["A"]["chhw"]
    raw_code = float(P[upos[u_hc], ppos[p_hc]])
    mx_code = float(pmax[upos[u_hc]])
    for ch, hw, sc, sh, pr in terms[:3]:
        print(f"     channel {ch} hw {hw:3d}: share_c {sc:.8f} x share_h {sh:.8f} = {pr:.8f}")
    print(f"     ... {max(0, len(terms) - 4)} further airings omitted (same per-airing formula) ...")
    for ch, hw, sc, sh, pr in terms[-1:]:
        print(f"     channel {ch} hw {hw:3d}: share_c {sc:.8f} x share_h {sh:.8f} = {pr:.8f}")
    print(f"  4) exposure over {len(terms)} airings: hand sum (float64) {raw_hand:.10f} | matrix element (float32) {raw_code:.10f}")
    print(f"     user max {mx_code:.10f} | n_hand {raw_hand / mx_code:.8f} | n_code {raw_code / mx_code:.8f}")
    fac_hand = float((0.1 + raw_hand / mx_code) ** hc["gamma"])
    fac_code = float((0.1 + raw_code / mx_code) ** hc["gamma"])
    print(f"     factor (0.1 + n)^gamma: hand {fac_hand:.10f} | from the matrix's own element {fac_code:.10f} (gamma {hc['gamma']})")
    hand_score = val_pre * fac_hand
    matrix_score = val_pre * fac_code
else:
    hand_score = val_pre
    matrix_score = val_pre

code_score = float(g_u[g_u["tv_show_id"] == p_hc]["score"].iloc[0])
print(f"  5) hand score (exposure recomputed in float64) = {hand_score:.10f}")
print(f"     score from the matrix's own exposure element = {matrix_score:.10f} | code = {code_score:.10f} | diff = {abs(matrix_score - code_score):.3e}")
if hc["exposure"] is not None:
    print(f"     float64 vs float32 exposure element: relative difference {abs(raw_hand - raw_code) / raw_code:.2e};")
    print(f"     the hand-vs-code gap {abs(hand_score - code_score):.3e} is float32 rounding in the exposure matrix, not the formula")
print(f"\n9d tuning runtime: {time.time() - t0_9e:.1f} s")


SECTION 9d — tuning against GT* = GT-3 theta=0.25

run                                        |    A GT* |   A GT-1 |    B GT* |   B GT-1
(a) hits(frac>=0.25), no exposure          | 0.396544 | 0.270053 | 0.400358 | 0.279561
(a) hits(frac>=0.5), no exposure           | 0.384936 | 0.249154 | 0.386416 | 0.254537
(a) hits(frac>=0.75), no exposure          | 0.367975 | 0.233796 | 0.359115 | 0.232475
(a) hits(frac>=0.9), no exposure           | 0.348142 | 0.218093 | 0.339502 | 0.217266
  -> best theta in (a): 0.25
(b) theta=0.25 x log1p(airings)            | 0.406816 | 0.265904 | 0.411554 | 0.273411
(b) theta=0.25 x sqrt(airings)             | 0.395043 | 0.246168 | 0.399489 | 0.255733
  -> best global multiplier in (b): log1p
(c) + chhw exposure gamma=0.25             | 0.416723 | 0.269831 | 0.420888 | 0.276811
(c) + chhw exposure gamma=0.5              | 0.420276 | 0.271287 | 0.424827 | 0.279168
  -> best gamma in (c): 0.5
(d) x days^0.25                            | 0.420741 | 0.273561 | 

### 9d probe submissions

Four new files, all new names (nothing in `outputs/` is overwritten): `submission_probe_frac25.csv`, `submission_probe_frac75.csv`, `submission_probe_frac90.csv` (value = airings with `frac >= theta` times `log1p(airings)`, same as the submitted `probe_frac` at theta 0.5) and `submission_best_gtstar.csv` (the Part 2 pick, with the exposure factor it uses, written only when a run passes the Fold B guard). Trained on weeks 11-30 with the weeks 31-42 schedule, same asserts and user order as every other file.


In [87]:
# --- Section 9d: probe submissions (fraction thresholds + the GT* pick) ---
import hashlib

print("=" * 78)
print("SECTION 9d — probe submissions")
print("=" * 78)
Path("outputs").mkdir(exist_ok=True)
pe_F9 = [None]

def pe_final9():
    if pe_F9[0] is None:
        print("building personalised-exposure matrices for the final window (weeks 31-42)...")
        pe_F9[0] = build_exposure(histR, candR, range(31, 43), sub_users_r)
    return pe_F9[0]

def write_sub9e(theta, path, label, fmult=None, exposure=None, gamma=None, beta=None, tau=None, fill_scores=None):
    recs, n_cold = rank9e(histR, candR, sched_cnt_F, sub_users_r, pop_R, theta, fmult,
                          exposure, gamma, pe_final9() if exposure is not None else None, beta, tau, fill_scores)
    recs = {u: recs[u] for u in sub_users_r}
    assert len(recs) == len(sub_users_r), (len(recs), len(sub_users_r))
    for u in sub_users_r:
        r = recs[u]
        assert len(r) == K and len(set(r)) == K and 0 not in r, (u, r)
        assert set(r) <= candR_set, (u, r)
    out = pd.DataFrame({"user_id": sub_users_r,
                        "tv_show_id": [" ".join(str(int(x)) for x in recs[u]) for u in sub_users_r]})
    assert len(out) == len(SAMPLE_R)
    assert out["user_id"].tolist() == SAMPLE_R["user_id"].tolist()
    out.to_csv(path, index=False)
    sha_ = hashlib.sha256(Path(path).read_bytes()).hexdigest()
    print(f"{label:42s} -> {path}: rows {len(out):,} | sha256[:8] {sha_[:8]} | cold fills {n_cold}")
    return recs

write_sub9e(0.25, "outputs/submission_probe_frac25.csv", "frac>=0.25 x log1p")
write_sub9e(0.75, "outputs/submission_probe_frac75.csv", "frac>=0.75 x log1p")
write_sub9e(0.90, "outputs/submission_probe_frac90.csv", "frac>=0.90 x log1p")
if pick is not None:
    write_sub9e(pick["theta"], "outputs/submission_best_gtstar.csv", f"GT* pick ({pick['label']})",
                fmult=pick["fmult"], exposure=pick["exposure"], gamma=pick["gamma"], beta=pick["beta"])
else:
    print("skip outputs/submission_best_gtstar.csv: no Part 2 run passed the Fold B guard")

e10_A, _ = map_at5(R9["E10"]["recsA"], GT_STAR["A"])
e10_B, _ = map_at5(R9["E10"]["recsB"], GT_STAR["B"])
opts = [("E10 already submitted (theta=0.5 x log1p)", e10_A, e10_B)]
for th, run in r_e.items():
    opts.append((f"outputs/submission_probe_frac{int(th * 100)}.csv", run["A"][0], run["B"][0]))
if pick is not None:
    opts.append(("outputs/submission_best_gtstar.csv", pick["A"][0], pick["B"][0]))
print(f"\n{'file':46s} | {'A GT*':>8s} | {'B GT*':>8s} | {'B guard vs E0':>13s}")
for name, a, b in opts:
    print(f"{name:46s} | {a:8.6f} | {b:8.6f} | {str(bool(b >= e0_B)):>13s}")
winner_pool = [o for o in opts if o[2] >= e0_B]
if winner_pool:
    winner = max(winner_pool, key=lambda o: o[1])
    print(f"\nupload first: {winner[0]} — highest Fold A GT* among the options that are not worse than E0 on Fold B")
else:
    print(f"\nno option is not worse than E0 on Fold B (E0 B GT* {e0_B:.6f}); keep the current repeat2 file")


SECTION 9d — probe submissions
frac>=0.25 x log1p                         -> outputs/submission_probe_frac25.csv: rows 1,260 | sha256[:8] 6c876c1a | cold fills 3
frac>=0.75 x log1p                         -> outputs/submission_probe_frac75.csv: rows 1,260 | sha256[:8] 84696a32 | cold fills 6
frac>=0.90 x log1p                         -> outputs/submission_probe_frac90.csv: rows 1,260 | sha256[:8] 444096cd | cold fills 6
building personalised-exposure matrices for the final window (weeks 31-42)...
GT* pick ((d) x days^0.25)                 -> outputs/submission_best_gtstar.csv: rows 1,260 | sha256[:8] c2805b0e | cold fills 3

file                                           |    A GT* |    B GT* | B guard vs E0
E10 already submitted (theta=0.5 x log1p)      | 0.396878 | 0.398006 |          True
outputs/submission_probe_frac25.csv            | 0.406816 | 0.411554 |          True
outputs/submission_probe_frac75.csv            | 0.377006 | 0.371719 |          True
outputs/submission_probe_fr

### 9e - manual final round (added by the student, NOT part of the OpenCode session)

Added by hand after the OpenCode session ended. The cell re-uses `eval9e` and `write_sub9e` from section 9d. Theta is NOT selected by GT* (circular: GT* is built with theta 0.25); it is chosen on the returned Kaggle private score.

In [88]:
# --- Section 9e (MANUAL, added by the student): threshold sweep with the best recipe ---
print("=" * 78)
print("SECTION 9e - manual final round: log1p x chhw exposure x days^0.25")
print("=" * 78)

MANUAL_CFGS = [
    # (file, theta, gamma)
    ("outputs/submission_gt_th50.csv",    0.50, 0.5),
    ("outputs/submission_gt_th60.csv",    0.60, 0.5),
    ("outputs/submission_gt_th75.csv",    0.75, 0.5),
    ("outputs/submission_gt_th50_g1.csv", 0.50, 1.0),
]

print(f"{'config':44s} | {'A GT*':>8s} | {'A GT-1':>8s} | {'B GT*':>8s} | {'B GT-1':>8s}")
for _path, _th, _gam in MANUAL_CFGS:
    eval9e(f"theta={_th:g} x log1p x chhw g={_gam:g} x days^0.25", _th,
           fmult="log1p", exposure="chhw", gamma=_gam, beta=0.25)

for _path, _th, _gam in MANUAL_CFGS:
    write_sub9e(_th, _path, f"theta={_th:g}, gamma={_gam:g}",
                fmult="log1p", exposure="chhw", gamma=_gam, beta=0.25)

SECTION 9e - manual final round: log1p x chhw exposure x days^0.25
config                                       |    A GT* |   A GT-1 |    B GT* |   B GT-1
theta=0.5 x log1p x chhw g=0.5 x days^0.25 | 0.412705 | 0.258502 | 0.414434 | 0.266637
theta=0.6 x log1p x chhw g=0.5 x days^0.25 | 0.408740 | 0.254315 | 0.405551 | 0.258525
theta=0.75 x log1p x chhw g=0.5 x days^0.25 | 0.396442 | 0.244414 | 0.392704 | 0.248571
theta=0.5 x log1p x chhw g=1 x days^0.25   | 0.419810 | 0.262686 | 0.419047 | 0.269735
theta=0.5, gamma=0.5                       -> outputs/submission_gt_th50.csv: rows 1,260 | sha256[:8] cbae38cb | cold fills 4
theta=0.6, gamma=0.5                       -> outputs/submission_gt_th60.csv: rows 1,260 | sha256[:8] fcbd0b74 | cold fills 6
theta=0.75, gamma=0.5                      -> outputs/submission_gt_th75.csv: rows 1,260 | sha256[:8] 258caceb | cold fills 6
theta=0.5, gamma=1                         -> outputs/submission_gt_th50_g1.csv: rows 1,260 | sha256[:8] d5d7b1a2 | c

## 10. Case study: 2 fixed users, top-5 for each method


### Case study — MF vs CBF vs hybrid for the two fixed users

Same users as in section 7. Hit marks refer to GT-1 in the Fold-A target window.


In [89]:
# --- Section 10: case study (MF vs CBF vs hybrid) ---
print("=" * 72)
print("SECTION 10 — case study: best MF vs best CBF vs hybrid")
print("=" * 72)
case_methods = [("SVD", best_recs_svd), ("CBF", best_cbf_recs_A), ("Hybrid", best_hybrid_recs_A)]
for u in CASE_USERS:
    print(f"\n=== case user {u} (labelled airings in history: {int(labelled_counts_all.get(u, 0))}) ===")
    hist_u = labA[labA["user_id"] == u].groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).head(5)
    print("history top-5:")
    for sid, sec in hist_u.items():
        si, title, cat, gen = show_info(sid)
        print(f"  {si:6d} | {sec:7.0f}s | {cat:12s} | {gen:26s} | {str(title)[:55]}")
    gt_u = sorted(int(s) for s in gt1_eval.get(u, set()))
    print("GT-1 top-5:", gt_u)
    hist_set_u = set(int(s) for s in labA.loc[labA["user_id"] == u, "tv_show_id"].unique())
    for mname, recs in case_methods:
        top5 = [int(s) for s in recs.get(u, [])[:K]]
        hits = [s for s in top5 if s in gt1_eval.get(u, set())]
        print(f"{mname} top-5: hits {len(hits)}/5")
        for rank, sid in enumerate(top5, 1):
            si, title, cat, gen = show_info(sid)
            mark = "HIT" if sid in gt1_eval.get(u, set()) else "-"
            print(f"  {rank}. {si:6d} | {cat:12s} | {gen:24s} | {str(title)[:55]} | {mark}")
        rep = sum(1 for s in top5 if s in hist_set_u)
        print(f"  repeats of own history in top-5: {rep}/5")
    ls = [set(int(s) for s in recs.get(u, [])[:K]) for _, recs in case_methods]
    print(f"  overlap SVD&CBF={len(ls[0] & ls[1])}, SVD&Hybrid={len(ls[0] & ls[2])}, CBF&Hybrid={len(ls[1] & ls[2])}")


SECTION 10 — case study: best MF vs best CBF vs hybrid

=== case user 13571199046623737884 (labelled airings in history: 9614) ===
history top-5:
  2400480 |  892243s | Инфо         | общество и политика        | 'ТСН. Неделя'.
   20088 |  350704s | Развлечения  | информация (комплексная)   | 'Завтрак с 1+1'. Информационно-развлекательная программ
  240081 |  253744s | Инфо         | общество и политика        | Факты. День.
  2400508 |  247527s | Инфо         | информация (комплексная)   | Сегодня. Итоги с Олегом Панютой.
  6500479 |  244989s | Познавательное | путешествия                | 'Мир наизнанку. Бразилия', 36 эп.
GT-1 top-5: [200352, 240081, 700369, 2400480, 90074629]
SVD top-5: hits 1/5
  1. 200405 | Развлечения  | юмор | развлекательные   | 'Мамахохотала-шоу'. Лучшее. | -
  2. 10003159 | Сериалы      | драма | мелодрама | детектив | Т/с 'Ничто не случается дважды', 1 и 2 с. | -
  3. 700369 | Развлечения  | юмор | развлекательные   | 'Лига смеха', 6 сезон, 1 и 2 эп. | HIT
 

### Compact report tables for the two case users

Both users, Fold A window (history weeks 11-20, truth weeks 21-30). `x` marks a GT-1 hit within the top 5. The last block lists three observations computed from the tables above it (hit counts, own-history backing, and category diversity) — no numbers are typed by hand.


In [90]:
# --- Section 10b: compact report tables for the two case users (Fold A window) ---
print("=" * 72)
print("REPORT TABLES — case users, Fold A window (truth = weeks 21-30)")
print("=" * 72)
REPORT_METHODS = [
    (f"SVD k={best_svd_k}", best_recs_svd),
    ("ALS tuned", best_recs_als),
    ("CBF best", best_cbf_recs_A),
    ("Hybrid best", best_hybrid_recs_A),
    (f"Own-history: {exp9c_final_label}", exp9c_final_recs_A),
]
for u in CASE_USERS:
    gt = set(int(x) for x in gt1_eval.get(u, set()))
    hist_u = labA[labA["user_id"] == u]
    own = set(int(x) for x in hist_u["tv_show_id"].unique())
    own_pool = own & set(int(x) for x in candA)
    print(f"\n### user {u} | labelled Fold-A airings {len(hist_u):,} | own programs in pool {len(own_pool)}")
    print("| rank | history top-5 program | title | category |")
    print("|---|---|---|---|")
    h5 = hist_u.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).head(5)
    for k_, (sid, sec) in enumerate(h5.items(), start=1):
        sid_i, title, cat, _ = show_info(sid)
        print(f"| {k_} | {sid_i} | {title[:40]} | {cat[:18]} |")
    print(f"\nGT-1 in this window: {sorted(gt)}")
    print("\n| method | top-5 (x = GT-1 hit) | hits |")
    print("|---|---|---|")
    for name, recs in REPORT_METHODS:
        r = [int(x) for x in recs.get(u, [])][:5]
        marks = "".join("x" if x in gt else "." for x in r)
        print(f"| {name} | {marks} ({' '.join(str(x) for x in r)}) | {marks.count('x')}/5 |")
    counts = {name: sum(1 for x in [int(y) for y in recs.get(u, [])][:5] if x in gt) for name, recs in REPORT_METHODS}
    own_slots = {name: sum(1 for x in [int(y) for y in recs.get(u, [])][:5] if x in own) for name, recs in REPORT_METHODS}
    final_name, final_recs = REPORT_METHODS[-1]
    rfin = [int(x) for x in final_recs.get(u, [])][:5]
    print("\nobservations:")
    print("  1) GT-1 hits per method: " + " | ".join(f"{n}={v}/5" for n, v in counts.items()))
    print("  2) slots backed by an own-history program: " + " | ".join(f"{n}={v}/5" for n, v in own_slots.items()))
    print(f"  3) {final_name}: {len(set(rfin) & gt)}/5 GT-1 hits | fill slots used {5 - own_slots[final_name]} | distinct categories {len({show_info(x)[2] for x in rfin})}")


REPORT TABLES — case users, Fold A window (truth = weeks 21-30)

### user 13571199046623737884 | labelled Fold-A airings 9,614 | own programs in pool 638
| rank | history top-5 program | title | category |
|---|---|---|---|
| 1 | 2400480 | 'ТСН. Неделя'. | Инфо |
| 2 | 20088 | 'Завтрак с 1+1'. Информационно-развлекат | Развлечения |
| 3 | 240081 | Факты. День. | Инфо |
| 4 | 2400508 | Сегодня. Итоги с Олегом Панютой. | Инфо |
| 5 | 6500479 | 'Мир наизнанку. Бразилия', 36 эп. | Познавательное |

GT-1 in this window: [200352, 240081, 700369, 2400480, 90074629]

| method | top-5 (x = GT-1 hit) | hits |
|---|---|---|
| SVD k=32 | ..x.. (200405 10003159 700369 10003178 400426) | 1/5 |
| ALS tuned | ..... (400426 5800509 90074731 90076007 200405) | 0/5 |
| CBF best | ..... (1000608 10002454 1000289 1000236 10002829) | 0/5 |
| Hybrid best | x.x.. (2400480 20088 240081 2400508 6500479) | 2/5 |
| Own-history: E7  best personalised (chhw) gamma=0.25 | x.x.. (2400480 20088 240081 2400508 6500479)

In [91]:
# --- Section 10c (MANUAL): case-study table for the FINAL configuration ---
FINAL_CFG = dict(theta=0.50, fmult="log1p", exposure="chhw", gamma=1.0, beta=0.25)
recsA_fin, _ = rank9e(labA, candA, sched_cnt_A, EVAL_USERS, pop_fallback_A,
                      FINAL_CFG["theta"], FINAL_CFG["fmult"], FINAL_CFG["exposure"],
                      FINAL_CFG["gamma"], PE9["A"], FINAL_CFG["beta"])
for u in CASE_USERS:
    gt_s = set(int(x) for x in GT_STAR["A"].get(u, set()))
    gt_1 = set(int(x) for x in gt1_eval.get(u, set()))
    print(f"\n### user {u}")
    print("| rank | tv_show_id | title | category | in GT* | in GT-1 |")
    print("|---|---|---|---|---|---|")
    for k_, sid in enumerate([int(x) for x in recsA_fin.get(u, [])][:5], start=1):
        sid_i, title, cat, _ = show_info(sid)
        print(f"| {k_} | {sid_i} | {title[:40]} | {cat[:18]} | {'yes' if sid in gt_s else 'no'} | {'yes' if sid in gt_1 else 'no'} |")


### user 13571199046623737884
| rank | tv_show_id | title | category | in GT* | in GT-1 |
|---|---|---|---|---|---|
| 1 | 2400480 | 'ТСН. Неделя'. | Инфо | yes | yes |
| 2 | 240081 | Факты. День. | Инфо | yes | yes |
| 3 | 20088 | 'Завтрак с 1+1'. Информационно-развлекат | Развлечения | no | no |
| 4 | 12001682 | М/с 'Губка Боб Квадратные Штаны'. | Детям | no | no |
| 5 | 2400508 | Сегодня. Итоги с Олегом Панютой. | Инфо | no | no |

### user 8150816946177670084
| rank | tv_show_id | title | category | in GT* | in GT-1 |
|---|---|---|---|---|---|
| 1 | 2400480 | 'ТСН. Неделя'. | Инфо | no | no |
| 2 | 240081 | Факты. День. | Инфо | yes | yes |
| 3 | 2400508 | Сегодня. Итоги с Олегом Панютой. | Инфо | no | no |
| 4 | 20088 | 'Завтрак с 1+1'. Информационно-развлекат | Развлечения | no | no |
| 5 | 6500479 | 'Мир наизнанку. Бразилия', 36 эп. | Познавательное | no | no |


## 11. Final training and submission.csv


### Safe submission — MF only (SVD)

Best matrix-factorization variant from section 7: TruncatedSVD (seeded, random_state=42) with k=32 on log1p(total_seconds) of the full history (weeks 11-30). Candidate pool = shows with tv_show_id != 0 airing in weeks 31-42 (week derived from `start_dt` via isocalendar, same rule as build_split; asserted equal to `candF`). Already-watched shows are kept; users with no history get the popularity-by-seconds top-5. Written to `outputs/submission_mf.csv`.

In [92]:
# --- Safe submission: MF (SVD k=32) trained on the full history ---
import hashlib
from pathlib import Path

print("=" * 72)
print("SAFE SUBMISSION — SVD on full history (weeks 11-30) -> weeks 31-42")
print("=" * 72)

sched_final = schedule_nd[schedule_nd["start_dt"].dt.isocalendar().week.astype(int).between(31, 42)]
cand_final = sorted(int(s) for s in sched_final["tv_show_id"].dropna().unique() if int(s) != 0)
cand_final_set = set(cand_final)
print(f"candidate pool (w31-42): {len(cand_final):,} shows | schedule rows: {len(sched_final):,}")
assert len(cand_final) == 6642, len(cand_final)
if "candF" in globals():
    assert set(int(s) for s in candF) == cand_final_set
    print("  pool matches candF exactly: True")

hist_final = pairs[(pairs["tv_show_id"] != 0) & (pairs["iso_week"].between(11, 30))
                   & (pairs["overlap_sec"] >= 60) & (pairs["overlap_frac"] >= 0.0)].copy()
print(f"full history interactions (>=60s): {len(hist_final):,}")

gF = hist_final.groupby(["user_id", "tv_show_id"])["overlap_sec"].sum().reset_index(name="sec")
usersF = sorted(gF["user_id"].unique())
itemsF = sorted(gF["tv_show_id"].unique())
upF = {u: i for i, u in enumerate(usersF)}
ipF = {i: j for j, i in enumerate(itemsF)}
secF = np.log1p(gF["sec"].astype("float64").values)
MF = sp.csr_matrix((secF, (gF["user_id"].map(upF).astype(np.int32).values,
                           gF["tv_show_id"].map(ipF).astype(np.int32).values)),
                   shape=(len(usersF), len(itemsF)))
svd_final = TruncatedSVD(n_components=best_svd_k, random_state=42)
UF = svd_final.fit_transform(MF)
VF = svd_final.components_
scoresF = UF @ VF
print(f"matrix: {MF.shape[0]:,} users x {MF.shape[1]:,} items | factors={best_svd_k} (seeded)")

sample = pd.read_csv("data/submission.csv", dtype={"user_id": str})
sub_users = sample["user_id"].tolist()
print(f"submission users: {len(sub_users):,}")

pop_final = hist_final.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False)
pop_final5 = [int(s) for s in pop_final.index if int(s) in cand_final_set][:K]
assert len(pop_final5) == K, pop_final5

cand_cols_F = np.array([ipF[s] for s in cand_final if s in ipF], dtype=np.int64)
cand_shows_F = np.array([int(s) for s in cand_final if s in ipF], dtype=np.int64)
assert (cand_shows_F != 0).all()
print(f"candidate shows with factors: {len(cand_shows_F):,} / {len(cand_final):,}")

def rec_top5(u):
    if u not in upF:
        return pop_final5
    sub = scoresF[upF[u]][cand_cols_F]
    if not np.isfinite(sub).any():
        return pop_final5
    top = np.argpartition(-sub, kth=K - 1)[:K]
    top = top[np.argsort(-sub[top])]
    return [int(cand_shows_F[int(c)]) for c in top]

recs_final = {u: rec_top5(u) for u in sub_users}
n_fallback = sum(1 for u in sub_users if u not in upF)

assert len(sub_users) == len(sample), (len(sub_users), len(sample))
for u in sub_users:
    r = recs_final[u]
    assert len(r) == K and len(set(r)) == K and 0 not in r, (u, r)
    assert set(r) <= cand_final_set, (u, r)

out = pd.DataFrame({"user_id": sub_users,
                    "tv_show_id": [" ".join(str(int(s)) for s in recs_final[u]) for u in sub_users]})
Path("outputs").mkdir(exist_ok=True)
out.to_csv("outputs/submission_mf.csv", index=False)
sha = hashlib.sha256(Path("outputs/submission_mf.csv").read_bytes()).hexdigest()
print(f"fallback users (no history): {n_fallback}")
print("\nfirst 3 rows:")
print(out.head(3).to_string(index=False))
print(f"rows: {len(out):,}")
print(f"sha256[:8]: {sha[:8]}")

SAFE SUBMISSION — SVD on full history (weeks 11-30) -> weeks 31-42
candidate pool (w31-42): 6,642 shows | schedule rows: 1,295,408
  pool matches candF exactly: True
full history interactions (>=60s): 3,742,246
matrix: 4,895 users x 4,362 items | factors=32 (seeded)
submission users: 1,260
candidate shows with factors: 3,114 / 6,642
fallback users (no history): 2

first 3 rows:
            user_id                                 tv_show_id
8377619604347126107        20088 2400480 6500479 200477 200337
8381667675275833309      2400480 240081 200352 12001682 200432
8387147770138767246 12001682 12002856 2400467 12001734 2400480
rows: 1,260
sha256[:8]: c9f84e23


### Final submission files — hybrid and CBF

Components rebuilt on the full history (weeks 11-30, >= 60 s); MF = SVD k=32 (seeded), CBF = TF-IDF variant chosen on Fold A with log1p weights, personal history and popularity from the same window. Hybrid weights are the ones selected on Fold A above. Same users/order and format as `data/submission.csv`.


In [93]:
# --- Section 11: final submission files (hybrid + CBF) ---
import hashlib
from pathlib import Path

print("=" * 72)
print("FINAL SUBMISSIONS — hybrid and CBF on the full history")
print("=" * 72)

sample = pd.read_csv("data/submission.csv", dtype={"user_id": str})
sub_users2 = sample["user_id"].tolist()
pool_shows_F = np.array([int(s) for s in cand_final], dtype=np.int64)

def final_components():
    n_users = len(sub_users2)
    n_pool = len(pool_shows_F)
    scores_all = UF @ VF
    cols = np.array([ipF.get(int(s), -1) for s in pool_shows_F], dtype=np.int64)
    valid = cols >= 0
    mf = np.zeros((n_users, n_pool), dtype="float64")
    for i, u in enumerate(sub_users2):
        if u in upF and valid.any():
            mf[i, valid] = scores_all[upF[u]][cols[valid]]
    ids = [s for s in sorted(set(int(x) for x in cand_final) | set(int(x) for x in hist_final["tv_show_id"].unique())) if s in first_item.index]
    docs = [text_c(first_item.loc[s]) for s in ids]
    vecF = TfidfVectorizer(token_pattern=r"[^\s]+", lowercase=True, sublinear_tf=True, norm="l2")
    XF = vecF.fit_transform(docs)
    idxF = {s: i for i, s in enumerate(ids)}
    g = hist_final.groupby(["user_id", "tv_show_id"])["overlap_sec"].sum().reset_index()
    g2 = g[g["tv_show_id"].isin(idxF)]
    u2i = {u: i for i, u in enumerate(sub_users2)}
    g2 = g2[g2["user_id"].isin(u2i)]
    w = np.log1p(g2["overlap_sec"].values.astype("float64"))
    Wmat = sp.csr_matrix((w, (g2["user_id"].map(u2i).astype(np.int32).values,
                              g2["tv_show_id"].map(idxF).astype(np.int32).values)),
                         shape=(n_users, len(ids)))
    PF = Wmat @ XF
    norms = np.sqrt(np.asarray(PF.multiply(PF).sum(axis=1)).ravel())
    norms[norms == 0] = 1.0
    PF = sp.diags(1.0 / norms) @ PF
    col_cand = np.array([idxF[s] for s in pool_shows_F if s in idxF], dtype=np.int64)
    shows_cand = np.array([s for s in pool_shows_F if s in idxF], dtype=np.int64)
    S_cbf = (PF @ XF[col_cand].T).toarray()
    pos_map = {int(s): j for j, s in enumerate(shows_cand)}
    cand_pos = np.array([pos_map.get(int(s), -1) for s in pool_shows_F], dtype=np.int64)
    valid_c = cand_pos >= 0
    cbf = np.zeros((n_users, n_pool), dtype="float64")
    for i, u in enumerate(sub_users2):
        if u in u2i:
            cbf[i, valid_c] = S_cbf[u2i[u]][cand_pos[valid_c]]
    pool_pos = {int(s): j for j, s in enumerate(pool_shows_F)}
    own = np.zeros((n_users, n_pool), dtype="float64")
    g3 = g[g["tv_show_id"].isin(pool_pos) & g["user_id"].isin(u2i)]
    if len(g3):
        rr = g3["user_id"].map(u2i).values.astype(np.int64)
        cc = g3["tv_show_id"].map(pool_pos).values.astype(np.int64)
        own[rr, cc] = np.log1p(g3["overlap_sec"].values.astype("float64"))
    pop_s = hist_final.groupby("tv_show_id")["overlap_sec"].sum()
    pop_v = np.array([np.log1p(float(pop_s.get(int(s), 0.0))) for s in pool_shows_F])
    pop = np.tile(pop_v, (n_users, 1))
    return dict(mf=mf, cbf=cbf, own=own, pop=pop)

compF = final_components()
normF = {n: per_user_minmax(compF[n]) for n in NAMES}

POP_F = [int(s) for s in list(pop_final5)[:K]]

def top5_from_matrix(S, label):
    recs = {}
    n_fb = 0
    for i, u in enumerate(sub_users2):
        row = S[i]
        if not np.isfinite(row.max()) or row.max() <= 0:
            recs[u] = list(POP_F)
            n_fb += 1
            continue
        top = np.argpartition(-row, kth=K - 1)[:K]
        top = top[np.argsort(-row[top])]
        recs[u] = [int(pool_shows_F[int(c)]) for c in top]
    print(f"{label}: popularity-fallback rows = {n_fb} / {len(sub_users2)}")
    return recs

S_hyb = np.zeros_like(compF["mf"])
for wi, n in zip(best_w, NAMES):
    if wi:
        S_hyb += wi * normF[n]
hist_users_final = set(hist_final["user_id"].unique())
n_cold_final = sum(1 for u in sub_users2 if u not in hist_users_final)
print(f"submission users with no history at all: {n_cold_final} / {len(sub_users2)}")
recs_hyb = top5_from_matrix(S_hyb, "hybrid")
recs_cbf_only = top5_from_matrix(normF["cbf"], "CBF")

def write_submission(recs, path):
    out = pd.DataFrame({"user_id": sub_users2,
                        "tv_show_id": [" ".join(str(int(s)) for s in recs[u]) for u in sub_users2]})
    out.to_csv(path, index=False)
    sha = hashlib.sha256(Path(path).read_bytes()).hexdigest()
    assert len(out) == len(sample)
    for u in sub_users2:
        r = recs[u]
        assert len(r) == K and len(set(r)) == K and 0 not in r
        assert set(r) <= cand_final_set
    print(f"\n{path}: rows {len(out):,} | sha256[:8] {sha[:8]}")
    print(out.head(3).to_string(index=False))
    return out

write_submission(recs_hyb, "outputs/submission_hybrid.csv")
write_submission(recs_cbf_only, "outputs/submission_cbf.csv")
print(f"\ncold users (no history): {n_cold_final}; hybrid rows using popularity fallback: see counts above")
print("\nhybrid weights used:", best_w)
print("which file is which: submission_mf.csv = SVD only; submission_cbf.csv = CBF only; submission_hybrid.csv = hybrid")


FINAL SUBMISSIONS — hybrid and CBF on the full history
submission users with no history at all: 2 / 1260
hybrid: popularity-fallback rows = 2 / 1260
CBF: popularity-fallback rows = 2 / 1260

outputs/submission_hybrid.csv: rows 1,260 | sha256[:8] 2550c033
            user_id                                 tv_show_id
8377619604347126107         2400480 240081 20088 200337 200352
8381667675275833309       2400480 500346 200432 240081 6700482
8387147770138767246 12001682 2400467 6500479 12001732 12002355

outputs/submission_cbf.csv: rows 1,260 | sha256[:8] 9c0c56fb
            user_id                                 tv_show_id
8377619604347126107         700369 400335 700389 200476 200344
8381667675275833309         700369 500346 700389 200344 500359
8387147770138767246 700369 10003062 12003110 10002454 10001856

cold users (no history): 2; hybrid rows using popularity fallback: see counts above

hybrid weights used: (0, 0, 1, 0)
which file is which: submission_mf.csv = SVD only; submissi

## 12. Limitations and failure cases


### Limitations and failure cases

All numbers below are printed in the cells above.

1. **Repeat behaviour dominates the task.** GT-1 hits that are shows the user already watched: popularity 97.9%, personal history 99.9%, ALS 99.1%, SVD 99.7%; the new-show share of hits is 0.0034 (CBF) and 0.0000 (SVD). The signal is mostly "which of my shows air again in weeks 31-42".
2. **The tuned hybrid collapses to one component.** The 15-weight grid on Fold A selected (mf, cbf, own, pop) = (0, 0, 1, 0): GT-1 0.301423 (A) / 0.303467 (B). Blending hurts: (0.5, 0, 0.5, 0) gives 0.280420, (0.5, 0.5, 0, 0) gives 0.113044, (0.25, 0.25, 0.25, 0.25) gives 0.205324. MF and CBF add nothing on top of own history under this normalization and this pool.
3. **CBF is near-useless on this metadata.** Best CBF 0.023807 / 0.028240 vs popularity 0.180193 / 0.192483. `tv_show_genre_3` is missing in 76.58% of pool shows, `tv_show_genre_2` in 38.14%, and titles mix episode-level and show-level strings.
4. **Method A loses to SVD by about 2x** after tuning (0.122533 vs 0.221413 on Fold A) at about 13x the fit cost (1.19 s vs 0.09 s). The pre-fix value 0.002911 was an alpha-scaling bug; the remaining gap is a model/dataset mismatch, not a bug.
5. **GT-1 and GT-2 disagree on the winner.** Best SVD wins GT-1 (0.221413 / 0.225958) while popularity-by-users wins GT-2 (0.553070 / 0.552938). GT-2 (airings watched at least half) rewards whatever airs everywhere.
6. **Fold B is not independent.** Fold A history (weeks 11-20) overlaps the Fold B target (weeks 19-28), and both folds use the same evaluated users and pool rule, so B is a consistency check rather than a held-out test.
7. **GT-1 is defined by seconds watched**, which is right-skewed (quantiles at alpha_t=40: 165 / 271 / 351 / 581 s), so one long film can enter GT-1 ahead of several short news airings.
8. **The candidate pool is schedule-bounded.** Only the 6,642 shows airing in weeks 31-42 can be recommended; 88.9% of schedule rows carry `tv_show_id == 0` and are unusable.
9. **Cold start is a fallback, not a model.** 2 of 1,260 submission users have no history at all; their rows are the popularity-by-seconds top-5 in both `submission_hybrid.csv` and `submission_cbf.csv`. Case user 2 has only 1 labelled airing, so its recommendations are dominated by that single show.
10. **Not attempted:** session/sequence models, per-user confidence weighting in ALS, and evaluation on more than two temporal folds.
11. **The own-history ceiling is real.** A perfect ranking restricted to the user's own history would reach GT-1 0.699757 (A) / 0.681307 (B), but only 34.59% (431 of 1,246) of Fold-A and 30.82% (384 of 1,246) of Fold-B users have their whole GT-1 inside their history. The remaining headroom is in programs the user never watched; the tuned hybrid gives that part weight 0, and CBF alone scores 0.023807 / 0.028240.
12. **Personalised exposure is history-thin.** The channel x hour profile is estimated from one user's airings over 10 weeks (Fold A) or 8 weeks (Fold B) with add-one smoothing over 405 channels and 168 hour bins; for a user with a single labelled airing (case user 2) the factor is driven by that one channel-hour cell. The gain over the global log1p multiplier is small: +0.004008 GT-1 on Fold A (0.314644 -> 0.318652) and +0.004738 on Fold B (0.320928 -> 0.325666), and its per-user variance is not measured.


### Conclusions

1. On this dataset the decisive signal is repetition: the best configuration of the four-component hybrid is its own-history component alone (GT-1 0.301423 / 0.303467, GT-2 0.510269 / 0.504754).
2. Model quality still matters inside the non-repeat part: SVD (0.221413 / 0.225958) clearly beats tuned ALS (0.122533 / 0.126070), and both beat popularity on GT-1, while popularity stays the best GT-2 model.
3. Content features built from the available metadata are too sparse (`tv_show_genre_3` missing 76.58%) to help: CBF alone is about 8x below popularity and its weight is driven to zero by the grid.
4. A well-tuned blend is not automatically better than its best component: every mixed weight combination scored below the pure history component on the tuning fold, and the same ordering held on Fold B.
5. The returned Kaggle scores showed that GT-1 (top-5 by watched seconds) is a poor proxy: counting only airings watched for at least half (theta 0.5) raised the private score from 0.36849 (distinct days) to 0.41211, and the first five files returned the opposite order of validation GT-1. Later variants (theta, exposure gamma, days exponent, recency decay) reached 0.42853 (`outputs/submission_gt_th50_g1_tau18.csv`, designated final).
6. Mild recency decay (tau 9-36 weeks) helps GT* slightly on both folds and the Kaggle score by more; strong decay (tau <= 3) hurts. Validation cannot separate tau 12, 18 and 24, so tau=18 is designated only because it had the best returned private score.
7. Filling short own-history lists with SVD scores instead of popularity does not help; only about 1-4% of users have fewer than five own candidates, and they carry about 1% of MAP@5.
8. Caveat: the final file was chosen by the returned private score after more than a dozen leaderboard-informed variants, and the GT* threshold was itself calibrated against the Kaggle order, so 0.42853 is an optimistic estimate.
9. Label note: personal history is marked "diagnostic, not counted" in sections 6-7 because it started as a sanity check; in section 9 it is one of four weighted components, and the tuner selected it with weight 1.
10. Sections 9e, 9h and 9i were added by the student after the OpenCode session ended (see their markdown cells); they are not in the session export.

## 13. Environment and runtime


In [94]:
# --- environment snapshot + full-notebook runtime ---
import os, sys, platform, subprocess, resource, time
import importlib.metadata as md

def _sh(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=False).stdout.strip()
    except Exception as exc:
        return f"n/a ({type(exc).__name__})"

print(platform.platform())
print(f"machine: {platform.machine()} | processor: {platform.processor() or 'n/a'}")
print(f"CPU model: {_sh(['sysctl', '-n', 'machdep.cpu.brand_string']) or 'n/a'}")
_mem_b = _sh(["sysctl", "-n", "hw.memsize"])
print(f"RAM: {int(_mem_b) / 1024**3:.1f} GiB" if _mem_b.isdigit() else "RAM: n/a")
print(f"logical cores: {os.cpu_count()}")
print(f"thread env: OMP_NUM_THREADS={os.environ.get('OMP_NUM_THREADS', 'unset')} | OPENBLAS_NUM_THREADS={os.environ.get('OPENBLAS_NUM_THREADS', 'unset')} | MKL_NUM_THREADS={os.environ.get('MKL_NUM_THREADS', 'unset')}")
print(f"python: {sys.version.split()[0]} ({sys.executable})")
for _pkg in ["numpy", "pandas", "scipy", "scikit-learn", "implicit", "pyarrow", "matplotlib"]:
    try:
        print(f"{_pkg}: {md.version(_pkg)}")
    except md.PackageNotFoundError:
        print(f"{_pkg}: not installed")
_peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
_peak_gb = _peak / 1024**3 if sys.platform == "darwin" else _peak / 1024**2
print(f"peak RSS: {_peak_gb:.2f} GiB")
print(f"notebook wall-clock runtime: {time.time() - NOTEBOOK_T0:.1f} s")


macOS-26.6.2-arm64-arm-64bit-Mach-O
machine: arm64 | processor: arm
CPU model: Apple M4
RAM: 24.0 GiB
logical cores: 10
thread env: OMP_NUM_THREADS=unset | OPENBLAS_NUM_THREADS=unset | MKL_NUM_THREADS=unset
python: 3.13.6 (/usr/local/bin/python3)
numpy: 2.5.4
pandas: 2.3.3
scipy: 1.16.2
scikit-learn: 1.7.2
implicit: 0.7.3
pyarrow: 22.0.0
matplotlib: 3.10.6
peak RSS: 8.65 GiB
notebook wall-clock runtime: 768.4 s


In [95]:
write_sub9e(0.50, "outputs/submission_gt_th50_g15.csv", "th50 g1.5", fmult="log1p", exposure="chhw", gamma=1.5, beta=0.25)
write_sub9e(0.50, "outputs/submission_gt_th50_b50.csv", "th50 b0.5", fmult="log1p", exposure="chhw", gamma=1.0, beta=0.5)
# write_sub9e(0.40, "outputs/submission_gt_th40_g1.csv", "th40 g1", fmult="log1p", exposure="chhw", gamma=1.0, beta=0.25)
# write_sub9e(0.50, "outputs/submission_gt_th50_sqrt.csv", "th50 sqrt", fmult="sqrt", exposure="chhw", gamma=1.0, beta=0.25)

th50 g1.5                                  -> outputs/submission_gt_th50_g15.csv: rows 1,260 | sha256[:8] fb6902c6 | cold fills 4
th50 b0.5                                  -> outputs/submission_gt_th50_b50.csv: rows 1,260 | sha256[:8] e1fa63b8 | cold fills 4


{'8377619604347126107': [2400480, 240081, 20088, 2400508, 12002955],
 '8381667675275833309': [2400480, 240081, 12001682, 6700482, 200344],
 '8387147770138767246': [12001682, 12002856, 12002355, 12001734, 12001732],
 '8397181578236218580': [2400480, 20088, 240081, 6700482, 2400467],
 '8404698046253197367': [2400480, 240081, 20088, 6200371, 6700482],
 '8431897575013155337': [2400480, 500315, 1100411, 240081, 2400508],
 '8437234709656005918': [12001682, 2400480, 12001734, 12002856, 240081],
 '8441113802219662639': [2400480, 240081, 2400467, 12001682, 2400508],
 '8443187512168717371': [2400480, 240081, 2400508, 20088, 12001682],
 '8449544243572273063': [12002896, 12002955, 2400480, 1000376, 12001682],
 '8451885646926746962': [2400480, 2400467, 20088, 2400508, 500331],
 '8465045669150315534': [240081, 2400480, 2400508, 20088, 10001261],
 '8469176037437965007': [12002856, 2400480, 12001682, 240081, 12001732],
 '8477890851739099969': [12001682, 12001734, 12002896, 12002955, 12002856],
 '84810

## 9f. Recency weighting of own-history hits

In sections 9d-9e every airing with `overlap_frac >= theta` counts exactly 1 in the own-history value, whether it happened in week 11 or in week 30. This section changes one thing only: each hit is decayed by `w = exp(-age_weeks / tau)`, with `age_weeks = (last airing date of the history window - airing date) / 7 days`, and the weighted sum replaces the raw hit count. `ndays` and all other factors are unchanged.

Rationale: the target window is weeks 31-42, so recent viewing should predict it better than viewing from months earlier.

Implementation: `frame9e`, `rank9e`, `eval9e` and `write_sub9e` gained an optional `tau` (default `None` = old behaviour, bit-identical output). The recipe is held fixed at the best returned configuration (`theta=0.50, fmult="log1p", exposure="chhw", gamma=1.0, beta=0.25`); only `tau` varies over `None, 12, 6, 3, 1.5` weeks.

In [96]:
# --- Section 9f: recency weighting of own-history hits (exp decay, tau in weeks) ---
import hashlib, tempfile
from pathlib import Path

print("=" * 78)
print("SECTION 9f — recency weighting of own-history hits")
print("=" * 78)
t0_9f = time.time()

BASE9F = dict(theta=0.50, fmult="log1p", exposure="chhw", gamma=1.0, beta=0.25)
TAUS9F = (None, 12.0, 6.0, 3.0, 1.5)

def run9f(label, tau):
    r = eval9e(label, BASE9F["theta"], fmult=BASE9F["fmult"], exposure=BASE9F["exposure"],
               gamma=BASE9F["gamma"], beta=BASE9F["beta"], tau=tau)
    r["tau"] = tau
    return r

print("\n[1] sanity: tau=None must reproduce the section-9e th50_g1 run exactly")
ref9f = next(r for r in RUN9E if r["label"].startswith("theta=0.5 x log1p x chhw g=1 x days^0.25"))
chk9f = run9f("9f tau=None (reproduction check)", None)
print(f"{'':30s} | {'A GT*':>8s} | {'A GT-1':>8s} | {'B GT*':>8s} | {'B GT-1':>8s}")
for _nm, _r in (("section 9e cell (recorded)", ref9f), ("tau=None rerun", chk9f)):
    print(f"{_nm:30s} | {_r['A'][0]:8.6f} | {_r['A'][1]:8.6f} | {_r['B'][0]:8.6f} | {_r['B'][1]:8.6f}")
assert chk9f["A"] == ref9f["A"] and chk9f["B"] == ref9f["B"], (chk9f["A"], ref9f["A"], chk9f["B"], ref9f["B"])
assert chk9f["recsA"] == ref9f["recsA"] and chk9f["recsB"] == ref9f["recsB"]
print("assert passed: identical MAP values and identical recommendations")

print("\n[2] sanity: tau=None must reproduce outputs/submission_gt_th50_g1.csv byte for byte")
_ref_path = Path("outputs/submission_gt_th50_g1.csv")
_ref_sha = hashlib.sha256(_ref_path.read_bytes()).hexdigest()
_tmp_dir = Path(tempfile.mkdtemp(prefix="tau_check_"))
_tmp_path = _tmp_dir / "tau_none_check.csv"
write_sub9e(BASE9F["theta"], str(_tmp_path), "tau=None sha check", fmult=BASE9F["fmult"],
            exposure=BASE9F["exposure"], gamma=BASE9F["gamma"], beta=BASE9F["beta"], tau=None)
_tmp_sha = hashlib.sha256(_tmp_path.read_bytes()).hexdigest()
print(f"sha256: rerun {_tmp_sha[:8]} | existing {_ref_path.name} {_ref_sha[:8]}")
assert _tmp_sha == _ref_sha, (_tmp_sha, _ref_sha)
print("assert passed: byte-identical submission")
_tmp_path.unlink()
_tmp_dir.rmdir()

print("\n[3] tau sweep (recipe fixed at the returned th50_g1 configuration)")
RUN9F = []
base9f = None
for _tau in TAUS9F:
    _r = run9f(f"9f tau={_tau}", _tau)
    RUN9F.append(_r)
    if _tau is None:
        base9f = _r
print()
print(f"{'tau (weeks)':12s} | {'A GT*':>8s} | {'A GT-1':>8s} | {'B GT*':>8s} | {'B GT-1':>8s} | {'dA GT*':>8s} | {'dB GT*':>8s}")
for _r in RUN9F:
    _nm = "none (old)" if _r["tau"] is None else f"{_r['tau']:g}"
    print(f"{_nm:12s} | {_r['A'][0]:8.6f} | {_r['A'][1]:8.6f} | {_r['B'][0]:8.6f} | {_r['B'][1]:8.6f} | "
          f"{_r['A'][0] - base9f['A'][0]:+8.6f} | {_r['B'][0] - base9f['B'][0]:+8.6f}")

print("\nverdict per tau (computed from the table above):")
for _r in RUN9F:
    if _r["tau"] is None:
        continue
    _dA = _r["A"][0] - base9f["A"][0]
    _dB = _r["B"][0] - base9f["B"][0]
    if _dA > 0 and _dB > 0:
        _v = "helped on both folds"
    elif _dA > 0:
        _v = "helped on Fold A only (Fold B not improved)"
    elif _dB > 0:
        _v = "helped on Fold B only (Fold A not improved)"
    else:
        _v = "did not help on either fold"
    print(f"  tau={_r['tau']:g}: dA GT* {_dA:+.6f} | dB GT* {_dB:+.6f} -> {_v}")

_cand9f = [r for r in RUN9F if r["tau"] is not None]
_bA = max(_cand9f, key=lambda r: r["A"][0])
_bB = max(_cand9f, key=lambda r: r["B"][0])
print(f"\nbest tau by Fold A GT*: {_bA['tau']:g} | best tau by Fold B GT*: {_bB['tau']:g} | folds agree: {_bA['tau'] == _bB['tau']}")

print("\n[4] submissions for the best two taus (Fold A GT* order, Fold B not worse than tau=None)")
_qual = sorted([r for r in _cand9f if r["B"][0] >= base9f["B"][0]], key=lambda r: -r["A"][0])[:2]
if not _qual:
    print("no tau is at least as good as tau=None on Fold B -> no new submission file written")
for _r in _qual:
    _tag = f"{_r['tau']:g}"
    _path = f"outputs/submission_gt_th50_g1_tau{_tag}.csv"
    write_sub9e(BASE9F["theta"], _path, f"th50 g1 tau={_tag}", fmult=BASE9F["fmult"],
                exposure=BASE9F["exposure"], gamma=BASE9F["gamma"], beta=BASE9F["beta"], tau=_r["tau"])
    _df = pd.read_csv(_path, dtype={"user_id": str})
    _n = [len(set(str(x).split())) for x in _df["tv_show_id"]]
    _zero = [1 for x in _df["tv_show_id"] if "0" in str(x).split()]
    _sha = hashlib.sha256(Path(_path).read_bytes()).hexdigest()
    assert len(_df) == len(SAMPLE_R), (len(_df), len(SAMPLE_R))
    assert _df["user_id"].tolist() == SAMPLE_R["user_id"].tolist()
    assert set(_n) == {K}, set(_n)
    assert not _zero
    print(f"  check {Path(_path).name}: rows {len(_df):,} | distinct ids per row {sorted(set(_n))} | no id 0: True | user order == sample: True | sha256 {_sha[:8]}")
print(f"\n9f runtime: {time.time() - t0_9f:.1f} s")

SECTION 9f — recency weighting of own-history hits

[1] sanity: tau=None must reproduce the section-9e th50_g1 run exactly
9f tau=None (reproduction check)           | 0.419810 | 0.262686 | 0.419047 | 0.269735
                               |    A GT* |   A GT-1 |    B GT* |   B GT-1
section 9e cell (recorded)     | 0.419810 | 0.262686 | 0.419047 | 0.269735
tau=None rerun                 | 0.419810 | 0.262686 | 0.419047 | 0.269735
assert passed: identical MAP values and identical recommendations

[2] sanity: tau=None must reproduce outputs/submission_gt_th50_g1.csv byte for byte
tau=None sha check                         -> /var/folders/b_/ntqb5st965z0z5cvz5qqv7pw0000gn/T/tau_check_wpcxfqke/tau_none_check.csv: rows 1,260 | sha256[:8] d5d7b1a2 | cold fills 4
sha256: rerun d5d7b1a2 | existing submission_gt_th50_g1.csv d5d7b1a2
assert passed: byte-identical submission

[3] tau sweep (recipe fixed at the returned th50_g1 configuration)
9f tau=None                                | 0.419810 

### 9f result

The effect is not monotone in strength: mild decay helps, strong decay hurts.

- `tau=12` and `tau=6` weeks improve both folds over the unweighted count. Fold A prefers `tau=12` and Fold B prefers `tau=6`, so validation does not settle which of the two is better; both were written as candidates.
- `tau=3` and `tau=1.5` weeks are worse than the unweighted count on both folds. Recent weeks alone do not carry enough signal — most of the predictive value sits in the whole history window, not only in its last month.
- Every value, delta and verdict is printed by the code cell above; nothing here is a hard-coded number. The two qualifying configurations are `outputs/submission_gt_th50_g1_tau12.csv` and `outputs/submission_gt_th50_g1_tau6.csv`.
- `tau=None` remains the default, so all earlier results in the notebook are unaffected: section 9f re-ran the section-9e configuration and reproduced both its scores and its submission file byte for byte.


## 9g. Finer tau grid and a smarter fill for short own-history lists

Two follow-ups to section 9f.

**Part 1 — finer tau grid.** Section 9f tested `None, 12, 6, 3, 1.5` weeks. The Kaggle scores of those two candidate files came back above the unweighted baseline, so the interesting region is the *mild* end. This part re-runs the same recipe (`theta=0.50, fmult="log1p", exposure="chhw", gamma=1.0, beta=0.25`) over `None, 9, 12, 18, 24, 36` weeks and reports Fold A and Fold B with GT* and GT-1. `tau=None` and `tau=12` are asserted to reproduce section 9f exactly.

**Part 2 — fill for users with fewer than five own-history candidates.** `rank9e` appends global popularity for slots the user's own history cannot fill. This part adds an optional `fill_scores` argument: a per-user ordered list over the target-window candidate pool, here taken from the already-fitted TruncatedSVD (`k=32`) model, with popularity as the tail so a list is never too short. The own-history items stay first and in their original order; only the slots after them change. First the section counts how many users need such a fill and how much of MAP@5 those users carry under the current recipe, then it compares popularity fill against SVD fill at `tau=12` on both folds.

**Part 3 — files.** At most three new submission files: the best `tau` from Part 1 (skipped when it is 12 or 6, whose files already exist), the best `tau` with SVD fill (only when it beats popularity fill on both folds), and `tau=12` with SVD fill when the best `tau` is 12. Existing files are never overwritten.

`fill_scores` defaults to `None` everywhere, so every earlier number in the notebook is unchanged; `eval9e` takes the dict keyed by fold (`{"A": ..., "B": ...}`), `rank9e` and `write_sub9e` take a plain user-to-list dict.


In [97]:
# --- Section 9g: finer tau grid + SVD fill for users short on own-history candidates ---
import hashlib
from pathlib import Path

print("=" * 78)
print("SECTION 9g — finer tau grid and SVD fill for short own-history lists")
print("=" * 78)
t0_9g = time.time()

BASE9G = dict(theta=0.50, fmult="log1p", exposure="chhw", gamma=1.0, beta=0.25)
TAUS9G = (None, 9.0, 12.0, 18.0, 24.0, 36.0)
OUT_BAK = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(Path("outputs").glob("*.csv"))}
print(f"submission files present at the start of 9g: {len(OUT_BAK)}")

# ---------------- Part 1: finer tau grid ----------------
print("\n[Part 1] finer tau grid, th50_g1 recipe held fixed (theta 0.50, log1p, chhw, gamma 1.0, beta 0.25)")
RUN9G = []
for _tau in TAUS9G:
    _r = eval9e(f"9g tau={_tau}", BASE9G["theta"], fmult=BASE9G["fmult"], exposure=BASE9G["exposure"],
                gamma=BASE9G["gamma"], beta=BASE9G["beta"], tau=_tau)
    _r["tau"] = _tau
    RUN9G.append(_r)

ref9g = {r["tau"]: r for r in RUN9F}
assert RUN9G[0]["A"] == ref9g[None]["A"] and RUN9G[0]["B"] == ref9g[None]["B"], (RUN9G[0]["A"], ref9g[None]["A"])
assert RUN9G[2]["A"] == ref9g[12.0]["A"] and RUN9G[2]["B"] == ref9g[12.0]["B"], (RUN9G[2]["A"], ref9g[12.0]["A"])
assert RUN9G[0]["recsA"] == ref9g[None]["recsA"] and RUN9G[2]["recsA"] == ref9g[12.0]["recsA"]
print("assert passed: tau=None and tau=12 reproduce the section-9f scores and recommendations exactly")

base9g = RUN9G[0]
print()
print(f"{'tau (weeks)':12s} | {'A GT*':>8s} | {'A GT-1':>8s} | {'B GT*':>8s} | {'B GT-1':>8s} | {'dA GT*':>8s} | {'dB GT*':>8s}")
for _r in RUN9G:
    _nm = "none (old)" if _r["tau"] is None else f"{_r['tau']:g}"
    print(f"{_nm:12s} | {_r['A'][0]:8.6f} | {_r['A'][1]:8.6f} | {_r['B'][0]:8.6f} | {_r['B'][1]:8.6f} | "
          f"{_r['A'][0] - base9g['A'][0]:+8.6f} | {_r['B'][0] - base9g['B'][0]:+8.6f}")

print("\nverdict per tau (computed from the table above):")
for _r in RUN9G:
    if _r["tau"] is None:
        continue
    _dA = _r["A"][0] - base9g["A"][0]
    _dB = _r["B"][0] - base9g["B"][0]
    if _dA > 0 and _dB > 0:
        _v = "helped on both folds"
    elif _dA > 0:
        _v = "helped on Fold A only (Fold B not improved)"
    elif _dB > 0:
        _v = "helped on Fold B only (Fold A not improved)"
    else:
        _v = "did not help on either fold"
    print(f"  tau={_r['tau']:g}: dA GT* {_dA:+.6f} | dB GT* {_dB:+.6f} -> {_v}")

best9g = max([r for r in RUN9G if r["tau"] is not None], key=lambda r: r["A"][0])
bestB9g = max([r for r in RUN9G if r["tau"] is not None], key=lambda r: r["B"][0])
print(f"\nbest tau by Fold A GT*: {best9g['tau']:g} | best tau by Fold B GT*: {bestB9g['tau']:g} | folds agree: {best9g['tau'] == bestB9g['tau']}")

# ---------------- Part 2: SVD fill for short own-history lists ----------------
print("\n[Part 2a] own-history candidates per user under the current recipe (theta 0.50, tau=None)")
def own_counts(hist, pool, cnt, pe_fold):
    g = frame9e(hist, pool, cnt, BASE9G["theta"], BASE9G["fmult"], BASE9G["exposure"],
                BASE9G["gamma"], pe_fold, BASE9G["beta"], None)
    return g.groupby("user_id").size().to_dict()

cnt_A = own_counts(labA, candA, sched_cnt_A, PE9["A"])
cnt_B = own_counts(labB, candB, sched_cnt_B, PE9["B"])
cnt_R = own_counts(histR, candR, sched_cnt_F, pe_final9())
short_A = [u for u in EVAL_USERS if cnt_A.get(u, 0) < K]
short_B = [u for u in EVAL_USERS if cnt_B.get(u, 0) < K]
short_R = [u for u in sub_users_r if cnt_R.get(u, 0) < K]

def report_short(name, users, cnt, short):
    zero = [u for u in users if cnt.get(u, 0) == 0]
    slots = sum(max(0, K - min(K, cnt.get(u, 0))) for u in users)
    print(f"{name:32s} | {len(users):6,d} | {len(short):7,d} | {len(short) / len(users):7.2%} | {len(zero):6,d} | {slots:8,d}")

print(f"{'fold':32s} | {'users':>6s} | {'<5 own':>7s} | {'share':>7s} | {'0 own':>6s} | {'slots':>8s}")
report_short("Fold A (w11-20 -> w21-30)", EVAL_USERS, cnt_A, short_A)
report_short("Fold B (w11-18 -> w19-28)", EVAL_USERS, cnt_B, short_B)
report_short("final (w11-30 -> w31-42)", sub_users_r, cnt_R, short_R)

print("\n[Part 2b] MAP@5 carried by those users under the current recipe (tau=None, popularity fill)")
def share_block(name, recs, gt, short):
    ap = {u: ap_at5(recs.get(u, []), gt.get(u, set()), K) for u in EVAL_USERS if gt.get(u)}
    tot = float(np.sum(list(ap.values())))
    s_ap = [v for u, v in ap.items() if u in short]
    r_ap = [v for u, v in ap.items() if u not in short]
    print(f"{name:22s} | {tot / len(ap) if ap else 0.0:8.6f} | {len(s_ap):7,d} | "
          f"{(float(np.mean(s_ap)) if s_ap else 0.0):10.6f} | {(float(np.mean(r_ap)) if r_ap else 0.0):9.6f} | "
          f"{(float(np.sum(s_ap)) / tot if tot else 0.0):11.2%}")

print(f"{'fold / truth':22s} | {'MAP@5':>8s} | {'short n':>7s} | {'short mean':>10s} | {'rest mean':>9s} | {'short share':>11s}")
share_block("A GT*", RUN9G[0]["recsA"], GT_STAR["A"], set(short_A))
share_block("A GT-1", RUN9G[0]["recsA"], gt1_eval, set(short_A))
share_block("B GT*", RUN9G[0]["recsB"], GT_STAR["B"], set(short_B))
share_block("B GT-1", RUN9G[0]["recsB"], gt1_B, set(short_B))
print("  (denominator = users with non-empty truth; 'short share' = sum of their AP@5 / total AP@5)")

def svd_fill_dict(Us, Vs, upos, cand_cols, cand_shows, users, pop_order, depth=50):
    Vc = Vs[:, cand_cols]
    out = {}
    for u in users:
        lst = []
        if u in upos:
            row = Us[upos[u]] @ Vc
            if np.isfinite(row).any():
                kk = min(depth, row.size)
                top = np.argpartition(-row, kth=kk - 1)[:kk]
                top = top[np.argsort(-row[top])]
                lst = [int(cand_shows[int(c)]) for c in top]
        seen = set(lst)
        for s in pop_order:
            if int(s) not in seen:
                lst.append(int(s))
                seen.add(int(s))
        out[u] = lst
    return out

popA_order = [int(s) for s in labA.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).index if int(s) in poolA_set][:20]
popB_order = [int(s) for s in labB.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).index if int(s) in poolB_set][:20]
popR_order = [int(s) for s in histR.groupby("tv_show_id")["overlap_sec"].sum().sort_values(ascending=False).index if int(s) in candR_set][:20]

fill_A = svd_fill_dict(best_svd_Us, best_svd_Vs, up_svd, cand_cols_svd, cand_shows_svd, EVAL_USERS, popA_order)
fill_B = svd_fill_dict(UsB, VsB, upB, cand_colsB, cand_showsB, EVAL_USERS, popB_order)
fill_R = svd_fill_dict(UR, VR, upR, cand_cols_R, cand_shows_R, sub_users_r, popR_order)
print(f"\nSVD fill lists built (k={best_svd_k}, SVD depth {len(fill_A[EVAL_USERS[0]])} incl. popularity tail): "
      f"Fold A {len(fill_A):,} users | Fold B {len(fill_B):,} users | final {len(fill_R):,} users")

print("\n[Part 2c] fill comparison at tau=12 (own-history part unchanged)")
RUN9GF = []
for _nm, _fs in (("popularity fill", None), ("SVD fill", {"A": fill_A, "B": fill_B})):
    _r = eval9e(f"9g tau=12 + {_nm}", BASE9G["theta"], fmult=BASE9G["fmult"], exposure=BASE9G["exposure"],
                gamma=BASE9G["gamma"], beta=BASE9G["beta"], tau=12.0, fill_scores=_fs)
    _r["fill"] = _nm
    RUN9GF.append(_r)
popf12, svdf12 = RUN9GF
print(f"\n{'variant':22s} | {'A GT*':>8s} | {'A GT-1':>8s} | {'B GT*':>8s} | {'B GT-1':>8s} | {'dA GT*':>8s} | {'dB GT*':>8s}")
for _r in RUN9GF:
    print(f"{_r['fill']:22s} | {_r['A'][0]:8.6f} | {_r['A'][1]:8.6f} | {_r['B'][0]:8.6f} | {_r['B'][1]:8.6f} | "
          f"{_r['A'][0] - popf12['A'][0]:+8.6f} | {_r['B'][0] - popf12['B'][0]:+8.6f}")
fill_ok_12 = bool(svdf12["A"][0] > popf12["A"][0] and svdf12["B"][0] > popf12["B"][0])
print(f"SVD fill beats popularity fill on both folds at tau=12 (GT*): {fill_ok_12}")
print(f"cold-fill rows: popularity {popf12['cold']} | SVD {svdf12['cold']}")

if abs(best9g["tau"] - 12.0) > 1e-9:
    print(f"\n[Part 2d] fill comparison at the best tau ({best9g['tau']:g})")
    RUN9GB = []
    for _nm, _fs in (("popularity fill", None), ("SVD fill", {"A": fill_A, "B": fill_B})):
        _r = eval9e(f"9g tau={best9g['tau']:g} + {_nm}", BASE9G["theta"], fmult=BASE9G["fmult"],
                    exposure=BASE9G["exposure"], gamma=BASE9G["gamma"], beta=BASE9G["beta"],
                    tau=best9g["tau"], fill_scores=_fs)
        _r["fill"] = _nm
        RUN9GB.append(_r)
    for _r in RUN9GB:
        print(f"{_r['fill']:22s} | {_r['A'][0]:8.6f} | {_r['A'][1]:8.6f} | {_r['B'][0]:8.6f} | {_r['B'][1]:8.6f} | "
              f"{_r['A'][0] - RUN9GB[0]['A'][0]:+8.6f} | {_r['B'][0] - RUN9GB[0]['B'][0]:+8.6f}")
    fill_ok_bt = bool(RUN9GB[1]["A"][0] > RUN9GB[0]["A"][0] and RUN9GB[1]["B"][0] > RUN9GB[0]["B"][0])
    print(f"SVD fill beats popularity fill on both folds at tau={best9g['tau']:g} (GT*): {fill_ok_bt}")
else:
    fill_ok_bt = fill_ok_12
    print("\nbest tau is 12, so the comparison above is also the best-tau comparison")

# ---------------- Part 3: submission files ----------------
print("\n[Part 3] new submission files (existing files are never overwritten)")
PLAN = []
if abs(best9g["tau"] - 12.0) > 1e-9 and abs(best9g["tau"] - 6.0) > 1e-9:
    PLAN.append((best9g["tau"], "pop"))
else:
    print(f"best tau is {best9g['tau']:g}: its file already exists from section 9f, nothing written for it")
if fill_ok_bt:
    PLAN.append((best9g["tau"], "svd"))
else:
    print("SVD fill does not beat popularity fill on both folds at the best tau -> no svdfill file")
if abs(best9g["tau"] - 12.0) <= 1e-9 and fill_ok_12:
    PLAN.append((12.0, "svd"))

written = []
for _tau, _kind in PLAN:
    _path = f"outputs/submission_gt_th50_g1_tau{_tau:g}" + ("" if _kind == "pop" else "_svdfill") + ".csv"
    if _path in written:
        continue
    if Path(_path).exists():
        print(f"skip {_path}: already exists")
        continue
    write_sub9e(BASE9G["theta"], _path, f"th50 g1 tau={_tau:g}" + ("" if _kind == "pop" else " + svd fill"),
                fmult=BASE9G["fmult"], exposure=BASE9G["exposure"], gamma=BASE9G["gamma"],
                beta=BASE9G["beta"], tau=_tau, fill_scores=None if _kind == "pop" else fill_R)
    _df = pd.read_csv(_path, dtype={"user_id": str})
    _n = [len(set(str(x).split())) for x in _df["tv_show_id"]]
    _zero = [1 for x in _df["tv_show_id"] if "0" in str(x).split()]
    _sha = hashlib.sha256(Path(_path).read_bytes()).hexdigest()
    assert len(_df) == len(SAMPLE_R), (len(_df), len(SAMPLE_R))
    assert _df["user_id"].tolist() == SAMPLE_R["user_id"].tolist()
    assert set(_n) == {K}, set(_n)
    assert not _zero
    written.append(_path)
    print(f"  check {Path(_path).name}: rows {len(_df):,} | distinct ids per row {sorted(set(_n))} | no id 0: True | "
          f"order == data/submission.csv: True | sha256 {_sha[:8]}")
if not written:
    print("no new submission file written in section 9g")

print("\n[Part 3b] pre-existing submission files unchanged")
_changed = [nm for nm, h in OUT_BAK.items() if hashlib.sha256(Path("outputs", nm).read_bytes()).hexdigest() != h]
print(f"files present at 9g start: {len(OUT_BAK)} | changed: {len(_changed)} {_changed}")
assert not _changed, _changed
print(f"\n9g runtime: {time.time() - t0_9g:.1f} s")


SECTION 9g — finer tau grid and SVD fill for short own-history lists
submission files present at the start of 9g: 21

[Part 1] finer tau grid, th50_g1 recipe held fixed (theta 0.50, log1p, chhw, gamma 1.0, beta 0.25)
9g tau=None                                | 0.419810 | 0.262686 | 0.419047 | 0.269735
9g tau=9.0                                 | 0.420337 | 0.264155 | 0.421939 | 0.271982
9g tau=12.0                                | 0.420883 | 0.263962 | 0.421295 | 0.271825
9g tau=18.0                                | 0.420482 | 0.263719 | 0.421396 | 0.271354
9g tau=24.0                                | 0.421145 | 0.263363 | 0.420894 | 0.270787
9g tau=36.0                                | 0.420554 | 0.263036 | 0.419889 | 0.270556
assert passed: tau=None and tau=12 reproduce the section-9f scores and recommendations exactly

tau (weeks)  |    A GT* |   A GT-1 |    B GT* |   B GT-1 |   dA GT* |   dB GT*
none (old)   | 0.419810 | 0.262686 | 0.419047 | 0.269735 | +0.000000 | +0.000000
9    

### 9g result (written by the student after the agent's usage limit ran out)

- Finer tau grid (printed by the 9g cell): every tau from 9 to 36 weeks beats the unweighted count on GT* on both folds (Fold A +0.0005 to +0.0013, Fold B +0.0008 to +0.0029). Fold A prefers tau=24 and Fold B prefers tau=9, so validation does not pick a winner.
- Short lists: only 39 (Fold A), 52 (Fold B) and 13 (final) of 1,260 users have fewer than five own-history candidates, and they carry about 1% of MAP@5.
- SVD fill is worse than popularity fill at tau=12 and tau=24 on both folds (GT*: -0.0005 on Fold A, -0.0002 on Fold B), so no SVD-fill file was written.

### 9h-9i: cells added by the student (NOT part of the OpenCode session)

After OpenCode's usage limit ran out I pasted and ran the following cells by hand; the code was suggested in a separate Claude chat. 9h routes `rank9e` through an SVD-based fill and repeats the fill comparison at tau=18 and tau=24 (same conclusion as the agent's 9g: no gain). The last cell writes `outputs/submission_gt_th50_g1_tau18.csv` and `..._tau24.csv`. `tau18` is the designated final submission because it had the best returned Kaggle private score (0.42853; tau24 0.42824, tau12 0.42702); validation cannot separate these three.

In [98]:
# --- 9h (added by me): SVD fill instead of popularity fill ---
FILL_MODE = None   # None = old behaviour, "svd" = fill by SVD scores

def svd_fill_factory(U, V, up, cols, shows, topn=60):
    Vc = V[:, cols]
    def f(u):
        i = up.get(u)
        if i is None:
            return []
        s = U[i] @ Vc
        o = np.argsort(-s)[:topn]
        return [int(shows[j]) for j in o]
    return f

FILL_FNS = {
    "A": svd_fill_factory(best_svd_Us, best_svd_Vs, up_svd, cand_cols_svd, cand_shows_svd),
    "B": svd_fill_factory(UsB, VsB, upB, cand_colsB, cand_showsB),
    "F": svd_fill_factory(UF, VF, upF, cand_cols_F, cand_shows_F),
}

# self-check: the fill functions must reproduce the pure-SVD recommendations stored earlier
for key, recs_ref, up in (("A", best_recs_svd, up_svd), ("B", recsB_svd, upB), ("F", recs_final, upF)):
    us = [u for u in list(recs_ref)[:300] if u in up]
    ok = sum(FILL_FNS[key](u)[:K] == [int(x) for x in recs_ref[u]] for u in us)
    print(f"fill check {key}: {ok}/{len(us)} users identical to stored pure-SVD top-5")

if "rank9e_orig" not in globals():
    rank9e_orig = rank9e

def rank9e_fill(hist, pool, cnt, users, fill_pop, theta, *a, **kw):
    if FILL_MODE is None:
        return rank9e_orig(hist, pool, cnt, users, fill_pop, theta, *a, **kw)
    key = "A" if hist is labA else "B" if hist is labB else "F" if hist is histR else None
    assert key is not None, "unknown history frame"
    recs0, _ = rank9e_orig(hist, pool, cnt, users, [], theta, *a, **kw)   # own history only
    fn = FILL_FNS[key]
    pool_set = set(int(x) for x in pool)
    recs, n_cold = {}, 0
    for u in users:
        r = list(recs0.get(u, []))
        if not r:
            n_cold += 1
        for v in fn(u) + [int(x) for x in fill_pop]:
            if len(r) >= K:
                break
            if v not in r and v in pool_set:
                r.append(v)
        recs[u] = r[:K]
    return recs, n_cold

rank9e = rank9e_fill
print("rank9e now routes through rank9e_fill (FILL_MODE =", FILL_MODE, ")")

fill check A: 296/296 users identical to stored pure-SVD top-5
fill check B: 296/296 users identical to stored pure-SVD top-5
fill check F: 299/299 users identical to stored pure-SVD top-5
rank9e now routes through rank9e_fill (FILL_MODE = None )


In [ ]:
# how many users have fewer than 5 own-history candidates (recipe th50 g1)?
for key, h, c, cn, pe in (("A", labA, candA, sched_cnt_A, PE9["A"]), ("B", labB, candB, sched_cnt_B, PE9["B"])):
    r0, _ = rank9e_orig(h, c, cn, EVAL_USERS, [], 0.50, fmult="log1p", exposure="chhw",
                        gamma=1.0, pe=pe, beta=0.25, tau=12)
    n_short = sum(len(r0.get(u, [])) < K for u in EVAL_USERS)
    n_zero = sum(len(r0.get(u, [])) == 0 for u in EVAL_USERS)
    print(f"Fold {key}: users with <{K} own candidates: {n_short}/{len(EVAL_USERS)} (of them with 0: {n_zero})")

print(f"\n{'run':42s} | {'A GT*':>8s} | {'A GT-1':>8s} | {'B GT*':>8s} | {'B GT-1':>8s}")
for mode in (None, "svd"):
    FILL_MODE = mode
    eval9e(f"th50 g1 tau=18, fill={mode}", 0.50, fmult="log1p", exposure="chhw",
           gamma=1.0, beta=0.25, tau=18)
FILL_MODE = None

for mode in (None, "svd"):
    FILL_MODE = mode
    eval9e(f"th50 g1 tau=24, fill={mode}", 0.50, fmult="log1p", exposure="chhw",
           gamma=1.0, beta=0.25, tau=24)

FILL_MODE = None

Fold A: users with <5 own candidates: 39/1260 (of them with 0: 15)
Fold B: users with <5 own candidates: 52/1260 (of them with 0: 16)

run                                        |    A GT* |   A GT-1 |    B GT* |   B GT-1
th50 g1 tau=18, fill=None                  | 0.420482 | 0.263719 | 0.421396 | 0.271354
th50 g1 tau=18, fill=svd                   | 0.419949 | 0.263291 | 0.421203 | 0.271236
th50 g1 tau=24, fill=None                  | 0.421145 | 0.263363 | 0.420894 | 0.270787
th50 g1 tau=24, fill=svd                   | 0.420613 | 0.262935 | 0.420701 | 0.270669


In [103]:
FILL_MODE = None   # no SVD fill: popularity fill, as in the best submission
for tau in (18, 24):
    write_sub9e(0.50, f"outputs/submission_gt_th50_g1_tau{tau}.csv", f"th50 g1 tau{tau}",
                fmult="log1p", exposure="chhw", gamma=1.0, beta=0.25, tau=tau)

th50 g1 tau18                              -> outputs/submission_gt_th50_g1_tau18.csv: rows 1,260 | sha256[:8] 518d8753 | cold fills 4
th50 g1 tau24                              -> outputs/submission_gt_th50_g1_tau24.csv: rows 1,260 | sha256[:8] ff5eb700 | cold fills 4


In [104]:
# --- Section 10c/10d (MANUAL): case-study table, all methods incl. the FINAL configuration ---
FILL_MODE = None
final_recs, _ = rank9e(labA, candA, sched_cnt_A, EVAL_USERS, pop_fallback_A, 0.50,
                       fmult="log1p", exposure="chhw", gamma=1.0, pe=PE9["A"], beta=0.25, tau=18)
METHODS = [("Popularity", {u: list(pop_fallback_A)[:K] for u in CASE_USERS}),
           ("SVD", best_recs_svd), ("ALS", best_recs_als), ("CBF", best_cbf_recs_A),
           ("Hybrid", best_hybrid_recs_A), ("Final", final_recs)]
def cell(sid, u):
    si, title, cat, _ = show_info(sid)
    mk = ("*" if sid in gt1_eval.get(u, set()) else "") + ("+" if sid in GT_STAR["A"].get(u, set()) else "")
    return f"{si} {title[:26].replace('|','/')} ({cat[:6]}){mk}"
for u in CASE_USERS:
    own = set(int(s) for s in labA.loc[labA["user_id"] == u, "tv_show_id"].unique())
    print(f"\n### user {u}  (* = GT-1 hit, + = GT* hit)")
    print("| rank | " + " | ".join(n for n, _ in METHODS) + " |")
    print("|" + "---|" * (len(METHODS) + 1))
    for r in range(K):
        print(f"| {r+1} | " + " | ".join(cell(int(rc[u][r]), u) for _, rc in METHODS) + " |")
    for n, rc in METHODS:
        t = [int(x) for x in rc[u][:K]]
        print(f"{n}: GT-1 hits {sum(x in gt1_eval.get(u,set()) for x in t)}/5, "
              f"own-history slots {sum(x in own for x in t)}/5, categories {len({show_info(x)[2] for x in t})}")


### user 13571199046623737884  (* = GT-1 hit, + = GT* hit)
| rank | Popularity | SVD | ALS | CBF | Hybrid | Final |
|---|---|---|---|---|---|---|
| 1 | 2400480 'ТСН. Неделя'. (Инфо)*+ | 200405 'Мамахохотала-шоу'. Лучшее (Развле) | 400426 'Меняю жену'. (Развле) | 1000608 Т/с 'Приключения Геркулеса (Сериал) | 2400480 'ТСН. Неделя'. (Инфо)*+ | 2400480 'ТСН. Неделя'. (Инфо)*+ |
| 2 | 240081 Факты. День. (Инфо)*+ | 10003159 Т/с 'Ничто не случается дв (Сериал) | 5800509 Реальная мистика. (Развле) | 10002454 Т/с 'Гримм', 1 сезон, 1 и  (Сериал) | 20088 'Завтрак с 1+1'. Информаци (Развле) | 240081 Факты. День. (Инфо)*+ |
| 3 | 2400508 Сегодня. Итоги с Олегом Па (Инфо) | 700369 'Лига смеха', 6 сезон, 1 и (Развле)* | 90074731 Х/ф 'Мстители'. (Фильмы) | 1000289 Т/с 'Затерянные', 1 сезон, (Сериал) | 240081 Факты. День. (Инфо)*+ | 20088 'Завтрак с 1+1'. Информаци (Развле) |
| 4 | 20088 'Завтрак с 1+1'. Информаци (Развле) | 10003178 Премьера. Т/с 'Агенты спра (Сериал) | 90076007 Х/ф 'Железный челове

In [105]:
# --- 14 (MANUAL): integrity check of the designated final file + manifest of outputs ---
import hashlib, time
import pandas as pd
from pathlib import Path
FINAL = Path("outputs/submission_gt_th50_g1_tau18.csv")
sample = pd.read_csv("data/submission.csv", dtype={"user_id": str})
df = pd.read_csv(FINAL, dtype={"user_id": str})
ids = [str(x).split() for x in df["tv_show_id"]]
assert len(df) == len(sample) == 1260
assert df["user_id"].tolist() == sample["user_id"].tolist()
assert all(len(r) == 5 and len(set(r)) == 5 for r in ids)
assert not any("0" in r for r in ids)
assert all(int(x) in candR_set for r in ids for x in r)
print("final file OK:", FINAL.name, "| sha256", hashlib.sha256(FINAL.read_bytes()).hexdigest())
print(f"csv files in outputs/: {len(list(Path('outputs').glob('*.csv')))}")
for p in sorted(Path("outputs").glob("*.csv")):
    print(f"  {p.name:45s} {hashlib.sha256(p.read_bytes()).hexdigest()[:8]}")
print(f"kernel time since the first cell: {time.time() - NOTEBOOK_T0:.0f} s (section 13 was measured before 9f-9i)")

final file OK: submission_gt_th50_g1_tau18.csv | sha256 518d87537ce2760cd827c6432b7f1ed899d616b6a10162a645d358c9a136db81
csv files in outputs/: 22
  submission_best_gtstar.csv                    c2805b0e
  submission_cbf.csv                            9c0c56fb
  submission_gt_th50.csv                        cbae38cb
  submission_gt_th50_b50.csv                    e1fa63b8
  submission_gt_th50_g1.csv                     d5d7b1a2
  submission_gt_th50_g15.csv                    fb6902c6
  submission_gt_th50_g1_tau12.csv               966fb76e
  submission_gt_th50_g1_tau18.csv               518d8753
  submission_gt_th50_g1_tau24.csv               ff5eb700
  submission_gt_th50_g1_tau6.csv                0a36f231
  submission_gt_th60.csv                        fcbd0b74
  submission_gt_th75.csv                        258caceb
  submission_hybrid.csv                         2550c033
  submission_mf.csv                             c9f84e23
  submission_mf_recheck.csv                     c9f84e2